# PDF 파싱·검토·청킹 baseline — 목차 있음/없음 겸용

PDF 한 건을 `PDF 준비 → 파싱 → 검토 → 청킹 → 표 설명·Prefix → QA → CSV` 순서로 처리합니다.

이 파일은 `baseline_save3.ipynb`의 제목 각주 처리 수정본입니다.
제목의 `<sup>18)</sup>`과 기존 정제본의 `제도 개요140)`을 자동 정리합니다.
본문·표의 각주 번호와 각주 설명은 유지하며, 기존 정제본은 `preserve`로 읽어도 요소 파싱·매칭에 적용됩니다.

- **목차 있음** (`HAS_TOC=True`): 검토한 목차 제목을 본문 제목과 매칭해 나눕니다.
- **목차 없음** (`HAS_TOC=False`, 기본): 본문 제목의 실제 위치에서 나눕니다. 제목도 없으면 문단·길이 기준입니다.

표는 제목·단위·출처·설명과 함께 별도 청크로 보존하고, 짧은 본문은 같은 상위 제목 안에서 병합합니다.

**사용 순서**

1. `0. 초기화` 두 셀을 실행합니다. 각 단계 시작의 접힌 함수 정의 셀은 처음 진행할 때 함께 실행합니다.
2. `1. 문서 설정`을 입력하고 위에서부터 순서대로 실행합니다. 각 단계의 설정은 그 단계 바로 앞 셀에 있습니다.
3. 검토 단계는 `준비 → (파일 편집·저장) → 재로딩·미리보기 → 승인` 순서입니다. 승인 셀은 값을 `True`로 바꾼 뒤 실행합니다.

**결과물**: 최종 CSV는 원본 PDF 옆 `{PDF 파일명}.csv`, 중간 산출물은 `{PDF 파일명}_work/`에 저장합니다.

**무엇을 바꾸면 어디부터 다시 실행하나**

| 변경한 내용 | 다시 실행할 범위 |
|---|---|
| 크롭 여부·비율 | 3. 크롭 실행부터 파싱·후속 단계 |
| 검토 목차 파일(`toc_titles_reviewed.md`) | 5. 목차 재로딩·승인 → 7. 경계 매칭부터 |
| 경계 별칭 | 7. 경계 매칭부터 |
| 본문 수정 Markdown(노이즈 후보 체크 포함) | 6-B 반영·미리보기 → 6-C 승인 → 7. 요소 파싱부터 |
| 파싱 원문이 바뀌어 정제본을 새로 만들 때 | 수정본 백업 → 6-A에서 rebuild → 6-B → 6-C |
| 청크 길이 기준 | 8. 청킹부터 |
| 표 설명 수정 (`TABLE_SUMMARY_OVERRIDES`) | 10. 표 설명 반영·Prefix 생성부터 |
| Prefix 수정 (`PREFIX_OVERRIDES`) | 11. Prefix 반영·최종 QA → 승인 → 저장 |

실행 셀은 선행 결과가 현재 입력·설정과 맞는지 확인하고, 맞지 않으면 다시 실행할 셀을 알려 줍니다.
검토 대상 파일이 바뀌면 이전 승인은 자동으로 무효가 됩니다.

## 0. 초기화

환경·공통 함수와 단계 상태 관리만 준비합니다. 두 셀을 실행한 뒤 1번 문서 설정으로 진행하세요.

단계별 처리 함수는 해당 단계 시작의 접힌 정의 셀에 있습니다. 처음 실행할 때는 정의 셀도 위에서부터 함께 실행합니다. 설정만 바꿀 때는 이미 실행한 정의 셀을 다시 실행할 필요가 없습니다.

두 번째 셀(단계 상태)을 다시 실행하면 진행 상태가 초기화되어 1번부터 다시 실행해야 합니다.

In [87]:
from __future__ import annotations

import asyncio
import hashlib
import html
import importlib.metadata
import json
import os
import re
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path
from typing import Callable, Iterable

import nest_asyncio
import pandas as pd
from dotenv import load_dotenv
from IPython.display import Image, Markdown, display
from llama_index.core.node_parser import MarkdownElementNodeParser
from openai import AsyncOpenAI
from tenacity import retry, stop_after_attempt, wait_random_exponential

nest_asyncio.apply()


def find_notebook_dir() -> Path:
    from IPython import get_ipython

    shell = get_ipython()
    notebook_file = shell.user_ns.get("__vsc_ipynb_file__") if shell else None
    if notebook_file:
        return Path(notebook_file).expanduser().resolve().parent
    script_file = globals().get("__file__")
    if script_file:
        return Path(script_file).resolve().parent
    # 일반 Jupyter는 노트북 폴더에서 커널을 시작합니다. 폴더가 다르면
    # 1번에서 PDF_PATH를 절대 경로로 지정하세요.
    return Path.cwd().resolve()


# PDF 상대 경로의 기준 폴더이며, 공용 .env는 상위 폴더까지 탐색합니다.
PROJECT_ROOT = find_notebook_dir()
ENV_PATH = next(
    (folder / ".env" for folder in [PROJECT_ROOT, *PROJECT_ROOT.parents]
     if (folder / ".env").is_file()),
    PROJECT_ROOT / ".env",
)

HTML_ENTITY_PATTERN = re.compile(
    r"&(?:#\d+|#x[0-9A-Fa-f]+|[A-Za-z][A-Za-z0-9]+);"
)

HTML_FORMATTING_TAG_PATTERN = re.compile(
    r"</?(?P<tag>sup|sub|mark|u)\b[^>]*>",
    re.IGNORECASE,
)

TITLE_SUP_FOOTNOTE_PATTERN = re.compile(
    r"<sup\b[^>]*>(?P<label>.*?)</sup\s*>",
    re.IGNORECASE,
)
TITLE_FOOTNOTE_NUMBER_PATTERN = re.compile(r"[0-9]{1,4}[)）]")
TITLE_FLATTENED_FOOTNOTE_PATTERN = re.compile(
    r"(?<=[A-Za-z가-힣)\]])(?:[0-9]{1,4}[)）])+"
    r"(?=[ \t]*(?:\*\*|__|`)*[ \t]*$)"
)


def strip_title_footnote_markers(text: str) -> str:
    """제목에 붙은 숫자 각주 참조만 제거합니다. 본문 문장에는 적용하지 않습니다."""
    value = html.unescape(text)

    def remove_numbered_sup(match: re.Match[str]) -> str:
        label = re.sub(r"<[^>]+>", "", match.group("label")).strip()
        # 18) 같은 각주만 제거합니다. 지수 <sup>2</sup>나 ®는 보존합니다.
        return "" if TITLE_FOOTNOTE_NUMBER_PATTERN.fullmatch(label) else match.group()

    value = TITLE_SUP_FOOTNOTE_PATTERN.sub(remove_numbered_sup, value)
    # 이전 정제본은 <sup> 태그만 사라져 '제도 개요140)'로 남아 있습니다.
    # 태그 없는 번호는 제목 끝에 공백 없이 붙은 경우에만 후보로 봅니다.
    candidate = TITLE_FLATTENED_FOOTNOTE_PATTERN.search(value)
    if candidate is None:
        return value

    # '(ISO9001)'·'(제18)'처럼 대응하는 여는 괄호가 있으면 제목 내용입니다.
    # '(버전2)140)'에서는 버전 번호를 보존하고 뒤의 140)만 제거합니다.
    depth = 0
    paired_closers: set[int] = set()
    for index, character in enumerate(value[:candidate.end()]):
        if character in "(（":
            depth += 1
        elif character in ")）" and depth:
            depth -= 1
            paired_closers.add(index)

    markers = list(TITLE_FOOTNOTE_NUMBER_PATTERN.finditer(candidate.group()))
    for marker in reversed(markers):
        start = candidate.start() + marker.start()
        end = candidate.start() + marker.end()
        if end - 1 not in paired_closers:
            value = value[:start] + value[end:]
    return value

BLOCKQUOTE_LINE_PATTERN = re.compile(
    r"^\s*(?:#{1,6}\s+)?(?:>\s*)+",
    re.MULTILINE,
)


def show_user_checkpoint(
    title: str,
    lines: list[str],
    next_step: str | None = None,
) -> None:
    body = "\n".join(f"- {line}" for line in lines)
    next_text = f"\n\n**다음 작업:** {next_step}" if next_step else ""
    display(Markdown(f"### 사용자 확인: {title}\n\n{body}{next_text}"))


def clean_value(value: object) -> str:
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    return unicodedata.normalize("NFC", str(value)).strip()


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for block in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def stable_json_hash(value: object) -> str:
    serialized = json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    )
    return hashlib.sha256(serialized.encode("utf-8")).hexdigest()


_MISSING = object()


def load_json(path: Path, default: object = _MISSING) -> object:
    if not path.exists():
        if default is not _MISSING:
            return default
        raise FileNotFoundError(path)
    return json.loads(path.read_text(encoding="utf-8"))


def save_json(value: object, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(value, ensure_ascii=False, indent=2, default=str),
        encoding="utf-8",
    )


def load_cache(path: Path) -> dict[str, str]:
    value = load_json(path, default={})
    if not isinstance(value, dict):
        raise ValueError(f"캐시는 JSON 객체여야 합니다: {path}")
    return {str(key): str(item) for key, item in value.items()}


def save_cache(cache: dict[str, str], path: Path) -> None:
    save_json(cache, path)


def normalize_spaces(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def pdf_page_count(pdf_path: Path) -> int:
    from pypdf import PdfReader

    return len(PdfReader(str(pdf_path)).pages)


# FROM_URL에서 문서번호로 읽는 파라미터입니다. 다른 사이트의 번호 파라미터가 있으면 여기에 추가합니다.
DOC_NO_URL_PARAMS = ("brdNo", "bbstxSn", "pNttSn", "nttSn", "pRptNo")
_DOC_NO_IN_URL = re.compile(rf"[?&](?:{'|'.join(DOC_NO_URL_PARAMS)})=(\d+)")


def _doc_key(doc_no=None, from_url=None, title=None) -> str:
    """문서 식별키. 우선순위: 문서번호 > URL의 번호 파라미터 > (URL이 없을 때만) 제목 해시."""
    if doc_no and str(doc_no).strip():
        return clean_value(doc_no)
    if from_url and str(from_url).strip():
        match = _DOC_NO_IN_URL.search(clean_value(from_url))
        if not match:
            raise ValueError(
                "FROM_URL에서 문서번호를 찾지 못했습니다 "
                f"(읽는 파라미터: {', '.join(DOC_NO_URL_PARAMS)}). "
                "DOCUMENT_NO에 문서번호를 직접 입력하거나, 문서 제목 해시를 문서키로 쓰려면 "
                "USE_URL_FOR_DOC_KEY=False로 바꾸세요."
            )
        return match.group(1)
    if title and str(title).strip():
        normalized_title = normalize_spaces(clean_value(title)).lower()
        return hashlib.sha1(normalized_title.encode("utf-8")).hexdigest()[:12]
    raise ValueError("문서 식별 정보가 없습니다 (doc_no / from_url / title 중 하나 필요)")


def resolve_document_key() -> tuple[str, str]:
    """현재 1번 설정으로 정해지는 문서키와 그 출처."""
    if not isinstance(USE_URL_FOR_DOC_KEY, bool):
        raise ValueError("USE_URL_FOR_DOC_KEY는 True 또는 False여야 합니다.")
    url = FROM_URL if USE_URL_FOR_DOC_KEY else ""
    key = _doc_key(DOCUMENT_NO, url, DOCUMENT_NAME)
    if DOCUMENT_NO and str(DOCUMENT_NO).strip():
        return key, "DOCUMENT_NO"
    return key, "FROM_URL의 번호" if url.strip() else "문서 제목 해시"

In [88]:
# 단계 상태: 각 단계는 실제로 사용하는 선행 결과에만 의존합니다.
STAGE_DEPENDENCIES: dict[str, list[str]] = {
    "paths": [],
    "body_pdf_file": ["paths"],
    "body_pdf": ["body_pdf_file"],
    "crop": ["body_pdf"],
    "parse_body": ["crop"],
    "parse_toc": ["body_pdf_file"],
    "toc_file": ["parse_toc"],
    "toc": ["toc_file"],
    "refine": ["parse_body"],
    "body_review": ["refine"],
    "elements": ["body_review"],
    "boundaries": ["elements", "toc"],
    "chunks": ["elements", "boundaries"],
    "table_summaries": ["chunks"],
    "prefixes": ["chunks", "table_summaries"],
    "final": ["prefixes"],
    "final_save": ["final"],
}

STAGE_CELLS: dict[str, str] = {
    "paths": "1. 경로·파생값",
    "body_pdf_file": "2. 본문 PDF 준비",
    "body_pdf": "2. 본문 PDF 재로딩·미리보기와 승인",
    "crop": "3. 크롭 실행",
    "parse_body": "4. 파싱 실행",
    "parse_toc": "4. 파싱 실행",
    "toc_file": "5. 목차 추출·검토 파일 준비",
    "toc": "5. 목차 재로딩·미리보기와 승인",
    "refine": "6-A. 정제본 생성·유지",
    "body_review": "6-C. 본문 수정본 승인",
    "elements": "7. 요소 파싱",
    "boundaries": "7. 경계 매칭",
    "chunks": "8. 청킹",
    "table_summaries": "9. 표 설명 생성",
    "prefixes": "10. 표 설명 반영·Prefix 생성",
    "final": "11. Prefix 반영·최종 QA",
    "final_save": "11. 최종 저장 승인",
}

# 각 단계의 결과를 결정하는 현재 설정·파일 지문입니다. 해당 단계를 확인할 때만 계산합니다.
STAGE_INPUTS: dict[str, Callable[[], object]] = {
    "paths": lambda: {
        "pdf": str(PDF_PATH.resolve()),
        "pdf_sha256": file_sha256(PDF_PATH),
        "document_name": DOCUMENT_NAME,
    },
    "body_pdf_file": lambda: {
        "has_toc": HAS_TOC,
        "toc_pages": [TOC_START_PAGE, TOC_END_PAGE] if HAS_TOC else None,
        "body_start_page": BODY_START_PAGE,
        "body_exclude_pdf_pages": BODY_EXCLUDE_PDF_PAGES,
    },
    "body_pdf": lambda: {"body_pdf_sha256": file_sha256(BODY_PDF_PATH)},
    "crop": lambda: {
        "enabled": BODY_CROP_ENABLED,
        "right_ratio": BODY_CROP_RIGHT_RATIO,
    },
    "parse_body": lambda: {
        "tier": LLAMA_PARSE_TIER,
        "version": LLAMA_PARSE_VERSION,
        "languages": LLAMA_PARSE_LANGUAGES,
        "parsed_body_sha256": file_sha256(BODY_MARKDOWN_PATH),
    },
    "parse_toc": lambda: {
        "tier": LLAMA_PARSE_TIER,
        "version": LLAMA_PARSE_VERSION,
        "languages": LLAMA_PARSE_LANGUAGES,
        "parsed_toc_sha256": file_sha256(TOC_MARKDOWN_PATH) if HAS_TOC else None,
    },
    "toc_file": lambda: {},
    "toc": lambda: {
        "reviewed_toc_sha256": (
            file_sha256(TOC_TITLES_REVIEWED_MD_PATH) if HAS_TOC else None
        ),
    },
    "refine": lambda: {},
    "body_review": lambda: {
        "refined_body_sha256": file_sha256(BODY_REFINED_MARKDOWN_PATH),
    },
    "elements": lambda: {
        "table_context_max_before_chars": TABLE_CONTEXT_MAX_BEFORE_CHARS,
        "table_context_stop": TABLE_CONTEXT_STOP_RE.pattern,
    },
    "boundaries": lambda: {"has_toc": HAS_TOC, "aliases": BOUNDARY_ALIASES},
    "chunks": lambda: {
        "min_chars": MIN_CHARS,
        "max_chars": MAX_CHARS,
        "drop_short_chars": DROP_SHORT_CHARS,
        "document": [
            DOCUMENT_NAME, DOCUMENT_SECTION, DOCUMENT_COUNTRY, DOCUMENT_NO, FROM_URL,
        ],
        "use_url_for_doc_key": USE_URL_FOR_DOC_KEY,
    },
    "table_summaries": lambda: {"mode": TABLE_SUMMARY_MODE, "model": OPENAI_MODEL},
    "prefixes": lambda: {
        "mode": PREFIX_MODE,
        "model": OPENAI_MODEL,
        "max_chars": PREFIX_MAX_CHARS,
        "table_summary_overrides": TABLE_SUMMARY_OVERRIDES,
    },
    "final": lambda: {"prefix_overrides": PREFIX_OVERRIDES},
    "final_save": lambda: {},
}

STAGE_STAMPS: dict[str, str] = {}


def stage_state(stage: str) -> str:
    """현재 입력과 선행 단계 스탬프로 계산한 이 단계의 기대 스탬프."""
    return stable_json_hash(
        {
            "stage": stage,
            "inputs": STAGE_INPUTS[stage](),
            "upstream": {
                dependency: STAGE_STAMPS.get(dependency)
                for dependency in STAGE_DEPENDENCIES[stage]
            },
        }
    )


def first_invalid_stage(
    stages: Iterable[str],
    checked: dict[str, tuple[str, str] | None],
) -> tuple[str, str] | None:
    """선행 단계부터 확인해 가장 앞의 무효 단계와 사유를 반환합니다."""
    for stage in stages:
        if stage not in checked:
            found = first_invalid_stage(STAGE_DEPENDENCIES[stage], checked)
            if found is None:
                if stage not in STAGE_STAMPS:
                    found = (stage, "missing")
                else:
                    try:
                        is_current = STAGE_STAMPS[stage] == stage_state(stage)
                    except FileNotFoundError:
                        is_current = False
                    if not is_current:
                        found = (stage, "stale")
            checked[stage] = found
        if checked[stage] is not None:
            return checked[stage]
    return None


def stage_rerun_hint(stage: str) -> str:
    """정제본 생성과 수동 수정 후 재승인에 필요한 셀을 구분합니다."""
    if stage == "refine":
        return (
            "6-A '정제본 생성·유지' 실행 셀부터 실행하세요. "
            "기존 수정본을 유지하려면 REFINED_MARKDOWN_MODE='preserve'를 사용합니다. "
            "그다음 6-B 반영·미리보기 → 6-C 승인 순서로 진행하세요. "
            "이 안내만으로 rebuild로 바꿀 필요는 없습니다."
        )
    if stage == "body_review":
        return (
            "6-B '수정본 반영·재로딩·미리보기' 셀을 실행한 뒤, "
            "6-C '본문 수정본 승인' 셀에서 BODY_REVIEW_APPROVED=True로 실행하세요. "
            "정제본을 다시 생성하는 6-A 셀은 실행할 필요가 없습니다."
        )
    return "해당 셀부터 다시 실행하세요."


def require_stages(stages: Iterable[str]) -> None:
    found = first_invalid_stage(stages, {})
    if found is None:
        return
    stage, reason = found
    label = STAGE_CELLS[stage]
    if stage in {"refine", "body_review"}:
        reason_text = "단계가 아직 완료되지 않았습니다." if reason == "missing" else "단계 이후 입력이나 설정이 바뀌었습니다."
        raise RuntimeError(f"'{label}' {reason_text} {stage_rerun_hint(stage)}")
    if reason == "missing":
        raise RuntimeError(
            f"'{label}' 단계가 아직 완료되지 않았습니다. 해당 셀을 먼저 실행하세요. "
            "승인 단계는 값을 True로 바꾼 뒤 실행합니다."
        )
    raise RuntimeError(
        f"'{label}' 단계 이후 입력이나 설정이 바뀌었습니다. "
        "해당 셀부터 다시 실행하세요."
    )


def require_valid(stage: str) -> None:
    """결과를 소비하는 셀: 이 단계의 결과가 현재 입력·설정과 맞는지 확인합니다."""
    require_stages([stage])


def begin_stage(stage: str) -> None:
    """실행 셀 시작: 선행 결과만 확인합니다. 이 단계의 이전 스탬프는 보지 않습니다."""
    require_stages(STAGE_DEPENDENCIES[stage])


def complete_stage(stage: str) -> None:
    """실행 셀 끝: 이번 실행의 입력·선행 결과를 이 단계의 스탬프로 기록합니다."""
    STAGE_STAMPS[stage] = stage_state(stage)


def preview_stage(stage: str) -> str:
    """재로딩·미리보기 셀: 검토 대상의 현재 상태를 반환하고, 바뀌었으면 이전 승인을 무효화합니다."""
    begin_stage(stage)
    state = stage_state(stage)
    if stage in STAGE_STAMPS and STAGE_STAMPS[stage] != state:
        del STAGE_STAMPS[stage]
        print(
            "검토 대상이 바뀌어 이전 승인을 무효화했습니다. "
            + ("현재 미리보기를 확인한 뒤 6-C 승인 셀을 실행하세요." if stage == "body_review"
               else "확인 후 승인 셀을 다시 실행하세요.")
        )
    return state


def approve_stage(stage: str, approved: bool, reviewed_state: str) -> None:
    """승인 셀: 미승인이면 즉시 중단하고, 미리보기에서 확인한 상태만 승인합니다."""
    begin_stage(stage)
    if stage_state(stage) != reviewed_state:
        STAGE_STAMPS.pop(stage, None)
        raise RuntimeError(
            "미리보기 이후 검토 대상이 바뀌었습니다. "
            + (stage_rerun_hint(stage) if stage == "body_review"
               else "재로딩·미리보기 셀부터 다시 실행하세요.")
        )
    if approved:
        STAGE_STAMPS[stage] = reviewed_state
        print("승인됨:", STAGE_CELLS[stage])
    else:
        STAGE_STAMPS.pop(stage, None)
        raise RuntimeError(
            f"'{STAGE_CELLS[stage]}' 승인이 필요합니다. "
            "확인 후 승인값을 True로 바꿔 이 셀을 다시 실행하세요."
        )

## 1. 문서 설정

새 문서에 맞게 아래 값을 입력합니다. `PDF_PATH`는 이 노트북 폴더 기준이며 절대 경로도 지정할 수 있습니다.

- 목차가 있으면 `HAS_TOC=True`로 바꾸고 목차 페이지 범위를 지정합니다. 없으면 `False`로 두고 `BODY_START_PAGE`만 맞춥니다.
- 페이지 번호는 PDF 뷰어에 표시되는 1부터 시작하는 물리 페이지 번호입니다.
- 청크 ID의 문서키는 `DOCUMENT_NO` → `FROM_URL`의 번호(`brdNo`, `pRptNo` 등) 순으로 정합니다. 다음 셀이 바로 확인해 출력합니다.
- URL에서 번호를 찾지 못하면 다음 셀이 멈춥니다. `DOCUMENT_NO`를 직접 입력하거나 `USE_URL_FOR_DOC_KEY=False`(문서 제목 해시 사용)로 바꾸세요.
- `FROM_URL`이 비어 있는 문서는 문서 제목 해시를 문서키로 씁니다.
- 문서키 관련 값을 나중에 바꾸면 8번 청킹부터 다시 실행합니다.
- `BODY_EXCLUDE_PDF_PAGES`는 `body_pages.pdf`를 처음 만들 때만 적용합니다. 이후에는 `body_pages.pdf`를 직접 편집합니다.

In [89]:
PDF_PATH = PROJECT_ROOT / "26년무역통상진흥시책.pdf"

DOCUMENT_NAME = "26년무역통상진흥시책"
DOCUMENT_SECTION = "무역통상진흥시책"
DOCUMENT_COUNTRY = "대한민국"
DOCUMENT_NO: str | None = None # 별도 문서번호가 있으면 입력; 없으면 URL에서 추출
FROM_URL = "" # 원문 URL. 결과의 from_url 컬럼에 들어갑니다. 없으면 비워 두고 생성된 CSV에 나중에 추가해도 됨
USE_URL_FOR_DOC_KEY = True #url에서 문서번호를 추출하여 문서키로 사용합니다. (기본값: True)

HAS_TOC = True ##목차 기반으로 청킹이 필요하면 True로 변경 후 아래 페이지 정보 입력 (기본값: False)
TOC_START_PAGE = 2 #목차 시작 페이지
TOC_END_PAGE = 21 #목차 끝 페이지
BODY_START_PAGE = 25 #본문이 시작되는 페이지
BODY_EXCLUDE_PDF_PAGES: list[int] = [964] #본문에서 제외할 페이지 (공백, 로고, 불필요한 요약 등)

# preserve: 기존 body_pages.pdf 편집본 보존, 파일이 없으면 최초 생성
# rebuild: 원본 PDF와 위 페이지 설정으로 body_pages.pdf 재생성(페이지 설정을 바꾼 경우에만 사용)
BODY_PDF_BUILD_MODE = "preserve"  # preserve | rebuild

# macOS/Finder에서 복사한 한글은 자모 분리형(NFD)일 수 있습니다.
# 경로는 실제 파일시스템 표기를 유지하고, 출력용 문서 메타데이터만 NFC로 통일합니다.
DOCUMENT_NAME = unicodedata.normalize("NFC", DOCUMENT_NAME)
DOCUMENT_SECTION = unicodedata.normalize("NFC", DOCUMENT_SECTION)
DOCUMENT_COUNTRY = unicodedata.normalize("NFC", DOCUMENT_COUNTRY)

In [90]:
# 경로·파생값: 문서 설정을 바꾸면 이 셀부터 다시 실행합니다.
missing_settings = [
    name
    for name, value in {
        "DOCUMENT_NAME": DOCUMENT_NAME,
        "DOCUMENT_SECTION": DOCUMENT_SECTION,
        "DOCUMENT_COUNTRY": DOCUMENT_COUNTRY,
    }.items()
    if not value.strip()
]
if missing_settings:
    raise ValueError("1번 문서 설정을 먼저 입력하세요: " + ", ".join(missing_settings))
if not PDF_PATH.is_file():
    raise FileNotFoundError(f"1번 PDF_PATH를 실제 입력 PDF로 변경하세요: {PDF_PATH}")

OUTPUT_DIR = PDF_PATH.parent / f"{PDF_PATH.stem}_work"

TOC_PDF_PATH = OUTPUT_DIR / "toc_pages.pdf"
BODY_PDF_PATH = OUTPUT_DIR / "body_pages.pdf"
BODY_CROPPED_PDF_PATH = OUTPUT_DIR / "body_pages_cropped.pdf"
TOC_MARKDOWN_PATH = OUTPUT_DIR / "parsed_toc.md"
BODY_MARKDOWN_PATH = OUTPUT_DIR / "parsed_body.md"
TOC_PARSE_JSON_PATH = OUTPUT_DIR / "parsed_toc.json"
BODY_PARSE_JSON_PATH = OUTPUT_DIR / "parsed_body.json"
PARSE_MANIFEST_PATH = OUTPUT_DIR / "parse_manifest.json"

TOC_TITLES_EXTRACTED_JSON_PATH = OUTPUT_DIR / "toc_titles_extracted.json"
TOC_TITLES_REVIEWED_MD_PATH = OUTPUT_DIR / "toc_titles_reviewed.md"
# 이전 버전의 검토 파일. 있으면 처음 한 번 toc_titles_reviewed.md로 옮겨 씁니다.
TOC_TITLES_REVIEWED_JSON_PATH = OUTPUT_DIR / "toc_titles_reviewed.json"

BODY_REFINED_MARKDOWN_PATH = OUTPUT_DIR / "parsed_body_refined.md"
BODY_NOISE_REPORT_CSV = OUTPUT_DIR / "body_noise_report.csv"
BODY_REFINEMENT_MANIFEST_PATH = OUTPUT_DIR / "body_refinement_manifest.json"

BOUNDARY_COVERAGE_CSV = OUTPUT_DIR / "boundary_title_coverage.csv"
BODY_CHUNKS_CSV = OUTPUT_DIR / "body_chunks.csv"
DROPPED_SHORT_CHUNKS_CSV = OUTPUT_DIR / "dropped_short_chunks.csv"
TABLE_CHUNKS_CSV = OUTPUT_DIR / "table_chunks.csv"
TABLE_SUMMARIES_CSV = OUTPUT_DIR / "table_summaries.csv"
TABLE_SUMMARY_CACHE_JSON = OUTPUT_DIR / "table_summary_cache.json"
PREFIX_RESULTS_CSV = OUTPUT_DIR / "prefix_results.csv"
PREFIX_CACHE_JSON = OUTPUT_DIR / "prefix_cache.json"
FINAL_CHUNKS_CSV = OUTPUT_DIR / "final_chunks.csv"
FINAL_OUTPUT_CSV = PDF_PATH.with_suffix(".csv")
FINAL_QA_CSV = OUTPUT_DIR / "final_qa.csv"

TOC_PARSING_PROMPT = f"""이 PDF는 '{DOCUMENT_NAME}'의 목차 페이지입니다.
- 목차에 보이는 장, 절, 하위 항목을 누락하지 마세요.
- 계층이 보이도록 Markdown 헤더 또는 목록으로 표현하세요.
- 제목 번호와 페이지 번호를 유지하세요.
- 장식 이미지, 페이지 머리말, 바닥글은 제외하세요.
""".strip()


BODY_PARSING_PROMPT = f"""이 PDF는 '{DOCUMENT_NAME}'의 본문입니다.
- 요약하지 말고 원문을 그대로 전사하세요.
- 제목의 단어, 숫자, 괄호, 띄어쓰기를 임의로 바꾸지 마세요.
- 원문에 없는 제목을 만들지 마세요.
- 장, 절, 항목 제목은 Markdown 헤더로 구분하세요.
- 표는 행과 열 구조를 유지한 Markdown 표로 보존하세요.
- 기업·기관 로고와 로고 안의 회사명·슬로건·영문 표기는 전사하지 마세요.
- 이미지 설명으로 생성되는 `logo:` 문구도 제외하세요.
- 반복 페이지 머리말, 바닥글, 단독 페이지 번호만 제외하세요.
""".strip()

total_pdf_pages = pdf_page_count(PDF_PATH)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
complete_stage("paths")

print("notebook directory:", PROJECT_ROOT)
print("source PDF:", PDF_PATH, f"({total_pdf_pages} pages)")
print("document:", DOCUMENT_NAME)
print("output:", OUTPUT_DIR)
print("TOC pages:", f"{TOC_START_PAGE}-{TOC_END_PAGE}" if HAS_TOC else "없음 — 본문 기준")
print("body pages:", f"{BODY_START_PAGE}-end")
if not FROM_URL.strip():
    print("FROM_URL이 비어 있습니다: from_url 컬럼은 비워 둡니다.")
document_key, document_key_source = resolve_document_key()
print("document key:", document_key, f"({document_key_source})")

notebook directory: /Users/main/Downloads/Kotra_Chunking-main
source PDF: /Users/main/Downloads/Kotra_Chunking-main/26년무역통상진흥시책.pdf (964 pages)
document: 26년무역통상진흥시책
output: /Users/main/Downloads/Kotra_Chunking-main/26년무역통상진흥시책_work
TOC pages: 2-21
body pages: 25-end
FROM_URL이 비어 있습니다: from_url 컬럼은 비워 둡니다.
document key: 699775b8453f (문서 제목 해시)


## 2. 본문 PDF 준비·검토

목차가 있으면 목차 PDF를 분리하고, 본문 PDF(`body_pages.pdf`)를 만듭니다.
`body_pages.pdf`를 열어 불필요한 페이지를 직접 삭제·저장한 뒤 재로딩 셀을 실행하고 승인합니다.

In [91]:
# 본문 PDF 준비 함수

def source_body_pages(total_pages: int) -> list[int]:
    excluded = set(BODY_EXCLUDE_PDF_PAGES)
    return [
        page
        for page in range(BODY_START_PAGE, total_pages + 1)
        if page not in excluded
    ]

def validate_pdf_settings(total_pages: int) -> None:
    if not isinstance(HAS_TOC, bool):
        raise ValueError("HAS_TOC는 True 또는 False여야 합니다.")
    if not isinstance(BODY_START_PAGE, int) or not 1 <= BODY_START_PAGE <= total_pages:
        raise ValueError(f"본문 시작 페이지 범위 오류: {BODY_START_PAGE}")
    if HAS_TOC:
        if not (
            isinstance(TOC_START_PAGE, int) and isinstance(TOC_END_PAGE, int)
            and 1 <= TOC_START_PAGE <= TOC_END_PAGE <= total_pages
        ):
            raise ValueError(f"목차 페이지 범위 오류: {TOC_START_PAGE}-{TOC_END_PAGE}")
        if BODY_START_PAGE <= TOC_END_PAGE:
            raise ValueError("본문 시작 페이지는 목차 뒤여야 합니다.")
    invalid_excluded = [
        page
        for page in BODY_EXCLUDE_PDF_PAGES
        if not BODY_START_PAGE <= page <= total_pages
    ]
    if invalid_excluded:
        raise ValueError(f"본문 범위 밖 제외 페이지: {invalid_excluded}")

def body_pdf_generation_settings(total_pages: int) -> dict[str, object]:
    return {
        "source_pdf": str(PDF_PATH.resolve()),
        "source_sha256": file_sha256(PDF_PATH),
        "total_pages": total_pages,
        "body_start_page": BODY_START_PAGE,
        "body_exclude_pdf_pages": BODY_EXCLUDE_PDF_PAGES,
    }

def validate_preserved_body_pdf_settings(
    cached_manifest: object,
    expected_settings: dict[str, object],
) -> None:
    if not isinstance(cached_manifest, dict):
        print(
            "기존 body_pages.pdf를 보존합니다. 생성 이력이 없으므로 "
            "원본·페이지 설정을 바꿨다면 BODY_PDF_BUILD_MODE='rebuild'를 사용하세요."
        )
        return
    mismatched = [
        key
        for key, expected in expected_settings.items()
        if cached_manifest.get(key) != expected
    ]
    if mismatched:
        raise RuntimeError(
            "body_pages.pdf 생성 설정이 이전 파싱과 다릅니다: "
            f"{mismatched}. 설정을 반영하려면 "
            "BODY_PDF_BUILD_MODE='rebuild'로 재생성하세요."
        )

def prepare_body_pdf(total_pages: int, mode: str) -> str:
    normalized_mode = mode.strip().lower()
    if normalized_mode not in {"preserve", "rebuild"}:
        raise ValueError("BODY_PDF_BUILD_MODE는 preserve 또는 rebuild여야 합니다.")
    should_rebuild = normalized_mode == "rebuild" or not BODY_PDF_PATH.exists()
    if should_rebuild:
        write_pdf_segment(
            PDF_PATH,
            BODY_PDF_PATH,
            BODY_START_PAGE,
            total_pages,
            excluded_pages=set(BODY_EXCLUDE_PDF_PAGES),
        )
        expected_page_count = len(source_body_pages(total_pages))
        actual_page_count = pdf_page_count(BODY_PDF_PATH)
        if actual_page_count != expected_page_count:
            raise RuntimeError(
                "body_pages.pdf 최초 생성 페이지 수가 예상과 다릅니다: "
                f"expected={expected_page_count}, actual={actual_page_count}"
            )
        return "rebuilt" if normalized_mode == "rebuild" else "created"
    validate_preserved_body_pdf_settings(
        load_json(PARSE_MANIFEST_PATH, default=None),
        body_pdf_generation_settings(total_pages),
    )
    return "preserved_manual_review_file"

def write_pdf_segment(
    source_path: Path,
    output_path: Path,
    start_page: int,
    end_page: int,
    excluded_pages: set[int] | None = None,
) -> None:
    from pypdf import PdfReader, PdfWriter

    reader = PdfReader(str(source_path))
    writer = PdfWriter()
    excluded_pages = excluded_pages or set()
    for physical_page in range(start_page, end_page + 1):
        if physical_page not in excluded_pages:
            writer.add_page(reader.pages[physical_page - 1])
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("wb") as output_file:
        writer.write(output_file)

In [92]:
begin_stage("body_pdf_file")
validate_pdf_settings(total_pdf_pages)
if HAS_TOC:
    write_pdf_segment(
        PDF_PATH, TOC_PDF_PATH, TOC_START_PAGE, TOC_END_PAGE,
    )
    print("TOC PDF:", TOC_PDF_PATH, pdf_page_count(TOC_PDF_PATH))
else:
    print("목차 PDF 분리를 건너뜁니다.")
body_pdf_status = prepare_body_pdf(total_pdf_pages, BODY_PDF_BUILD_MODE)
complete_stage("body_pdf_file")

print("body PDF:", BODY_PDF_PATH)
print("body PDF status:", body_pdf_status)
show_user_checkpoint(
    "본문 PDF 페이지 검토",
    [
        f"편집할 PDF: `{BODY_PDF_PATH}`",
        "불필요한 페이지를 PDF에서 직접 삭제해도 됩니다.",
        "BODY_PDF_BUILD_MODE='preserve'이면 재실행해도 편집본을 덮어쓰지 않습니다.",
    ],
    "페이지 편집을 끝낸 뒤 아래 재로딩·미리보기 셀을 실행하세요.",
)

TOC PDF: /Users/main/Downloads/Kotra_Chunking-main/26년무역통상진흥시책_work/toc_pages.pdf 20
body PDF: /Users/main/Downloads/Kotra_Chunking-main/26년무역통상진흥시책_work/body_pages.pdf
body PDF status: preserved_manual_review_file


### 사용자 확인: 본문 PDF 페이지 검토

- 편집할 PDF: `/Users/main/Downloads/Kotra_Chunking-main/26년무역통상진흥시책_work/body_pages.pdf`
- 불필요한 페이지를 PDF에서 직접 삭제해도 됩니다.
- BODY_PDF_BUILD_MODE='preserve'이면 재실행해도 편집본을 덮어쓰지 않습니다.

**다음 작업:** 페이지 편집을 끝낸 뒤 아래 재로딩·미리보기 셀을 실행하세요.

In [93]:
# 재로딩·미리보기: body_pages.pdf를 편집했으면 이 셀부터 다시 실행합니다.
body_pdf_page_count = pdf_page_count(BODY_PDF_PATH)
if body_pdf_page_count < 1:
    raise RuntimeError("body_pages.pdf에 페이지가 없습니다.")
body_pdf_review_state = preview_stage("body_pdf")
print("reviewed body PDF:", BODY_PDF_PATH)
print("reviewed body pages:", body_pdf_page_count)
print("reviewed body SHA-256:", file_sha256(BODY_PDF_PATH))

reviewed body PDF: /Users/main/Downloads/Kotra_Chunking-main/26년무역통상진흥시책_work/body_pages.pdf
reviewed body pages: 939
reviewed body SHA-256: fab827d9c55436a9f055fd21071c56beb1657b38fed69b10d0958c970f4138ad


In [94]:
BODY_PDF_REVIEW_APPROVED = True
approve_stage("body_pdf", BODY_PDF_REVIEW_APPROVED, body_pdf_review_state)

승인됨: 2. 본문 PDF 재로딩·미리보기와 승인


## 3. 선택적 우측 크롭

본문 페이지 오른쪽에 반복 인쇄된 목차가 파싱 노이즈로 섞일 때만 켭니다.
크롭을 끄면 승인된 `body_pages.pdf`가 그대로 파싱 입력이 됩니다.

In [95]:
# 크롭·미리보기 함수

def validate_crop_settings() -> None:
    if not isinstance(BODY_CROP_ENABLED, bool):
        raise ValueError("BODY_CROP_ENABLED는 True 또는 False여야 합니다.")
    if not 0 <= BODY_CROP_RIGHT_RATIO < 0.5:
        raise ValueError(
            "BODY_CROP_RIGHT_RATIO는 0 이상 0.5 미만이어야 합니다: "
            f"{BODY_CROP_RIGHT_RATIO}"
        )

def crop_pdf_right_margin(
    input_pdf: Path,
    output_pdf: Path,
    right_ratio: float,
) -> Path:
    import fitz

    output_pdf.parent.mkdir(parents=True, exist_ok=True)
    with fitz.open(input_pdf) as source:
        result = fitz.open()
        for page_number, page in enumerate(source):
            rect = page.rect
            clip = fitz.Rect(
                rect.x0,
                rect.y0,
                rect.x1 - rect.width * right_ratio,
                rect.y1,
            )
            target = result.new_page(width=clip.width, height=clip.height)
            target.show_pdf_page(target.rect, source, page_number, clip=clip)
        result.save(output_pdf, garbage=4, deflate=True)
        result.close()
    return output_pdf

def cropped_pdf_is_current() -> bool:
    """이미 파싱에 사용한 크롭본이 현재 본문 PDF·비율로 만든 것인지 확인합니다.

    크롭 결과는 만들 때마다 바이트가 달라지므로, 같은 조건이면 다시 만들지 않아야
    파싱 캐시(parse_manifest.json의 입력 지문)가 유지됩니다.
    """
    manifest = load_json(PARSE_MANIFEST_PATH, default=None)
    return (
        isinstance(manifest, dict)
        and BODY_CROPPED_PDF_PATH.exists()
        and manifest.get("body_crop_enabled") is True
        and manifest.get("body_crop_right_ratio") == BODY_CROP_RIGHT_RATIO
        and manifest.get("body_pdf_sha256") == file_sha256(BODY_PDF_PATH)
        and manifest.get("body_parse_input_sha256") == file_sha256(BODY_CROPPED_PDF_PATH)
    )

def render_pdf_page_png(
    pdf_path: Path,
    page_index: int,
    zoom: float = 1.2,
) -> bytes:
    import fitz

    with fitz.open(pdf_path) as document:
        page = document[page_index]
        pixmap = page.get_pixmap(
            matrix=fitz.Matrix(zoom, zoom),
            alpha=False,
        )
        return pixmap.tobytes("png")

def selected_crop_preview_body_pages(body_page_count: int) -> list[int]:
    if body_page_count < 1:
        return []
    selected = CROP_PREVIEW_BODY_PAGES or [
        1,
        (body_page_count + 1) // 2,
        body_page_count,
    ]
    selected = list(dict.fromkeys(selected))
    invalid = [page for page in selected if not 1 <= page <= body_page_count]
    if invalid:
        raise ValueError(f"body_pages.pdf 범위 밖 크롭 미리보기: {invalid}")
    return selected

In [96]:
BODY_CROP_ENABLED = False
BODY_CROP_RIGHT_RATIO = 0.08
# 편집된 body_pages.pdf 기준입니다. 비어 있으면 첫/중간/마지막을 선택합니다.
CROP_PREVIEW_BODY_PAGES: list[int] = []

In [97]:
begin_stage("crop")
validate_crop_settings()
if BODY_CROP_ENABLED:
    if cropped_pdf_is_current():
        print("기존 크롭본 재사용: 본문 PDF와 크롭 비율이 이전 파싱 때와 같습니다.")
    else:
        crop_pdf_right_margin(
            BODY_PDF_PATH,
            BODY_CROPPED_PDF_PATH,
            BODY_CROP_RIGHT_RATIO,
        )
    if pdf_page_count(BODY_CROPPED_PDF_PATH) != body_pdf_page_count:
        raise RuntimeError("크롭 전후 본문 PDF 페이지 수가 다릅니다.")
    BODY_PARSE_INPUT_PATH = BODY_CROPPED_PDF_PATH
else:
    BODY_PARSE_INPUT_PATH = BODY_PDF_PATH
complete_stage("crop")

print("crop enabled:", BODY_CROP_ENABLED)
print("crop right ratio:", f"{BODY_CROP_RIGHT_RATIO:.1%}")
print("body parse input:", BODY_PARSE_INPUT_PATH)

crop enabled: False
crop right ratio: 8.0%
body parse input: /Users/main/Downloads/Kotra_Chunking-main/26년무역통상진흥시책_work/body_pages.pdf


In [98]:
require_valid("crop")
if BODY_CROP_ENABLED:
    preview_pages = selected_crop_preview_body_pages(body_pdf_page_count)
    for body_page in preview_pages:
        print(
            f"body PDF {body_page}페이지 — 편집본(위) / "
            f"우측 {BODY_CROP_RIGHT_RATIO:.1%} 크롭(아래)"
        )
        display(
            Image(
                data=render_pdf_page_png(BODY_PDF_PATH, body_page - 1),
                width=430,
            )
        )
        display(
            Image(
                data=render_pdf_page_png(
                    BODY_CROPPED_PDF_PATH,
                    body_page - 1,
                ),
                width=395,
            )
        )
    show_user_checkpoint(
        "본문 우측 크롭",
        [
            f"우측 {BODY_CROP_RIGHT_RATIO:.1%}가 제거됐습니다.",
            "반복 목차가 제거됐는지 확인하세요.",
            "본문과 표의 오른쪽 내용이 잘리지 않았는지 확인하세요.",
        ],
        "문제가 있으면 크롭 비율을 수정하고 크롭 실행 셀부터 다시 실행하세요.",
    )
else:
    print("본문 크롭 비활성화: 승인된 body_pages.pdf를 파싱 입력으로 사용합니다.")

본문 크롭 비활성화: 승인된 body_pages.pdf를 파싱 입력으로 사용합니다.


## 4. LlamaParse 파싱

본문을 파싱하고, 목차가 있으면 목차도 함께 파싱합니다. 입력과 설정이 같으면 캐시를 재사용합니다.

In [99]:
# 파싱·페이지 구분 함수

PAGE_BREAK_MARKER = "<!-- KOTRA_PAGE_BREAK -->"

PAGE_BREAK_RE = re.compile(
    rf"(?m)^[ \t]*{re.escape(PAGE_BREAK_MARKER)}[ \t]*$"
)

def split_markdown_pages(markdown_text: str) -> list[str]:
    """Split only pipeline page markers; Markdown horizontal rules stay intact."""
    return [page.strip() for page in PAGE_BREAK_RE.split(markdown_text)]

def join_markdown_pages(pages: Iterable[object]) -> str:
    separator = f"\n\n{PAGE_BREAK_MARKER}\n\n"
    return separator.join(str(page or "").strip() for page in pages).strip()

def markdown_from_llamaparse_result(parse_result: object) -> str:
    markdown_result = getattr(parse_result, "markdown", None)
    if markdown_result is None:
        raise RuntimeError("LlamaParse Markdown 결과가 없습니다.")

    markdown_pages: list[str] = []
    failed_pages: list[int | str] = []
    for page in markdown_result.pages:
        if not getattr(page, "success", False):
            failed_pages.append(getattr(page, "page_number", "unknown"))
            continue
        markdown_pages.append(str(page.markdown).strip())

    if failed_pages:
        raise RuntimeError(f"LlamaParse 페이지 파싱 실패: {failed_pages}")
    if not markdown_pages or not any(markdown_pages):
        raise RuntimeError("LlamaParse Markdown 결과가 비어 있습니다.")
    return join_markdown_pages(markdown_pages)

def parse_pdf_with_llamaparse(
    pdf_path: Path,
    prompt: str,
) -> tuple[str, object]:
    load_dotenv(ENV_PATH)
    api_key = os.getenv("LLAMA_CLOUD_API_KEY")
    if not api_key:
        raise RuntimeError("LLAMA_CLOUD_API_KEY가 필요합니다.")
    from llama_cloud import LlamaCloud

    tier = LLAMA_PARSE_TIER.strip().lower()
    valid_tiers = {"fast", "cost_effective", "agentic", "agentic_plus"}
    if tier not in valid_tiers:
        raise ValueError(f"지원하지 않는 LlamaParse tier입니다: {tier!r}")

    parse_options: dict[str, object] = {
        "upload_file": pdf_path,
        "tier": tier,
        "version": LLAMA_PARSE_VERSION,
        "expand": ["markdown"],
        "processing_options": {
            "ocr_parameters": {"languages": LLAMA_PARSE_LANGUAGES},
        },
        "output_options": {
            "markdown": {
                "tables": {"output_tables_as_markdown": True},
            }
        },
        "verbose": True,
    }
    if tier in {"agentic", "agentic_plus"}:
        parse_options["agentic_options"] = {"custom_prompt": prompt}

    client = LlamaCloud(api_key=api_key)
    parse_result = client.parsing.parse(**parse_options)
    markdown = markdown_from_llamaparse_result(parse_result)
    json_data = parse_result.model_dump(mode="json")
    print(
        "LlamaParse completed:",
        f"tier={parse_result.job.tier or tier}",
        f"version={LLAMA_PARSE_VERSION}",
        f"job_id={parse_result.job.id}",
    )
    return markdown, json_data

def current_parse_manifest() -> dict[str, object]:
    return {
        "source_pdf": str(PDF_PATH.resolve()),
        "source_sha256": file_sha256(PDF_PATH),
        "total_pages": total_pdf_pages,
        "has_toc": HAS_TOC,
        "toc_start_page": TOC_START_PAGE if HAS_TOC else None,
        "toc_end_page": TOC_END_PAGE if HAS_TOC else None,
        "body_start_page": BODY_START_PAGE,
        "body_exclude_pdf_pages": BODY_EXCLUDE_PDF_PAGES,
        "body_pdf_sha256": file_sha256(BODY_PDF_PATH),
        "body_pdf_pages": body_pdf_page_count,
        "body_page_number_basis": "reviewed_body_pages_pdf",
        "body_crop_enabled": BODY_CROP_ENABLED,
        "body_crop_right_ratio": BODY_CROP_RIGHT_RATIO,
        "body_parse_input": BODY_PARSE_INPUT_PATH.name,
        "body_parse_input_sha256": file_sha256(BODY_PARSE_INPUT_PATH),
        "llama_cloud_sdk_version": importlib.metadata.version("llama-cloud"),
        "llama_parse_tier": LLAMA_PARSE_TIER,
        "llama_parse_version": LLAMA_PARSE_VERSION,
        "llama_parse_languages": LLAMA_PARSE_LANGUAGES,
        "toc_prompt": TOC_PARSING_PROMPT,
        "body_prompt": BODY_PARSING_PROMPT,
    }

def decode_markdown_html_entities(markdown_path: Path) -> int:
    markdown = markdown_path.read_text(encoding="utf-8")
    count = len(HTML_ENTITY_PATTERN.findall(markdown))
    decoded = html.unescape(markdown)
    if decoded != markdown:
        markdown_path.write_text(decoded, encoding="utf-8")
    return count

MARKDOWN_TABLE_SEPARATOR_RE = re.compile(
    r"^\s*\|(?:\s*:?-+:?\s*\|)+\s*$"
)

In [100]:
PDF_PARSE_MODE = "reuse"  # reuse | rebuild
LLAMA_PARSE_TIER = "agentic"  # fast | cost_effective | agentic | agentic_plus
LLAMA_PARSE_VERSION = "latest"
LLAMA_PARSE_LANGUAGES = ["ko", "en"]

In [101]:
begin_stage("parse_body")
begin_stage("parse_toc")
normalized_parse_mode = PDF_PARSE_MODE.strip().lower()
if normalized_parse_mode not in {"reuse", "rebuild"}:
    raise ValueError("PDF_PARSE_MODE는 reuse 또는 rebuild여야 합니다.")

parse_manifest = current_parse_manifest()
cached_manifest = load_json(PARSE_MANIFEST_PATH, default=None)
needs_pdf_parse = (
    normalized_parse_mode == "rebuild"
    or cached_manifest != parse_manifest
    or (HAS_TOC and not TOC_MARKDOWN_PATH.exists())
    or not BODY_MARKDOWN_PATH.exists()
    or (HAS_TOC and not TOC_PARSE_JSON_PATH.exists())
    or not BODY_PARSE_JSON_PATH.exists()
)

if needs_pdf_parse:
    if HAS_TOC:
        print("LlamaParse TOC:", TOC_PDF_PATH)
        toc_markdown, toc_json = parse_pdf_with_llamaparse(
            TOC_PDF_PATH,
            TOC_PARSING_PROMPT,
        )
        TOC_MARKDOWN_PATH.write_text(toc_markdown, encoding="utf-8")
        save_json(toc_json, TOC_PARSE_JSON_PATH)

    print("LlamaParse body:", BODY_PARSE_INPUT_PATH)
    body_markdown, body_json = parse_pdf_with_llamaparse(
        BODY_PARSE_INPUT_PATH,
        BODY_PARSING_PROMPT,
    )
    BODY_MARKDOWN_PATH.write_text(body_markdown, encoding="utf-8")
    save_json(body_json, BODY_PARSE_JSON_PATH)
    save_json(parse_manifest, PARSE_MANIFEST_PATH)
else:
    print("현재 설정과 일치하는 LlamaParse 캐시를 사용합니다.")

parsed_body_page_count = len(
    split_markdown_pages(BODY_MARKDOWN_PATH.read_text(encoding="utf-8"))
)
if parsed_body_page_count != body_pdf_page_count:
    raise RuntimeError(
        "parsed_body.md와 편집된 body_pages.pdf 페이지 수가 다릅니다: "
        f"Markdown={parsed_body_page_count}, body PDF={body_pdf_page_count}. "
        "PDF_PARSE_MODE='rebuild'로 다시 파싱하세요."
    )

toc_entities = decode_markdown_html_entities(TOC_MARKDOWN_PATH) if HAS_TOC else 0
body_entities = decode_markdown_html_entities(BODY_MARKDOWN_PATH)
complete_stage("parse_body")
complete_stage("parse_toc")
print("body parse input:", BODY_PARSE_INPUT_PATH)
print("HTML entities decoded — TOC:", toc_entities, "body:", body_entities)

현재 설정과 일치하는 LlamaParse 캐시를 사용합니다.
body parse input: /Users/main/Downloads/Kotra_Chunking-main/26년무역통상진흥시책_work/body_pages.pdf
HTML entities decoded — TOC: 0 body: 0


In [102]:
require_valid("parse_body")
require_valid("parse_toc")
if HAS_TOC:
    toc_markdown_preview = TOC_MARKDOWN_PATH.read_text(encoding="utf-8")
    display(Markdown("### 추출된 목차 Markdown\n\n" + toc_markdown_preview))
else:
    print("목차 없음: 본문만 확인합니다.")

body_markdown_preview = BODY_MARKDOWN_PATH.read_text(encoding="utf-8")
body_heading_preview = [
    line.strip()
    for line in body_markdown_preview.splitlines()
    if re.match(r"^#{1,6}\s+", line)
]
print("body parsed pages:", len(split_markdown_pages(body_markdown_preview)))
print("body headings:", len(body_heading_preview))
print(
    "body table candidates:",
    sum(
        bool(MARKDOWN_TABLE_SEPARATOR_RE.match(line))
        for line in body_markdown_preview.splitlines()
    ),
)
display(pd.DataFrame({"heading": body_heading_preview}).head(40))
display(Markdown("### 추출된 본문 Markdown (앞 12,000자)\n\n" + body_markdown_preview[:12000]))

show_user_checkpoint(
    "LlamaParse 결과",
    [
        f"목차 Markdown: `{TOC_MARKDOWN_PATH}`" if HAS_TOC else "목차 없음: 본문만 파싱했습니다.",
        f"본문 Markdown: `{BODY_MARKDOWN_PATH}`",
        f"본문 입력: `{BODY_PARSE_INPUT_PATH}`",
    ],
    ("5번에서 목차를 검토한 뒤 6-A 정제본 생성·유지로 진행하세요." if HAS_TOC else
     "아래 셀을 순서대로 실행하세요. 5번 목차 단계는 자동으로 통과합니다."),
)

### 추출된 목차 Markdown

# Contents

## Ⅰ '25년 수출 성과 및 평가 _ 001

## Ⅱ '26년 수출여건 및 도전과제 _ 005

## Ⅲ '26년 범부처 수출확대 전략 _ 009

1. 추진 방향 _ 011
2. 주요 내용 _ 012
3. 향후 계획 _ 028

<!-- KOTRA_PAGE_BREAK -->

## Ⅳ 각 기관별 세부 추진사업 _ 029

### 01 산업통상부

1. 대한무역투자진흥공사 (www.kotra.or.kr) 33
    1. 글로벌 통상애로 지원 사업 38
    2. 수출 및 투자진출 상담 40
    3. 글로벌 역량 진단(GCL) 42
    4. 수출24 글로벌 대행 서비스 (해외시장조사) 44
    5. 해외 비즈니스 출장 지원 47
    6. 열린무역관 49
    7. 이동KOTRA 51
    8. 지방무역사절단 53
    9. 수출바우처 사업 55
    10. 해외지사화사업 58
    11. 해외공동물류센터사업 60
    12. 수출인큐베이터 63
    13. 세계일류상품 육성사업 65
    14. 중견기업 글로벌 지원사업 68
    15. 해외취업지원사업 71
    16. 해외전문인력 유치 지원 72
    17. 내수기업 수출기업화 74
    18. 전문무역상사 활용 내수·초보기업 지원사업 76
    19. CES 혁신상 수상지원 사업 77
    20. 스타트업 주요 해외전시회 참가지원 사업 78
    21. K-수출스타 500 79
    22. 한국우수상품전 81
    23. SEOUL FOOD(서울푸드) 83

<!-- KOTRA_PAGE_BREAK -->

# Contents

| 24. 해외전시회 단체참가 지원사업                                | 85  |
| -------------------------------------------------- | --- |
| 25. 해외전시회 개별참가 지원사업                                | 86  |
| 26. 해외바이어 초청 수출상담회                                 | 87  |
| 27. 서비스거점 지원 사업                                    | 88  |
| 28. 서비스 브랜드 마케팅(BM) 해외진출 지원사업                      | 90  |
| 29. 스마트팜 해외수출지원                                    | 92  |
| 30. 스마트시티 해외수출지원                                   | 94  |
| 31. 스마트팜 컨소시엄 수출지원                                 | 96  |
| 32. 해외 프로젝트 수주 지원                                  | 98  |
| 33. 해외공공조달 지원 데스크                                  | 99  |
| 34. 파워셀러 육성사업                                      | 100 |
| 35. 한류박람회(KBEE)                                    | 102 |
| 36. 1무역관 1유통망 협력사업                                 | 103 |
| 37. AI정부 GovTech 해외진출지원                            | 104 |
| 38. 글로벌 의료기기 수출상담회(Global Medical Equipment Plaza) | 105 |
| 39. 글로벌 의약품 수출상담회(Global Bio\&Pharma Plaza)        | 107 |
| 40. 산업AI 전시회                                       | 109 |
| 41. AI 해외진출지원                                      | 111 |
| 42. 글로벌 파트너링(Global Partnering) 사업                 | 112 |
| 43. 해외 GP센터(Korea Global Partnering Center)        | 114 |
| 44. 해외 수요처 연계 기술개발사업 R\&D 수요 검증                    | 116 |
| 45. 소부장 공급망안정 종합지원 사업                              | 118 |
| 46. 경제외교활용포털 운영                                    | 120 |
| 47. 2027 세계시장 진출전략 설명회                             | 121 |
| 48. FTA 해외활용지원센터                                   | 122 |
| 49. 글로벌 신통상 포럼                                     | 124 |
| 50. KOTRA 해외시장뉴스                                   | 125 |
| 51. buyKOREA (B2B/이커머스 플랫폼 활용)                     | 126 |
| 52. AI 무역지원센터(AI Trade Support Center)             | 127 |
| 53. 빅데이터 플랫폼(트라이빅)                                 | 129 |
| 54. 해외경제정보드림 (해드림)                                 | 130 |
| 55. 중소중견기업 글로벌 역량 강화 교육연수 사업                       | 131 |

<!-- KOTRA_PAGE_BREAK -->

56. 한국투자기업지원센터 133
57. 글로벌 M&A 지원 134
58. 국내복귀기업 지원 135
59. 방산물자·정부간(G2G) 거래 교역지원 137

## 2. 한국무역보험공사 (www.ksure.or.kr) 139
1. 수출기업의 유동성 지원을 위한 수출신용보증 140
2. 수출대금 미회수 위험 커버를 위한 단기수출보험 142
3. 중장기 수출계약의 금융 지원을 위한 중장기성 수출보험 144
4. 안정적인 수입거래를 지원하기 위한 수입보험 146
5. 환율변동에 따른 위험을 헤지하는 환변동보험 148

## 3. 한국무역협회 (www.kita.net) 150
1. 무역업계 규제·애로 개선 건의 152
2. 무역현장 자문서비스 153
3. KITA무역진흥자금 융자 155
4. TradePro (디지털비대면 분야별 전문가 상담 서비스) 157
5. 무역분야 외국인 사증(VISA) 발급 추천 158
6. APEC 기업인 여행카드(ABTC) 발급 160
7. 용역 및 전자적 무체물, 선용품 공급 실적 증명발급 163
8. 전문무역상사-제조기업 매칭 수출상담회 164
9. KITA 해외바이어 초청 B2B 수출상담회 166
10. tradeKorea를 통한 B2B 온라인 해외마케팅 지원 168
11. 국내 전문전시회 개최 170
12. 해외전시회 참가지원 172
13. 글로벌 K-프리미엄 스토어 운영 174
14. K-프리미엄 스페이스 조성 및 운영 175
15. 해외네트워크 활용 해외시장 진출 지원 177
16. 중국, 일본, 아주 지역 진출 지원 179
17. 국내 대·중견기업·공공기관-스타트업 오픈이노베이션 181
18. 스타트업 브랜치 장소 운영 183
19. 스타트업 해외시장 진출 지원 185

<!-- KOTRA_PAGE_BREAK -->

# Contents

20. 무역아카데미 교육 서비스 제공 187
21. 지역특화 청년무역전문가 양성사업 (GTEP) 189
22. 비관세장벽(해외인증·지재권) 대응 지원 사업 191
23. OK FTA 현장 방문 컨설팅 194
24. 차이나데스크 현장방문컨설팅 지원 197
25. 찾아가는 FTA 서비스 199

## 4. 한국전시산업진흥회 (www.akei.or.kr) 201
1. 전시산업경쟁력강화지원 202

## 5. 한국무역정보통신 (www.KTNET.co.kr) 204
1. 원산지관리시스템 종합컨설팅 사업 205
2. 간접수출(구매확인서, 내국신용장) 실적증명서비스 207
3. O/A 수출채권 전자매입 서비스 208
4. 전자수입화물선취보증(e-L/G) 서비스 209
5. 수출신용장 통지 전자화 서비스 210
6. 구매확인서 서비스 211
7. 디지털물류 서비스 212
8. 전자상거래 수출업무지원 서비스 213
9. 전자무역데이터 서비스 214
10. 스마트 무역원장 서비스 215
11. 디지털문서유통 서비스 216

## 6. 세계한인경제무역협회 (www.okta.net) 217
1. 교포무역인네트워크 218
2. 해외지사화사업 224
3. 글로벌 취업지원사업 226
4. 수출바우처사업 229
5. JEC WORLD 2026 탄소기업 해외진출 지원사업 231
6. 탄소기업 해외진출 지원사업 233
7. 지역기업 성장사다리 지원사업 235
8. 경기청년 해외 취·창업 기회 확충사업 237

<!-- KOTRA_PAGE_BREAK -->

7. 한국수입협회 (www.koima.or.kr) 239
    1. 해외 원·부자재 구매사절단 파견 및 바이어 초청 프로그램 240
    2. 수입 DB 운영 241
    3. 무역 전시회 개최 (한국수입엑스포) 242
    4. 국제원자재 수입가격정보 제공 243
    5. 수입관리사 자격시험제도 및 무역 연수 244
    6. 해외 시장조사 및 거래선 발굴 서비스 245
    7. 국내 중소 수입업체들의 해외 비즈니스 활동 지원 246

8. 한국플랜트산업협회 (www.kopia.or.kr) 247
    1. 해외플랜트 시장개척 지원 248
    2. 플랜트 해외진출 플랫폼 운영 250

## 02 중소벤처기업부

1. 중소벤처기업부 (www.mss.go.kr) 253
    1. 수출바우처 254
    2. 대·중소기업 동반진출 지원사업 257
    3. 수출컨소시엄 사업 259
    4. 전자상거래수출시장진출 261
    5. 해외수출규제대응지원(해외규격인증획득지원) 263
    6. 글로벌비즈니스센터(GBC) 265

2. 중소벤처기업진흥공단 (www.kosmes.or.kr) 267
    1. 수출바우처사업 268
    2. 글로벌비즈니스센터(GBC) 270
    3. 코리아스타트업센터(KSC) 272
    4. 해외지사화사업 274
    5. 신시장진출지원자금 276
    6. 전자상거래 수출시장 진출지원 278
    7. 온라인수출플랫폼 280
    8. 통상변화대응지원사업 282

<!-- KOTRA_PAGE_BREAK -->

# Contents

## 03 재정경제부

* 1. 한국수출입은행 (www.koreaexim.go.kr) 287
    * 1. 대 출 288
    * 2. 보 증 291
    * 3. 외국환 293
    * 4. 중소·중견기업 우대 프로그램 294
    * 5. 혁신성장산업 우대지원 안내 295
    * 6. 첨단전략산업 우대지원 안내 296
    * 7. 공급망 안정화 금융 프로그램 안내 297

## 04 과학기술정보통신부

* 1. 과학기술정보통신부 (www.msit.go.kr) 301
    * 1. 해외IT지원센터 운영(KOTRA) 302
* 2. 정보통신산업진흥원 (www.nipa.kr) 304
    * 1. AI-반도체 해외실증 지원 305
    * 2. AI·디지털전환 혁신기업 해외실증 지원 307
    * 3. 해외IT지원센터 운영(NIPA) 309
    * 4. 정보통신방송 해외진출지원 311

## 05 외교부

* 1. 외교부 (www.mofa.go.kr) 315
    * 1. 재외공관 법률자문서비스 316

## 06 문화체육관광부

* 1. 문화체육관광부 (www.mcst.go.kr) 319
    * 1. 국제방송영상마켓 참가 지원 320

<!-- KOTRA_PAGE_BREAK -->

2. 게임 수출 활성화 지원 322
3. 콘텐츠 해외시장 개척 지원 324
4. 콘텐츠 해외 현지 출원·등록 지원 326

## 07 농림축산식품부

1. 한국농어촌공사 (www.ekr.or.kr) 331
    1. 농산업 글로벌 성장 패키지 지원 332
    2. 농기자재 해외 박람회 참가 지원 334
    3. 농기자재 신시장 개척 해외 로드쇼 336
    4. 농기자재 바이어 초청 수출상담회 338
    5. 농기자재 수출 시장 개척단 운영 340
    6. 농기자재 수출기업 맞춤형 정보제공 및 컨설팅 342
    7. 농산업수출업체 제조시설현대화 및 운영자금 지원 344

2. 한국농수산식품유통공사 (www.at.or.kr) 346
    1. 농식품글로벌육성지원자금(융자) 347
    2. 농식품 글로벌 경쟁력 강화 349
    3. 농식품 해외시장 진출 351
    4. 농식품 글로벌 비대면 마케팅 354
    5. 농식품시장개척 356

3. 한국동물약품협회 (www.kahpa.or.kr) 358
    1. 동물용의약품 해외수출시장 개척지원 359

## 08 기후에너지환경부

1. 기후에너지환경부 (www.mcee.go.kr) 363
    1. 녹색인프라 해외수출 지원펀드 365
    2. 녹색기술 해외 현지실증 지원사업 367
    3. 개도국 환경개선 마스터플랜 수립 지원사업 370

<!-- KOTRA_PAGE_BREAK -->

# Contents

4. 해외 환경프로젝트 타당성조사 지원사업 372
5. 녹색산업 수출선도화 지원사업 374
6. 해외사무소 운영 376
7. 혁신형 물기업 지정·지원 사업 378
8. 해외 물시장 기술테스트 및 현지 기술검증 지원사업 380
9. 해외 현지 성능평가 지원사업 382
10. 환경·에너지분야 온실가스 국제감축사업 384
11. 탄소중립･그린 국제개발협력(ODA) 386
12. 녹색기후기금 사업개발 및 개도국 지원사업 발굴 388

## 09 보건복지부

1. 한국보건산업진흥원 (www.khidi.or.kr) 393
    1. 글로벌 제약바이오 전문 네트워크 구축 394
    2. K-바이오헬스 글로벌 진출 및 수출 다변화 지원 396
    3. 글로벌 오픈이노베이션 활성화 지원 398
    4. 의료기기산업 전문가 자문위원회(MDCC) 운영 400
    5. 중소화장품 해외진출 지원 402

2. 대한화장품산업연구원 (www.kcii.re.kr) 405
    1. 화장품 종합지원센터 지원 406
    2. 국내 K-뷰티 체험 홍보관 상설운영 408
    3. 피부 유전체 분석센터 구축 410
    4. 미국 OTC 제조소 등록지원 411
    5. 해외 물류기지 구축 413
    6. 온라인 유통 플랫폼 디지털 마케팅 415

## 10 해양수산부

1. 해양수산부 (www.mof.go.kr) 419
    1. 수산물 해외시장개척 사업 420
    2. 우수수산물 지원 사업 422

<!-- KOTRA_PAGE_BREAK -->

## 11 관세청

| 1. 관세청 (www\.customs.go.kr)   | 427 |
| ----------------------------- | --- |
| 1. 보세가공제도 규제혁신방안 「PLUS 전략」 추진 | 428 |
| 2. 전자상거래 수출 통계 개발·서비스 제공      | 429 |
| 3. 수출입화물 검사비용 지원              | 430 |
| 4. 우수기업 상호인정약정(AEO MRA) 체결 확대 | 432 |
| 5. 원산지검증 대응 지원사업              | 434 |
| 6. 찾아가는 FTA 상담센터 운영           | 436 |
| 7. 원산지 간편 인정 대상 품목 확대         | 437 |
| 8. YES FTA 전문교육 사업            | 438 |
| 9. 전자통관시스템(UNI-PASS) 수출 확대    | 440 |
| 10. ｢무역 마이데이터｣ 서비스 확대         | 442 |


## 12 지식재산처

1. 지식재산처 (www.moip.go.kr) 447
    - 1. 해외지식재산센터(해외IP센터) 운영 사업 448
    - 2. 특허분쟁 대응전략 지원 450
    - 3. K-브랜드 분쟁 대응전략 지원 452
    - 4. 한류편승행위 대응지원 454
    - 5. 해외 지식재산권 분쟁정보 제공 456
    - 6. 수출 도전기업 IP위험 대응역량 강화 458

## 13 지방자치단체

1. 서울특별시 (www.seoul.go.kr) 463
    - 1. 글로벌 강소기업 육성 464
    - 2. 수출보험·보증료 지원 466
    - 3. 해외시장 진출 지원 468

<!-- KOTRA_PAGE_BREAK -->

# Contents

* 4. 글로벌커머스 사업 운영 470
* 5. 중소상공인 해외시장 개척단 파견 472
* 6. 중소기업 지식재산권 보호지원 474
* 7. 여성기업 해외진출 지원 476

## 2. 부산광역시 (https://trade.bepa.kr) 478
* 1. 해외무역사절단 파견 480
* 2. 해외전시(박람)회 참가지원 483
* 3. 바이어상담회 개최 486
* 4. 온라인플랫폼 활용지원 488
* 5. 부산수출원스톱센터 운영 490
* 6. 수출입 애로 중소기업 바우처 지원 491
* 7. 글로벌 규격인증, 지재권 확보 지원 [확대] 493
* 8. 해외마케팅 홍보물 제작 지원 494
* 9. 미주 수출 물류네트워크 지원 495
* 10. 부산 수출스타기업 점프업 지원 [개편] 496
* 11. 원부자재 공동구매 지원 [신규] 497
* 12. 수출보험료 지원 [확대] 498
* 13. 수출 신용보증료 지원 [확대] 500
* 14. 해외지사화 사업 지원 502
* 15. 해외무역사무소 운영 503
* 16. 부산 전문무역상사 해외마케팅 지원 504
* 17. FTA 활용지원 505
* 18. 수출중소기업 해외물류비 지원 506
* 19. 환율케어 특별자금 지원 [신규] 507
* 20. 관세피해기업 특별자금 지원 [신규] 508

## 3. 대구광역시 (www.daegu.go.kr) 509
* 1. KOTRA 해외시장정보조사 참가 지원 511
* 2. KOTRA 해외지사화 참가 지원 512
* 3. 수출기업 외국어 통번역 지원 513
* 4. 해외규격인증 획득 지원 514

<!-- KOTRA_PAGE_BREAK -->

5. 무역사절단 파견 지원 515

6. 1社 맞춤형 해외마케팅 지원 517

7. 수출초보기업 해외진출 지원 518

8. 해외공동관 참가 지원 519

9. 해외전시회 개별참가 지원 521

10. 수출상담회 개최 지원 522

11. 수출물류비 지원 523

12. FTA통상진흥센터 운영 지원 524

13. 통상전문인력 양성 사업 525

14. 청년무역사관학교 운영 지원 526

15. 국제특송(EMS) 물류비 지원 527

16. 수출보험료 지원 528

## 4. 인천광역시 (www.incheon.go.kr) 529

1. 해외 시장개척단 파견 531

2. 글로벌 챌린저 지원 사업 533

3. 해외 전시(박람)회 단체참가 지원 535

4. 해외 전시(박람)회 개별참가 지원 537

5. 해외 바이어 초청 수출상담회 개최 539

6. 국내 전시(박람)회 단체참가 지원 541

7. 수출인프라 패키지 지원 543

8. 세계한상대회 단체 참가 지원 544

9. 6차산업 경쟁력 강화 지원 545

10. FTA 활용 지원(인천FTA통상진흥센터 운영지원) 546

11. 중소기업제품 온라인 판매지원 547

12. 해외규격인증 획득 지원 548

13. 국내TV홈쇼핑 입점지원 550

14. 글로벌 e-비즈니스 지원 552

15. 수출 단계별 멘토링 지원사업 554

16. 수출 보험·보증료 지원 556

17. 수출 초보기업 교육 558

18. K-프로덕트 토털마케팅 559

<!-- KOTRA_PAGE_BREAK -->

# Contents

19. 글로벌 강소기업 육성사업 지원 561
20. 중국 마케팅 전담 지원 563

## 5. 광주광역시 (www.gwangju.go.kr) 565
1. 기업맞춤형 수출성공패키지 지원 567
2. 광주지역 FTA통상진흥센터 운영 568
3. 무역보험료 지원 569
4. 수출진흥자금 융자 570
5. 해외물류비 지원 571
6. 해외규격인증 지원 572
7. 해외시장개척단 파견활동 573
8. 해외 전시박람회 단체 참가 지원 574
9. 국내·외 전시박람회 개별 참가 지원 575
10. 온라인수출마케팅 지원 576
11. 해외바이어 단체초청 수출상담회 577
12. 해외지사화 지원 578
13. 해외 세일즈 출장 및 바이어 초청 지원 579
14. 광주·전남·전북 비즈니스 라운지 운영 580
15. 무역아카데미 운영 581

## 6. 대전광역시 (www.daejeon.go.kr) 582
1. 해외통상사무소 운영 584
2. 해외통상사무소주관 비즈니스 상담회 586
3. 해외전시·박람회 단체 참가 지원 588
4. 해외 전시·박람회 개별참가 지원 590
5. 해외 바이어초청 수출상담회 592
6. KOTRA 해외지사화 사업 594
7. 글로벌 강소기업 육성사업 596
8. 해외 진출 직접지원 598
9. 해외 공공조달시장 진입 지원 600
10. 해외 온라인 마케팅(전자상거래) 지원 602
11. KOTRA AI무역지원센터 활용 디지털 마케팅 지원 604

<!-- KOTRA_PAGE_BREAK -->

12. 수출기업 홍보물 제작 지원 606

13. 중소기업 수출역량 강화 교육 608

14. 해외규격인증획득 지원사업 610

15. 해외 물류비 지원 612

16. 수출보증보험료지원 614

17. 비즈니스 파트너십 매칭데이 615

18. FTA통상진흥센터 지원 사업 617

## 7. 울산광역시 (www.ulsan.go.kr) 619

1. 해외무역사절단(시장개척단) 파견 621

2. 해외 유명 전시·박람회 참가 지원 623

3. CES 2027참가 지원 625

4. ULSAN EXPORT PLAZA 2026 627

5. 수출 초보기업 발굴 지원 629

6. 수출 유망기업 육성 지원 631

7. 수출 강소기업 성장 지원 633

8. 국제특송 해외물류비 지원 635

9. 중소기업 해외지사화 사업 637

10. AI·빅데이터 활용 해외비즈니스 마케팅 지원 639

11. 울산 AI무역지원센터 활용 해외홍보 지원 641

12. 해외규격 인증 획득 지원 643

13. FTA 통상진흥센터 운영 645

14. 다문화가족 활용 무역인력 지원 647

15. 무역아카데미 운영 649

16. 중소기업 수출보험보증료 지원 650

17. 해외공공조달 시장진출 지원사업 652

## 8. 세종특별자치시 (www.sejong.go.kr) 654

1. 해외 전시박람회 개별참가 지원 655

2. CES 2027 참가 지원 656

3. 해외 물류비 지원 657

4. 해외판로 수출이음사업 658

<!-- KOTRA_PAGE_BREAK -->

# Contents

5. 해외지사화 참가 지원 659
6. 수출보증보험료 지원 660
7. 해외 전시박람회 단체참가 지원 661
8. 대전·세종 베트남 비즈니스 매칭데이 662
9. FTA 활용 지원 663
10. 글로벌 강소기업 육성사업 664

## 9. 경기도 (www.gg.go.kr) 665
1. 경기비즈니스센터(GBC) 운영 667
2. 무역 위기 대응 패키지 지원 669
3. 수출역량강화 교육 671
4. 수출기업 교류협력 672
5. 무역전문가 활용 해외마케팅 지원 673
6. FTA통상지원 675
7. 경기 안심 수출 보험 지원 677
8. 경기 수출 기회 바우처 679
9. 수출상담회 681
10. 해외규격인증획득 지원 683
11. 통상촉진단 685
12. 수출기업 물류비 지원 687
13. 대한민국우수상품전시회(G-FAIR) 689
14. 글로벌 마케팅 강화 해외 전시회 개최 690
15. 국내외 전시회 참가기업 지원 691
16. 해외전시회 단체참가 지원 693
17. 섬유 패션 마케팅 지원 695
18. 기후테크 스타트업 육성 697
19. 차세대 반도체 패키징 산업전(ASPS) 개최 699
20. 물산업 해외 통상촉진단 지원 700

## 10. 강원특별자치도 (https://state.gwd.go.kr/) 702
1. 온오프라인 무역교육 실시 704
2. 외국어 브로슈어 제작 지원 705

<!-- KOTRA_PAGE_BREAK -->

3. 트레이드코리아 활용 바이어매칭 지원 706
4. 수출상품 포장디자인 지원 708
5. 수출 물류비 지원 710
6. 수출 보험료 지원 712
7. 서울국제식품산업대전 참가지원 714
8. 강원형 수출바우처 지원 716
9. 수출 애로해소 합동지원창구 운영 718
10. 강원수출대상 수상기업 해외 마케팅 지원 719
11. 해외 수출판로 개척 지원 720
12. 강원 수출 스타기업 육성 722
13. 바이오산업 글로벌 기업 육성 지원 724
14. 전략품목 해외 경쟁력 강화 지원 726
15. 해외바이어 초청 통합수출 상담회 개최 728
16. 해외 온라인 마케팅 지원 730
17. 글로벌 강소기업 육성 지원 732
18. 지역특화 청년 무역전문가 양성 지원 734
19. 강원 FTA통상진흥센터 운영 지원 736

## 11. 충청북도 (cbgms.chungbuk.go.kr) 738
1. 관세대응 시장·품목 다변화 지원 739
2. 신시장 개척 및 수출실행력 제고 지원 741
3. 맞춤형 수출활성화 및 생태계 조성 지원 743
4. 무역리스크 대응 개별기업 무역활동 지원 745

## 12. 충청남도 (www.chungnam.go.kr) 747
1. 해외사무소 운영 748
2. 충남형 해외 바이어 초청 수출 상담회 749
3. 해외 전시박람회 참가 지원 750
4. 해외시장개척단 운영 752
5. 자율선택형 수출기반활용 지원 754
6. 글로벌 비즈니스 마케팅 756
7. 혁신기업 해외시장 개척단 운영 757

<!-- KOTRA_PAGE_BREAK -->

# Contents

8. 중소기업 해외지사화 및 시장조사 지원 758
9. 내수기업의 수출기업화 지원사업 759
10. 해외규격인증 획득 지원 운영 761
11. 국제특송(EMS) 지원 763
12. 수출입보험료 및 바이어 신용조사 지원 764
13. 무역의 날 기념식 766
14. 해외전시박람회 참가지원(민간위탁) 767

## 13. 전북특별자치도 (www.jeonbuk.go.kr) 768
1. 전시회 개별참가 지원 770
2. 전시회 단체참가 지원 771
3. 무역사절단 운영 772
4. JB-FAIR(초청상담회) 운영 773
5. 외국어 통·번역 지원 774
6. 마이오피스 지원 775
7. 전북특별자치도 수출통합지원센터 운영 776
8. 전북특별자치도 해외통상거점센터 운영 778
9. 전북형 수출바우처 지원 780
10. 해외규격인증 획득지원 782
11. OK FTA 컨설팅 783
12. FTA통상진흥센터 운영 784
13. 해외 온라인 마케팅 지원 785
14. 다문화가족 협력 해외마케팅 지원 787
15. 미주/유럽/중동 언택트 마케팅 지원사업 788
16. 코트라 글로벌 네트워크 활용사업 789
17. 수출보험 가입지원 791
18. 국제특송 물류비 지원 792
19. 수출새싹기업 지원사업 794

## 14. 전라남도 (www.jexport.go.kr) 797
1. 시장개척단 파견 799
2. 국제박람회 단체 참가지원(공산품·식품) 801

<!-- KOTRA_PAGE_BREAK -->

3. 국제박람회(공산품) 개별 참가지원 803
4. 알리바바 내 전남 브랜드관 입점 지원 804
5. 바이어 초청 수출상담회 개최 805
6. 바이어 초청 개별 수출상담 지원 806
7. 전남형 단계별 수출기업 육성(수출초보, 차세대, 히든챔피언) 808
8. 청년 수출기업 해외마케팅 지원 810
9. 공산품 중소기업 해외규격 인증 지원 812
10. 수출 기반 구축 바우처 지원 814
11. 소량 수출 국제특송비 지원 816
12. 수출보험료 지원 818
13. 통상닥터제 운영 819
14. 전남FTA활용지원센터 운영 821
15. 무역아카데미 개최 822
16. 수출기업 통·번역비 지원 823
17. 아마존 전남 브랜드관 운영 및 입점 지원 824
18. 농수산식품 해외 판촉행사 지원 826
19. 농수산물 수출경쟁력 강화 827
20. 농수산 수출 전략품목 발굴 및 세계 일류상품화 지원 828

## 15. 경상북도 (www.gb.go.kr) 829
1. 무역사절단 파견 831
2. 해외전시회 및 글로벌 경제행사 참가 833
3. 해외전시회 개별참가 835
4. 수출상담회 개최 836
5. 수출물류비 및 수출패키지 지원 837
6. 해외 지사화 및 공동물류센터 지원 838
7. 글로벌온라인몰 입점 839
8. 해외인증 비용 지원 840
9. 해외세일즈 개별출장 지원 841
10. 해외 시장조사 지원 842
11. 중소기업 통·번역 지원 843
12. FTA 맞춤형 컨설팅 지원 844

<!-- KOTRA_PAGE_BREAK -->

# Contents

13. 해외 상설전시판매장 운영 845
14. 중소기업 무역실무교육 846
15. 해외마케팅 패키지 지원 847
16. 경북수출기업협회 활동 지원 848
17. 국제특송(EMS) 해외물류비 지원 849
18. 수출보험료 지원 850
19. 수출기업 디지털콘텐츠 제작(AI무역센터 활용) 851
20. 경상북도 해외 통상투자주재관 운영 852

## 16. 경상남도 (www.gyeongnam.go.kr/trade) 853
1. 해외사무소 운영 855
2. 해외통상자문관 운영 856
3. 무역의 날 행사 및 경남무역인상 시상 857
4. 해외 무역사절단 파견 859
5. 해외 전시·박람회 참가지원 861
6. 바이어 초청 및 수출상담회 참가지원 863
7. 중소기업 해외지사화 지원 865
8. 중소기업 해외인증 획득지원 866
9. 중소기업 통번역 지원 867
10. 내수중소기업 수출전환 868
11. 온라인 해외마케팅 지원 869
12. 무역인력 양성지원 871
13. 수출보험료 지원 872
14. 수출물류비 지원 873
15. 수출 사다리 지원 874
16. 수출초보기업 지원 876
17. 경남 FTA통상진흥센터 운영 878

## 17. 제주특별자치도 (www.JEJU.go.kr) 881
1. 제주전자무역지원시스템 운영 884
2. 제주 수출인의 날 기념행사 개최 886
3. 해외통상사무소 운영 888

<!-- KOTRA_PAGE_BREAK -->

4. 제주상품 글로벌 커머스 지원사업 890

5. 제주FTA통상진흥센터 운영 891

6. OK FTA 컨설팅 892

7. 해외지사화 지원사업 894

8. 해외시장조사서비스 및 해외비즈니스 출장 지원 895

9. 아세안+α 제주상품 원스톱 디지털 수출 896

10. 수출보험료 지원 897

11. 수출 선도·성장 맞춤형 해외마케팅 지원 898

12. 해외 수출물류 네트워크 지원사업 900

13. 제주수출기업 협업마케팅 활성화 지원 902

14. 칭다오 항로 활용 해외시장 개척 지원 904

15. 제주수출역량진단 및 성과관리시스템 운영 905

16. 해외바이어 초청 수출상담회 907

17. 해외전시회 참가 지원 908

18. 지역산업 해외마케팅 지원 909

19. 수출기업 통합조직 수출활성화 지원 912

20. 싱가포르 제주사무소 운영 914

21. 2026년도 글로벌강소기업 1,000+프로젝트 - 강소 916

22. 국내외 박람회 제주화장품 홍보관 운영 및 기업 참가 지원 918

23. 과수바이어 초청 등 수출시장 개척 지원 920

24. 축산물 수출기업 인센티브 지원 922

25. 축산물 수출 해외 마케팅 지원 924

26. 축산물 수출기업 설비 현대화 사업 926

27. 제주수산물 수출마케팅 지원 928

28. 제주수산물 해외시장 개척 지원 929

29. 수출용 수산물 해상 물류비 지원 931

30. 제주수산물 수출 협력 증진 해외바이어 등 관계자 초청경비 지원 932

31. 국내개최 국제전시회 등 참가경비 지원 933

32. 수출상품 샘플 국제특송 비용 지원 935

33. 수출중소기업 외국어 홍보물 제작 지원 937

34. 수출 농수산식품 영양성분 분석검사비 지원 939

35. 수출육가공공장 시설장비 개선 941

body parsed pages: 939
body headings: 3499
body table candidates: 1119


,heading
0,# CHAPTER Ⅰ ’25년 수출 성과 및 평가
1,"## ◇ 美 관세 등 불확실성에도 사상 최초 수출 7,000억불 달성"
2,## ◇ 반도체 수출 강세 속 K-소비재 · 전력기기 등 유망품목도 성장
3,## ◇ 美 · 中 수출 감소에도 아세안 · 중남미 등 신흥시장 중심 수출 증가
4,# Ⅱ CHAPTER
5,## ’26년 수출여건 및 도전과제
6,## CHAPTER Ⅱ ’26년 수출여건 및 도전과제
7,### ◇ 반도체 수요 지속 · 한류 확산 등은 수출 긍정 요인
8,"### ◇ 관세, 보호무역주의 등 통상환경 불안정성은 증대될 전망"
9,"### ◇ 우리 수출구조는 일부 품목·시장·주체 의존도 高, 외부요인에 취약"


### 추출된 본문 Markdown (앞 12,000자)

# CHAPTER Ⅰ ’25년 수출 성과 및 평가

## ◇ 美 관세 등 불확실성에도 사상 최초 수출 7,000억불 달성

**’25년 수출은 7,093억불로 역대 최대, 774억불 흑자 달성**

* 새 정부 출범으로 상반기 국내·외 불안정성을 극복하고, 하반기 시장신뢰 회복으로 6월부터 7개월 연속 월 실적 최대치 경신

**전 세계 6번째로 7,000억불을 달성하며 수출이 가속 성장 중**

* 6,000억불 달성 이후 7년 만에 달성 성과, 현재 전세계 수출 7위

\* 세계 순위(‘25.1~10 기준) : 중국, 미국, 독일, 네덜란드, 일본, 이탈리아, 한국 順

## ◇ 반도체 수출 강세 속 K-소비재 · 전력기기 등 유망품목도 성장

**전세계 AI 확산 트렌드 下 반도체 역대 최대 실적 경신(1,734억불)**

* 자동차·선박·바이오 등 주력 제조업 수출도 확대, 글로벌 경쟁력 재확인

\* 품목별 수출액(억불, %) : <sup>반도체</sup>1,734(+22.2), <sup>자동차</sup>720(+1.7), <sup>선박</sup>318(+24.1), <sup>바이오</sup>163(+7.9)

**K-푸드·뷰티, 전력기기도 역대 최대 수출로 높은 성장세 시현**

\* 품목별 수출액(억불, %) : <sup>전력기기</sup>167(+7.2), <sup>농수산식품</sup>124(+6.4), <sup>화장품</sup>114(+12.2)

## ◇ 美 · 中 수출 감소에도 아세안 · 중남미 등 신흥시장 중심 수출 증가

※ 지역별 수출비중(‘24년 → ’25년, %) : <sup>중국</sup>19.5→18.4, <sup>미국</sup>18.7→17.3, <sup>아세안</sup>16.7→17.3, <sup>중남미</sup>4.2→4.4

**美 관세, 中 경쟁력 제고로 최대 수출시장인 美·中 수출비중은 감소**

* <sup>미국</sup>철강·자동차 등 관세영향품목, <sup>중국</sup>석유화학·디스플레이 감소폭 大

**아세안·중남미 등 신흥시장 수출 증가로 수출시장 다변화 추세**

* <sup>아세안</sup>반도체·바이오, <sup>중남미</sup>자동차 수출 호조세가 수출 확대 견인

<!-- KOTRA_PAGE_BREAK -->



<!-- KOTRA_PAGE_BREAK -->

# Ⅱ CHAPTER

## ’26년 수출여건 및 도전과제

<!-- KOTRA_PAGE_BREAK -->



<!-- KOTRA_PAGE_BREAK -->

## CHAPTER Ⅱ ’26년 수출여건 및 도전과제

### ◇ 반도체 수요 지속 · 한류 확산 등은 수출 긍정 요인

**AI 데이터센터, 서버 투자 급증으로 반도체 수요 확대 지속 전망**

\* ‘30년까지 글로벌 AI 데이터센터 투자금액 3조 달러 이상 전망 (무디스)

* 높은 전력수요·인프라 교체와 맞물려 전력기기 수요도 증가 예상

**K-Pop, K-콘텐츠 등 K-컬처 확산은 소비재 수출 확대의 기회요인**

### ◇ 관세, 보호무역주의 등 통상환경 불안정성은 증대될 전망

**미국의 새로운 관세조치 향방에 대한 불확실성 증가**

* 301조 등 추가 관세 여부, 향후 경쟁여건 변화 등 모니터링 필요

**EU CBAM 본격화, 철강 TRQ 등 보호무역주의도 수출 장애요인**

* 세이프가드, 반덤핑 관세 등 주요국 수입규제도 확산 추세

### ◇ 우리 수출구조는 일부 품목·시장·주체 의존도 高, 외부요인에 취약

**반도체 수출 비중은 역대 최대, 상위 5개 품목 비중도 53.6%**

\* 수출비중(‘25, %) 반도체(24.4), 자동차(10.1), 일반기계(6.6), 석유제품(6.4), 석유화학(6.0)

**美·中 수출 비중이 전체 35.8%, 美·中 정책에 따른 영향 多大**

**대기업 수출에 절대적 의존, 수출 증가에도 성과의 양극화는 여전**

\* 수출 중소기업(‘25년) : 97,897개(전체 95.6%), 1,135억불(전체 16.0%)

* 자금, 정보, 기회 부족 등으로 지방·중소기업 수출 사각지대 존재

**◈ 7,000억불을 넘어 수출 5강으로 도약하기 위한 근본적 변화 필요**

* 품목·시장·주체 다변화로 외부환경에 흔들리지 않는 구조 확립

* ‘모두의 수출’을 지원하기 위한 수출 지원체계 혁신

<!-- KOTRA_PAGE_BREAK -->



<!-- KOTRA_PAGE_BREAK -->

Ⅲ

CHAPTER

## ’26년 범부처 수출확대 전략

2026년도 무역통상진흥시책
MINISTRY OF TRADE, INDUSTRY & RESOURCES

<!-- KOTRA_PAGE_BREAK -->



<!-- KOTRA_PAGE_BREAK -->

## 01 추진 방향

**비전** 모두의 수출을 통한 수출 5대 강국 도약

**목표** '26년 7,400억불로 역대 최대수출 경신

**방향**
◈ 8대 전략 분야 중심 품목·시장 다변화 추진
◈ 금융·전시·인프라 수출 지원체계 혁신
◈ 지방·중소기업에서 글로벌 기업까지 수출 사다리 구축

### 품목·시장 다변화

❶ 新트렌드 대응
* 소비재
* 전력기기
* 바이오헬스

❷ 정상외교 활용
* 방산
* 원전

❸ 통상협력 강화
* 자동차
* 선박
* 철강

### 수출 지원체계 혁신

❶ 무역 금융
* 민·관 협력 금융 확대
* 품목·시장 특화 금융
* 무역금융 제도 정비

❷ 전시 산업
* 랜드마크 전시회 육성
* 지역 연계 전시
* 전시인프라 확대

❸ 인프라
* 무역장벽 119
* AI 수출 플랫폼
* 해외지원거점 효율화

### 수출기업 육성

❶ 지방·영세기업
* 수출희망 1000 프로젝트

❷ 중소기업
* K-수출스타 500 프로젝트

❸ 중견기업
* 월드클래스 플러스

❹ 선도-초보기업 협력
* 전문무역상사
* 동반진출 프로젝트

<!-- KOTRA_PAGE_BREAK -->

## 02 주요 내용

### 1 8대 전략분야 중점 지원

◈ 전략적 수출 품목 · 시장 다변화로 ‘탄탄한 수출구조’ 확립

❶ (新트렌드 대응) <sup>①</sup> 소비재 (한류), <sup>②</sup> 전력기기 (AI), <sup>③</sup> 바이오 (고령화)

❷ (정상외교 활용) <sup>④</sup> 방산, <sup>⑤</sup> 원전

❸ (통상협력 강화) <sup>⑥</sup> 자동차, <sup>⑦</sup> 선박, <sup>⑧</sup> 철강

< 8대 전략 분야별 기회요인 및 타겟시장 >

| 구분        | 분야      | 기회요인 및 타겟시장                                                                                  |
| --------- | ------- | -------------------------------------------------------------------------------------------- |
| 1 新트렌드 대응 | ❶ 소비재   | • (기회요인) K-컬처 확산, 온라인 쇼핑 트렌드<br/>• (타겟시장) 전세계, 중국·중동시장 특화***<br/>\\* 한중관계 개선, 중동지역 한류 호감도 증가 |
|           | ❷ 전력기기  | • (기회요인) AI 확산, 전력 인프라 수요<br/>• (타겟시장) 미국, 중동, 북아프리카                                         |
|           | ❸ 바이오헬스 | • (기회요인) 고령화, 건강·미용에 대한 관심증가<br/>• (타겟시장) 의약품미국, EU, 의료기기CIS, ASEAN                          |
| 2 정상외교 활용 | ❹ 방산    | • (기회요인) 정상외교 성과 및 정부협력 강화<br/>• (타겟시장) 북미, 중동, 유럽                                           |
|           | ❺ 원전    | • (기회요인) 정상외교 성과 및 정부협력 강화<br/>• (타겟시장) 미국, 중동, 유럽, ASEAN                                    |
| 3 통상협력 강화 | ❻ 자동차   | • (기회요인) 전기차 수요 확대, FTA 체결<br/>• (타겟시장) 중남미, 중동, ASEAN                                       |
|           | ❼ 선박    | • (기회요인) 한미 조선 협력(MASGA)<br/>• (타겟시장) 미국                                                     |
|           | ❽ 철강    | • (기회요인) 신흥국 경제성장에 따른 수요 증가<br/>• (타겟시장) 인도, ASEAN<br/>※ CBAM, TRQ, 수입규제 등 통상 리스크 대응         |

<!-- KOTRA_PAGE_BREAK -->

## 1 소비재 : 한류· 온라인 수출 활용 주력 수출품목으로 육성 <mark><u>新트렌드</u></mark>

\* 5대 유망소비재 : <sup>①</sup>식품, <sup>②</sup>화장품, <sup>③</sup>패션, <sup>④</sup>생활용품, <sup>⑤</sup>의약품

### ◈ (’25년) 463억불 → (’26년 목표) 525억불 (+13.4%)

### ◇ 글로벌 시장진출을 위한 마케팅 · 물류· 인증 ·지재권 애로 해소

#### ➊ (마케팅) 한류 연계 현지 프리미엄 마케팅 강화

* 한류 콘텐츠 + K-소비재 수출 결합 ‘한류박람회’ 개최(5회)<sup>산업·문체·관계부처</sup>
  \* LA(5월), 파리(6월), 두바이(6월), 하노이(9월), 멕시코시티(9월, 잠정)

* K-뷰티 상설 플래그십 스토어(8개) 및 해외 팝업부스(10개) 개소<sup>복지부</sup>
  \* (상설) 미국, EU, 동남아, 중남미 등 8개, (팝업) 미국, 이태리, 폴란드, 영국 등 10개

#### ➋ (물류) 반품·유통기한 등 소비재 특성에 맞는 물류 인프라 구축

* (식품) 콜드체인 해외공동물류센터 확대 (241→260개소)<sup>산업·농식품부</sup>, 국내 수산물 저온유통체계 구축<sup>해수부</sup>

* (화장품) 미국 內 화장품 전용 물류센터 구축 (2월, LA)<sup>복지부</sup>

#### ➌ (인증) 해외인증 지원 및 비관세장벽 해소 추진

* 기업별 1:1 밀착 지원을 위한 소비재 해외인증 전담조직 설치(2월~), 화장품 글로벌 규제조화 지원센터 운영 및 규제설명회 개최(10회)<sup>식약처</sup>
  \* (국내) 해외인증지원단 소비재 인증 T/F, (해외) 해외무역관 소비재인증데스크(20개)

* 신규 시장 진출 위해 우선 협상품목<sup>\*</sup> 중심 위생검역 협상 추진<sup>농식품부·식약처</sup>
  \* (예) (베트남) 열처리 가금육, (중국) 쇠고기 등, (EU) 우유·계란·꿀 함유 식품 등

#### ➍ (지재권) K-소비재 프리미엄을 유지하기 위한 지재권 보호 강화

* 수출기업 지재권 확보, 분쟁 대응 법무비용 지원 확대(’26년, 823억)<sup>지재처</sup>
  \* 해외 현지조사 및 권리확보, 특허·상표·디자인 분쟁대응 비용 지원 등

<!-- KOTRA_PAGE_BREAK -->

## ◇ 온라인 해외직접판매 (해외 역직구) 활성화를 위한 인프라 구축

➊ **해외 소비자 접근성을 높이기 위한 ‘글로벌 온라인몰’ 구축 (’26년, 5개)**

\* 홈페이지·모바일앱 구축, 콘텐츠 번역, 해외 서비스센터 운영 지원 등

➋ **자체 글로벌몰 구축이 어려운 중소기업 제품 해외역직구 대행 지원**

\* 역직구 대행 사이트 연동 통한 해외 결제 및 해외 배송 대행 서비스 지원

➌ **‘외국인 역직구 고객체험단’ 운영 (’26.2월~, 5개국 시범운영)**

\* 검색-결제-통관 등 단계별 장애 및 역직구 서비스 개선 방안 발굴

## ◇ 유통망-협력사 상생지원 확대 등 소비재 기업 육성

➊ **유통망-소비재기업 동반진출 ‘K-소비재 플래그십 프로젝트’ 추진(1Q)**

\* 유통기업 해외진출 지원사업(신규, 471억원) + K-수출스타 사업(신규, 402억원) 활용

* 유통망 : **팝업스토어·온라인몰 구축**, 인증·물류 지원 등 (기업당 45억원)

* 소비재기업 : **유통망 입점**, 마케팅·금융·특허·R&D 등 (기업당 5.6억원)

➋ **소비재 기업 지원을 위한 3,000억원 규모 자금 신규 조성**

* **소비재 기업 - 협력사 상생금융 2,600억원 규모 신설(1Q)**

\* (공급망강화보증) 소비재기업 출연을 통해 입점·협력사 대출금리 우대, 보증한도 확대

* **초기 생산자금 지원을 위한 ‘K-뷰티론’ 2배 확대**(200→400억원)<sup>중기부</sup>

\* 지원범위 : (‘25) 화장품 생산비용 → (’26) 화장품 용기 등 필수부자재 추가

➌ **글로벌 경쟁력을 갖춘 K-소비재 기업 육성**

* **유망 K-뷰티 브랜드와 대형 기업과의 협력 마케팅 지원**<sup>중기부</sup>

\* 유통 대기업 매장 입점, 글로벌몰 활용 프로모션, 퍼포먼스 마케팅 등 해외진출 지원

* **푸드테크<sup>\*</sup> 산업 육성을 위해 R&D 등 전주기 지원, 수출 전략 마련**<sup>농식품부</sup>

\* 주요 예시 : 한강라면기계, 3D 푸드프린터, 튀김로봇 등

* **‘화장품산업육성법’ 제정(’26.下)**, 경쟁력 강화 지원(<sup>’25</sup>149→<sup>’26</sup>446억원)<sup>복지부</sup>
**화장품 안정성 평가제도 기반 구축**<sup>식약처</sup>

<!-- KOTRA_PAGE_BREAK -->

## ◇ 한중 관계 개선 활용, 중국 시장 재진출 적극 확대

### ➊ 정부 간, 정부-기업간 교류 강화로 중국 시장 진출 기반 마련

* 한중 정부간 협력채널<sup>\*</sup>을 통한 對中 진출 협력

\* 한중 상무협력대화, 한중 산업장관회의, 한중 FTA 공동위 등

* 中 지방정부 협력 네트워크 확대로 우호적 비즈니스 환경 조성

\* 한-광동성, 한-장쑤성, 한-산둥성 협력 교류회 → 내륙·중서부 지역과도 협력 모색

\*\* 협력 교류회 방식을 기존 포럼에서 상담회 중심으로 전환해 실질적 기업 진출 지원

### ➋ 중국 19개 무역관 활용, 중국시장 진출 인증·물류 지원 거점化

* 중국 내 인증 취득 지원을 위한 ‘소비재인증 데스크’ 설치 (2월, 2개所)

* 1무역관-1유통망 연계를 통해 中 로컬 유통망 진입 지원 (’26년 600개社)

### ➌ 중국 내륙도시 진출, 新 소비층(Z세대, 실버세대 등) 대상 마케팅 강화

\* 중국 종합 무역사절단(‘26.上, 충칭), 2026 중국 역직구 유통망 로드쇼(7월, 청두) 등

## ◇ 「 <sup>가칭</sup> 對中 소비재 수출확대 방안 」 수립 추진 (1Q)

## ◇ 중동· 아시아 등 신흥시장 진출 위한 할랄 등 맞춤형 지원

※ 한국 문화콘텐츠 전년대비 관심도 증가율(‘25) : (지역)<sup>1위</sup>중동, (국가)<sup>1위</sup>인도, <sup>2위</sup>UAE

### ➊ K-컬처 확산과 연계한 할랄 분야 수출 마케팅 확대

* ‘K-뷰티·K-푸드 사절단’ 파견<sup>무역협회</sup>, ‘중동 K-할랄푸드페어’ 개최<sup>농식품부</sup>

* 유망 할랄시장 유통채널 활용 ‘K-할랄 팝업스토어’ 설치<sup>무역협회</sup>

\* 인니 소재 유통채널(K3마트) 협업을 통해 K-뷰티, K-푸드 팝업스토어 운영

### ➋ 인증, 컨설팅 등 현지 맞춤형 지원 강화

* 할랄시장 온·오프라인 상담·컨설팅 및 전문인력 양성 신규추진<sup>무역협회</sup>

\* (온라인) 「K-TRADE Pro」에 전문상담코너 신설, (오프라인) 찾아가는 컨설팅(200개社)

* 할랄·비건·코셔 등 인증 취득비용 부담 완화<sup>\*농식품부</sup>, 할랄 원료제조사 DB 구축<sup>식약처</sup>

\* ▲예산/지원비율 확대(‘25 60억원/70%→’26 70억원/80~90%), ▲상호인정 확대(UAE, 사우디)

<!-- KOTRA_PAGE_BREAK -->

## 2 전력기기 : AI 인프라 수요를 활용한 프로젝트 참여 확대 <mark><u>新트렌드</u></mark>

### ◈ (’25년) 167억불 → (’26년 목표) 177억불 (+6.0%)

❶ **데이터센터 등 수요 증가에 따른 에너지 인프라 프로젝트 참여 지원**

* ‘**프로젝트 발굴-기업매칭-선제적 금융**’으로 우리기업 참여 가능성 제고

❷ **UAE·사우디 등 정부간 협력 계기를 활용, 프로젝트 수주 지원산업·기후부**

* ‘**UAE 스타게이트<sup>\*</sup>**’ 국내기업 참여 방안 논의·모색

\* 중동·북아프리카 지역 중심 전력망을 비롯한 AI 인프라 구축 사업

* ‘**한-사우디 비전 2030<sup>\*</sup>**’ 등을 통해 수출연계 프로젝트 발굴 추진

\* 한-사우디 제조/에너지, 스마트인프라/디지털화, 역량강화 분야 장관급 협의채널(’17~)

## 3 바이오헬스 : 개발 · 생산역량 강화 및 신흥시장 수요 대응 <mark><u>新트렌드</u></mark>

### ◈ (’25년) 163억불 → (’26년 목표) 179억불 (+9.8%)

❶ **공공 CDMO(위탁개발생산) 고도화를 통한 중소·중견기업 생산 지원**

\* 동물세포기반 백신 공공 CDMO 전략적 육성 및 바이오텍 생산지원(’26~’30, 129억원)

❷ **바이오분야 특화 펀드 신설 등을 통해 의약품 개발 역량 확보<sup>복지부</sup>**

\* K-바이오·백신 펀드(~’27, 1조원), 임상 3상 특화펀드(’26, 1,500억원) 등

❸ **글로벌 기업과의 파트너십 구축, 수출 계약을 위한 네트워킹 지원**

\* BIO USA(6월), 바이오플러스-인터펙스코리아(10월), 글로벌 바이오&파마 플라자(10월) 등

* 현지 컨설팅 및 인증 지원 강화를 위한 **K-바이오데스크 확대 운영<sup>\*</sup>**

\* (現) 뉴욕·오사카·호치민 등 7개소 → CIS 등 신흥시장 중심 5개소 신규 지정

* 신흥국 병원 설립 수요 대응 핀포인트 의료기기 상담회 추진

\* 한-인니 바이오테크 로드쇼(4월), K클리닉 in Russia(4월), 한-태 바이오테크 로드쇼(8월) 등

<!-- KOTRA_PAGE_BREAK -->

## 4 방산 : 정상외교 기반 G2G 협력을 통한 수주 확대 <mark>정상외교</mark>

◈ 방산 : (’25년) 154억불 수주 → (’30년 목표) 200억불 수주

**➊ (북미) 한-미 함정 협력 및 캐나다 잠수함 수주에 역량 결집<sup>방사청</sup>**

* (미국) 한-미 함정 조선협력(건조 및 MRO) 구체화 논의 추진

* (캐나다) 산업·전략적 G2G 협력안 기반, 잠수함 우선협상대상자 선정 도모

**➋ (중동) 국가별 방산육성계획 고려 맞춤형 방산협력 제안<sup>방사청</sup>**

* (사우디) 핵심기술 협력 등 기반으로 전력증강사업 수주 추진

* (UAE) 제3국 공동수출 등 정상간 협의사항 기반 방산협력 추진

**➌ (유럽) EU, NATO 역내국 협의를 통해 한국산 무기 도입여건 조성<sup>방사청</sup>**

* (EU) SAFE 기금<sup>\*</sup> 사업 관련 한국 기업 참여 방안 협의 등 협력 확대

\* EU 회원국 방산물자 공동구매 시 EU에서 장기 상환 대출을 지원

* (NATO) 한-NATO 방산협의체 통해 무기 상호운용성 및 표준 확보

## 5 원전 : Post 체코 원전 수주를 위한 전략적 협력 강화 <mark>정상외교</mark>

◈ 원전 : (’25년) 체코 원전 수주 → (’26년 목표) 신규 원전 수주

**➊ 美 원전 확대 추진<sup>\*</sup>에 따른 韓·美 협력 사업 발굴 및 우리 기업 참여 추진**

\* 美 정부는 ’30년까지 미국 내 신규 대형원전 10기 착공 목표(’25.5월, 행정명령)

* 韓·美 협력 시너지 극대화 방향으로 협의 추진

**➋ 旣수주국(UAE, 체코) 대상 협력 강화를 통해 후속 호기 수주 추진**

* 베트남, 튀르키예 등 신규 원전 수주 활동 전개 병행

<!-- KOTRA_PAGE_BREAK -->

## 6 자동차 : FTA·ODA 등 기반조성 및 글로벌 파트너십 확대 <u>통상협력</u>

◈ 자동차 : (’25년) 720억불 → (’26년 목표) 731억불 (전년비 +1.5%)

※ ’26년 전기차 수출 100억불 달성 추진 (전년비 +14.6%)

### ◇ FTA·ODA를 통한 신흥시장 진출 기반조성 및 공급망 안정화 협력

➊ 기체결 FTA 신속 발효, 양허 개선을 통한 자동차 시장접근 제고

* UAE, 말레이시아, 에콰도르, 과테말라와의 FTA 조속 발효 추진

\* 한-UAE CEPA(‘24.5월 서명), 한-말련 FTA(’25.10월 타결), 한-에콰도르 SECA(‘25.9월 서명), 과테말라 한-중미 FTA 가입(’24.1월 서명)

* ’26년 관세가 완전 철폐되는 콜롬비아 시장 활용 중남미 진출 모색

\* 한-콜롬비아 FTA : ‘16년 발효 (자동차 관세 35% → ’25.7월부터 0% 적용)

➋ 전기차 신흥국(인니, 필리핀 등) 중심 ODA 기반 선제적 시장공략 추진

\* (인니) 온실가스 감축용 태양광 충전 e-vehicle 시스템 구축(‘22~’26, 190억원)

\*\* (필리핀) 스마트시티 조성에 필요한 전기차 충전·시험평가 인프라 구축 검토

➌ 핵심광물 협력을 통한 안정적 전기차 공급망 유지

* 주요국 공급망 협력을 통한 전기차 모터소재 등 원활한 수급 지원

### ◇ 글로벌 파트너십 확대로 자동차·부품 수출 경쟁력 제고

➊ 글로벌 파트너링을 통해 국내 부품의 글로벌 완성차 납품 지원(연 17회)

\* 북미(3), 중국(3), 유럽(3), 일본(4), 중남미(2), 동남아(1), 서남아(1) 등 추진(잠정)

➋ 말련 등 현지 OEM과 JV·파트너십 등 통한 시장 진입 모색 추진

\* 예 : 말레이시아 국제통상산업부·투자개발청과 정례 협의체를 통해 협력 방안 논의

<!-- KOTRA_PAGE_BREAK -->

## 7 선박 : 미국 등 주요국 협력 등을 통해 미래 수출 모멘텀 확보 <u>**통상협력**</u>

### ◈ (’25년) 318억불 → (’26년 목표) 340억불 (+6.9%)

**① 한-미 조선 분야에서 다양한 협력 및 수출연계 방안 모색**

\* ▴미측 건조 선박 대상 설계 및 기자재 조달, ▴선박 건조 협력시 블록 수출 등 검토

**② 미래 수주를 선점하기 위한 차세대 선박 핵심기술 선제 확보**

* 암모니아선, 전기추진선 등 차세대 친환경선박 및 AI, IoT 등 첨단기술을 결합한 완전자율운항선박 기술개발 추진(’26, 978억원)

**③ 중소형 조선사 수주를 위한 7,400억원 RG 특례보증 추진(’25, 1,421억원)**

## 8 철강 : 주요국 무역장벽 대응 및 신흥시장 기회요인 활용 <u>**통상협력**</u>

### ◈ (’25년) 303억불 → (’26년 목표) 303억불 (유지)

**① 양자 협의 등을 통해 EU CBAM, TRQ로 인한 업계 피해 최소화**

* CBAM 배출량 산정법 등 하위규정 협의, TRQ 최대쿼터 확보 추진

* 기업 설명회, 배출량 산정 컨설팅<sup>\*</sup> 등 중소·중견 대응 지원 병행

\* 배출량 산정 교육 컨설팅 사업 : ’26년 3.5억원

**② 무역구제 조치(반덤핑, 세이프가드 등) 및 美 파생상품 관세대응 지원 강화**

* 기업 대응 위한 법률·회계 컨설팅 사업 확대<sup>\*</sup>, 대응역량 강화 교육 실시

\* 중소/중견기업 수입규제 대응 지원사업(’25년 10.8억원 → ’26년 20억원)

**③ 인도 전력망, 전기차 등 수요확대 예상 분야 전기강판 생산 확대 추진**

<!-- KOTRA_PAGE_BREAK -->

## 2 수출 지원체계 혁신

◈ 금융· 전시· 수출지원 인프라 혁신으로 ‘모두의 수출’ 지원

### 1 무역금융 : 공급주체 · 대상 · 시장 다변화로 금융 사각지대 해소

◇ 민·관 협력 강화로 중소·중견 무역금융 공급 확대

➊ 

### 사용자 확인: LlamaParse 결과

- 목차 Markdown: `/Users/main/Downloads/Kotra_Chunking-main/26년무역통상진흥시책_work/parsed_toc.md`
- 본문 Markdown: `/Users/main/Downloads/Kotra_Chunking-main/26년무역통상진흥시책_work/parsed_body.md`
- 본문 입력: `/Users/main/Downloads/Kotra_Chunking-main/26년무역통상진흥시책_work/body_pages.pdf`

**다음 작업:** 5번에서 목차를 검토한 뒤 6-A 정제본 생성·유지로 진행하세요.

## 5. 목차 검토 (`HAS_TOC=True`일 때)

목차 Markdown(`parsed_toc.md`)에서 제목을 추출해 검토 파일 `toc_titles_reviewed.md`를 만듭니다.
검토 파일은 **한 줄에 제목 하나**(`- 제목`)인 목록입니다. 줄을 수정·삭제·추가하고 저장한 뒤 재로딩 셀을 실행하고 승인합니다.

- 본문을 나눌 기준으로 쓸 제목만 남깁니다. 순서는 본문에 나오는 순서와 같아야 합니다.
- 페이지 번호는 적지 않습니다. 계층은 제목 번호(제1장 / 1. / 가. 등)로 자동 판단합니다.
- 본문과 표기만 다른 제목은 7번의 `BOUNDARY_ALIASES`로 보정합니다.
- 원본 `parsed_toc.md`는 고치지 않습니다(다시 파싱하면 덮어씁니다).

`HAS_TOC=False`이면 목차 처리·승인은 자동 생략합니다. 함수 정의 셀을 포함해 위에서부터 순서대로 실행하세요.

In [103]:
# 목차 추출·검토 함수

TOC_PAGE_SUFFIX_PATTERN = re.compile(
    r"(?:\s*\.{2,}\s*|\s+)(?:p\.?\s*)?\d{1,4}(?:\s*[-–]\s*\d{1,4})?\s*$",
    re.IGNORECASE,
)
TOC_HEADER_LABELS = {
    "목차", "목 차", "차례", "contents", "table of contents",
    "제목", "페이지", "page", "구분", "분류", "순서", "번호",
}


def normalize_title(text: object) -> str:
    if not isinstance(text, str):
        return ""
    value = strip_title_footnote_markers(unicodedata.normalize("NFC", text).strip())
    value = value.replace("**", "").replace("__", "").replace("`", "")
    value = re.sub(r"^#{1,6}\s*", "", value)
    value = TOC_PAGE_SUFFIX_PATTERN.sub("", value)
    return normalize_spaces(value.strip(" -*•|□◈◆◇○◎▶▷▪■\t"))


def infer_toc_level(title: str, fallback: int = 2) -> int:
    value = normalize_title(title)
    if re.match(r"^(?:제\s*\d+\s*장|[IVX]+[.)\s]|[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩ]+[.)\s])", value, re.I):
        return 1
    if re.match(r"^(?:제\s*\d+\s*절|\d+(?:\.\d+)*[.)\s])", value):
        return 2
    if re.match(r"^(?:[가-하][.)\s]|[-*•])", value):
        return 3
    return max(1, min(int(fallback), 6))


def clean_toc_candidate(value: object) -> str:
    text = normalize_title(str(value or ""))
    text = re.sub(r"<[^>]+>", "", text)
    text = re.sub(r"^[-*•]\s+", "", text)
    text = normalize_spaces(text.strip(" |\t-–—·"))
    if not text or text.lower() in {label.lower() for label in TOC_HEADER_LABELS}:
        return ""
    if normalize_spaces(text).lower() == normalize_spaces(DOCUMENT_NAME).lower():
        return ""
    if re.fullmatch(r"[-:|.\d\s]+", text):
        return ""
    if len(text) < 2 or len(text) > 180:
        return ""
    if not re.search(r"[A-Za-z가-힣]", text):
        return ""
    return text


def extract_toc_items(markdown_path: Path) -> dict[str, object]:
    toc_text = markdown_path.read_text(encoding="utf-8")
    items: list[dict[str, object]] = []
    seen: set[tuple[int, str]] = set()

    def add(raw: object, level: int, source_line: str) -> None:
        title = clean_toc_candidate(raw)
        if not title:
            return
        normalized_level = max(1, min(int(level), 6))
        key = (normalized_level, normalize_spaces(title).lower())
        if key in seen:
            return
        seen.add(key)
        page_match = re.search(r"(?:p\.?\s*)?(\d{1,4})\s*$", source_line, re.I)
        items.append(
            {
                "order": len(items) + 1,
                "level": normalized_level,
                "source_title": title,
                "page_hint": int(page_match.group(1)) if page_match else None,
                "source_line": source_line.strip(),
            }
        )

    for raw_line in toc_text.splitlines():
        line = raw_line.strip()
        if not line:
            continue
        heading_match = re.match(r"^(#{1,6})\s+(.+)$", line)
        if heading_match:
            add(heading_match.group(2), len(heading_match.group(1)), raw_line)
            continue
        if line.startswith("|") and line.endswith("|"):
            cells = [cell.strip() for cell in line.strip("|").split("|")]
            if not cells or all(re.fullmatch(r"[-: ]*", cell) for cell in cells):
                continue
            if any(cell.lower() in {"제목", "페이지", "page"} for cell in cells):
                continue
            text_cells = [
                cell for cell in cells
                if cell and not re.fullmatch(r"(?:p\.?\s*)?\d{1,4}", cell, re.I)
            ]
            if not text_cells:
                continue
            if len(text_cells) >= 2 and re.search(r"장|절|^[IVXⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩ\d]+", text_cells[0], re.I):
                combined = normalize_spaces(f"{text_cells[0]} {text_cells[1]}")
                add(combined, infer_toc_level(text_cells[0]), raw_line)
            else:
                add(text_cells[-1], infer_toc_level(text_cells[-1]), raw_line)
            continue
        add(line, infer_toc_level(line), raw_line)

    if not items:
        raise ValueError(f"목차 제목을 추출하지 못했습니다: {markdown_path}")
    return {
        "source_fingerprint": file_sha256(markdown_path),
        "source_path": str(markdown_path),
        "items": items,
    }


TOC_REVIEW_GUIDE = """<!--
목차 검토 파일 — 한 줄에 제목 하나입니다.
- 본문을 나눌 기준으로 쓸 제목만 남기고, 줄을 고치거나 지우거나 추가하세요.
- 순서는 본문에 나오는 순서와 같아야 하고, 같은 제목을 두 번 쓸 수 없습니다.
- 페이지 번호는 적지 않습니다. 계층은 제목 번호(제1장 / 1. / 가. 등)로 자동 판단합니다.
- 이 안내 주석은 그대로 두어도 됩니다.
-->"""
TOC_REVIEW_COMMENT_RE = re.compile(r"<!--.*?-->", re.DOTALL)


def write_toc_review_markdown(titles: Iterable[object]) -> None:
    lines = "\n".join(f"- {clean_value(title)}" for title in titles)
    TOC_TITLES_REVIEWED_MD_PATH.write_text(
        f"{TOC_REVIEW_GUIDE}\n\n{lines}\n",
        encoding="utf-8",
    )


def read_toc_review_markdown() -> list[str]:
    """안내 주석과 빈 줄을 뺀 각 줄을 제목으로 읽습니다. 줄 앞의 `-`, `#` 등은 검증 단계에서 정리합니다."""
    text = TOC_REVIEW_COMMENT_RE.sub(
        "", TOC_TITLES_REVIEWED_MD_PATH.read_text(encoding="utf-8")
    )
    return [line.strip() for line in text.splitlines() if line.strip()]


def initialize_toc_review_file(
    auto_data: dict[str, object],
    rebuild: bool,
) -> str:
    if TOC_TITLES_REVIEWED_MD_PATH.exists() and not rebuild:
        return "preserved"
    if TOC_TITLES_REVIEWED_JSON_PATH.exists() and not rebuild:
        previous = load_json(TOC_TITLES_REVIEWED_JSON_PATH)
        if isinstance(previous, list) and all(isinstance(item, str) for item in previous):
            write_toc_review_markdown(previous)
            return "converted_from_previous_json"
    write_toc_review_markdown(item["source_title"] for item in auto_data["items"])
    return "created"


def canonical_review_title(value: object) -> str:
    return re.sub(r"\s+", "", normalize_title(value)).lower()


def validate_toc_review_data(value: object) -> pd.DataFrame:
    if not isinstance(value, list) or (HAS_TOC and not value):
        raise ValueError("목차 검토 파일에 제목이 없습니다. 한 줄에 제목 하나씩 입력하세요.")
    rows = []
    for index, item in enumerate(value, start=1):
        if not isinstance(item, str):
            raise ValueError(f"목차 검토 파일의 {index}번째 제목이 문자열이 아닙니다.")
        title = normalize_title(item)
        if not title:
            raise ValueError(f"목차 검토 파일의 {index}번째 줄에 제목으로 쓸 내용이 없습니다: {item!r}")
        rows.append({"order": index, "title": title})

    frame = pd.DataFrame(rows, columns=["order", "title"])
    duplicate_mask = frame["title"].map(canonical_review_title).duplicated(keep=False)
    if duplicate_mask.any():
        duplicates = frame.loc[duplicate_mask, "title"].tolist()
        raise ValueError(f"중복 목차 제목이 있습니다: {duplicates}")
    return frame

In [104]:
# True이면 기존 검토 파일을 버리고 추출 결과로 다시 만듭니다.
REBUILD_TOC_REVIEW_FILE = False

In [105]:
begin_stage("toc_file")
if HAS_TOC:
    auto_toc_data = extract_toc_items(TOC_MARKDOWN_PATH)
    save_json(auto_toc_data, TOC_TITLES_EXTRACTED_JSON_PATH)
    auto_toc_frame = pd.DataFrame(auto_toc_data["items"])
    review_file_status = initialize_toc_review_file(
        auto_toc_data,
        rebuild=REBUILD_TOC_REVIEW_FILE,
    )
    print("TOC items extracted:", len(auto_toc_frame))
    print("TOC review file:", review_file_status, TOC_TITLES_REVIEWED_MD_PATH)
    display(auto_toc_frame[["order", "level", "source_title", "page_hint"]])
    show_user_checkpoint(
        "목차 검토 파일",
        [
            f"수정할 파일: `{TOC_TITLES_REVIEWED_MD_PATH}`",
            "한 줄에 제목 하나입니다. 줄을 수정·삭제·추가하고 순서를 조정하세요.",
            f"원본 목차는 `{TOC_MARKDOWN_PATH}`에서 확인할 수 있습니다(이 파일은 고치지 않습니다).",
            "본문과 표기만 다른 제목은 7번의 BOUNDARY_ALIASES에 추가하세요.",
        ],
        "파일을 저장한 뒤 아래 재로딩·미리보기 셀을 실행하세요.",
    )
else:
    print("목차 없음: 목차 추출·검토 파일 생성 생략")
complete_stage("toc_file")

TOC items extracted: 553
TOC review file: preserved /Users/main/Downloads/Kotra_Chunking-main/26년무역통상진흥시책_work/toc_titles_reviewed.md


,order,level,source_title,page_hint
0,1,2,Ⅰ '25년 수출 성과 및 평가 _,1.0
1,2,2,Ⅱ '26년 수출여건 및 도전과제 _,5.0
2,3,2,Ⅲ '26년 범부처 수출확대 전략 _,9.0
3,4,2,1. 추진 방향 _,11.0
4,5,2,2. 주요 내용 _,12.0
...,...,...,...,...
548,549,2,31. 국내개최 국제전시회 등 참가경비 지원,933.0
549,550,2,32. 수출상품 샘플 국제특송 비용 지원,935.0
550,551,2,33. 수출중소기업 외국어 홍보물 제작 지원,937.0
551,552,2,34. 수출 농수산식품 영양성분 분석검사비 지원,939.0


### 사용자 확인: 목차 검토 파일

- 수정할 파일: `/Users/main/Downloads/Kotra_Chunking-main/26년무역통상진흥시책_work/toc_titles_reviewed.md`
- 한 줄에 제목 하나입니다. 줄을 수정·삭제·추가하고 순서를 조정하세요.
- 원본 목차는 `/Users/main/Downloads/Kotra_Chunking-main/26년무역통상진흥시책_work/parsed_toc.md`에서 확인할 수 있습니다(이 파일은 고치지 않습니다).
- 본문과 표기만 다른 제목은 7번의 BOUNDARY_ALIASES에 추가하세요.

**다음 작업:** 파일을 저장한 뒤 아래 재로딩·미리보기 셀을 실행하세요.

In [106]:
# 재로딩·미리보기: 검토 파일(toc_titles_reviewed.md)을 수정했으면 이 셀부터 다시 실행합니다.
if HAS_TOC:
    reviewed_toc_frame = validate_toc_review_data(read_toc_review_markdown())
    print("reviewed TOC items:", len(reviewed_toc_frame))
    display(reviewed_toc_frame)
else:
    reviewed_toc_frame = pd.DataFrame(columns=["order", "title"])
    print("목차 없음: 검토·승인 불필요 — 그대로 진행하세요.")
toc_review_state = preview_stage("toc")

reviewed TOC items: 553


,order,title
0,1,Ⅰ '25년 수출 성과 및 평가
1,2,Ⅱ '26년 수출여건 및 도전과제
2,3,Ⅲ '26년 범부처 수출확대 전략
3,4,1. 추진 방향
4,5,2. 주요 내용
...,...,...
548,549,31. 국내개최 국제전시회 등 참가경비 지원
549,550,32. 수출상품 샘플 국제특송 비용 지원
550,551,33. 수출중소기업 외국어 홍보물 제작 지원
551,552,34. 수출 농수산식품 영양성분 분석검사비 지원


In [107]:
TOC_REVIEW_APPROVED = True
approve_stage("toc", TOC_REVIEW_APPROVED or not HAS_TOC, toc_review_state)

승인됨: 5. 목차 재로딩·미리보기와 승인


## 6. 본문 노이즈 정제·검토

- **6-A 정제본 생성·유지**: 최초 생성 또는 원문에서 다시 만들 때 실행합니다. 평소에는 preserve입니다.
- **6-B 수정본 반영·재로딩·미리보기**: 정제본을 직접 고치거나 [x]를 체크한 뒤 실행합니다.
- **6-C 본문 수정본 승인**: 6-B의 결과를 확인한 뒤 승인합니다.

정제본을 직접 고쳤다면 **6-B → 6-C → 7번 요소 파싱부터** 실행합니다. 6-A를 거치거나 rebuild로 바꾸지 않습니다.
6번은 저장된 Markdown을 처리하며 LlamaParse를 호출하지 않습니다.

확실한 페이지 장식만 자동 삭제하고, 반복 줄·페이지 내 반복·긴 반복 조각은 검토 후보로 남깁니다.
수정본 `parsed_body_refined.md`를 열어 고치고 저장한 뒤 반영·재로딩 셀을 실행하고 승인합니다.

- 반복 줄을 한꺼번에 지우려면 수정본 맨 위 "본문 노이즈 검토 후보" 목록에서 `[ ]`를 `[x]`로 바꿉니다.
- 그 밖의 줄은 본문에서 직접 지우거나 고칩니다.
- 무엇을 지웠는지는 반영·재로딩 셀의 출력(삭제 기록)과 `body_noise_report.csv`에 남습니다.

In [108]:
# 본문 노이즈 정제·검토 함수

NOISE_CONTROL_START = "<!-- NOISE_REVIEW_CONTROL_START -->"
NOISE_CONTROL_END = "<!-- NOISE_REVIEW_CONTROL_END -->"
NOISE_CONTROL_BLOCK_RE = re.compile(
    rf"{re.escape(NOISE_CONTROL_START)}.*?{re.escape(NOISE_CONTROL_END)}\s*",
    re.DOTALL,
)
FRAGMENT_TOKEN_RE = re.compile(r"[A-Za-z가-힣0-9][A-Za-z가-힣0-9()]{17,}")
EMPTY_BLOCKQUOTE_LINE_RE = re.compile(r"^\s*(?:>\s*)+$")
BLOCKQUOTE_PREFIX_RE = re.compile(
    r"^(\s*(?:#{1,6}\s+)?)(?:>\s*)+"
)
CIRCLED_NUMBER_SPACING_RE = re.compile(r"([①-⑳])(?=[가-힣A-Za-z0-9])")


def normalize_markdown_formatting(
    markdown_text: str,
) -> tuple[str, dict[str, int]]:
    """Remove parser-only quote/HTML formatting while preserving visible text."""
    normalized_lines: list[str] = []
    stats: Counter[str] = Counter()
    in_code_block = False

    def append_normalized_line(value: str) -> None:
        if (
            not value.strip()
            and normalized_lines
            and not normalized_lines[-1].strip()
        ):
            stats["extra_blank_lines_removed"] += 1
            return
        normalized_lines.append(value)

    for line in markdown_text.splitlines():
        stripped = line.strip()
        is_fence = bool(re.match(r"^(?:```|~~~)", stripped))
        if is_fence:
            normalized_lines.append(line)
            in_code_block = not in_code_block
            continue
        if in_code_block:
            normalized_lines.append(line)
            continue

        if EMPTY_BLOCKQUOTE_LINE_RE.fullmatch(line):
            append_normalized_line("")
            stats["empty_blockquote_lines_removed"] += 1
            continue

        line, quote_count = BLOCKQUOTE_PREFIX_RE.subn(r"\1", line, count=1)
        stats["blockquote_prefixes_removed"] += quote_count

        if re.match(r"^\s*#{1,6}\s+", line):
            # 태그만 지우기 전에 제목의 각주 번호도 함께 제거합니다.
            # 일반 문장·각주 설명·표·코드의 숫자는 이 처리를 받지 않습니다.
            cleaned_heading = strip_title_footnote_markers(line)
            stats["heading_footnote_lines_cleaned"] += int(cleaned_heading != line)
            if cleaned_heading != line and re.fullmatch(r"\s*#{1,6}\s*", cleaned_heading):
                # 각주 번호만 있던 제목은 빈 경계를 만들지 않습니다.
                cleaned_heading = ""
            line = cleaned_heading

        def remove_formatting_tag(match: re.Match[str]) -> str:
            stats[f"{match.group('tag').lower()}_tags_removed"] += 1
            return ""

        line = HTML_FORMATTING_TAG_PATTERN.sub(remove_formatting_tag, line)
        line, spacing_count = CIRCLED_NUMBER_SPACING_RE.subn(r"\1 ", line)
        stats["circled_number_spaces_added"] += spacing_count
        append_normalized_line(line)

    normalized_text = "\n".join(normalized_lines)
    if markdown_text.endswith("\n"):
        normalized_text += "\n"
    return normalized_text, {
        key: value for key, value in stats.items() if value
    }


def normalize_noise_line(line: object) -> str:
    if not isinstance(line, str):
        return ""
    value = unicodedata.normalize("NFC", line).strip()
    value = re.sub(r"^#{1,6}\s*", "", value)
    value = value.replace("**", "").replace("__", "").replace("`", "")
    return normalize_spaces(value.strip(" |\t"))


def is_organization_logo_noise(line: object) -> bool:
    """회사명과 관계없이 파서가 생성한 로고 설명 한 줄을 판별합니다."""
    value = normalize_noise_line(line)
    if not value:
        return False
    if re.match(r"^logo\s*[:：]", value, re.IGNORECASE):
        return True
    # `회사명 로고`처럼 짧은 독립 문구만 제거하고 본문 문장은 보존합니다.
    return len(value) <= 160 and bool(re.search(r"(?:^|\s)로고$", value))


def page_line_metadata(
    page_text: str,
    body_pdf_page: int,
) -> list[dict[str, object]]:
    lines = page_text.splitlines()
    nonempty_indexes = [index for index, line in enumerate(lines) if line.strip()]
    nonempty_rank = {index: rank for rank, index in enumerate(nonempty_indexes)}
    nonempty_count = len(nonempty_indexes)
    metadata: list[dict[str, object]] = []
    in_code_block = False

    for line_index, line in enumerate(lines):
        stripped = line.strip()
        is_fence = bool(re.match(r"^(?:```|~~~)", stripped))
        is_code = in_code_block or is_fence
        is_table = stripped.startswith("|") and stripped.endswith("|")
        rank = nonempty_rank.get(line_index)
        edge3 = rank is not None and (rank < 3 or rank >= nonempty_count - 3)
        edge5 = rank is not None and (rank < 5 or rank >= nonempty_count - 5)
        # 빈 줄을 뺀 페이지의 맨 위·맨 아래에서 몇 번째 줄인지(0부터). 빈 줄은 None.
        edge_distance = None if rank is None else min(rank, nonempty_count - 1 - rank)
        position = (
            "blank" if rank is None else
            "top" if rank < 5 else
            "bottom" if rank >= nonempty_count - 5 else
            "middle"
        )
        normalized = normalize_noise_line(line)
        protected = (
            is_code
            or is_table
            or normalized.lower().startswith("자료원:")
        )
        metadata.append(
            {
                "line_index": line_index,
                "line_number": line_index + 1,
                "line": line,
                "normalized": normalized,
                "body_pdf_page": body_pdf_page,
                "edge3": edge3,
                "edge5": edge5,
                "edge_distance": edge_distance,
                "position": position,
                "is_table": is_table,
                "is_code": is_code,
                "protected": protected,
            }
        )
        if is_fence:
            in_code_block = not in_code_block
    return metadata


def repeated_fragment_tokens(line: object) -> set[str]:
    if not isinstance(line, str):
        return set()
    fragments = set()
    for match in FRAGMENT_TOKEN_RE.finditer(unicodedata.normalize("NFC", line)):
        fragment = re.sub(r"^[A-Za-z]\)", "", match.group(0)).strip("()")
        if len(fragment) >= 18 and re.search(r"[A-Za-z가-힣]", fragment):
            fragments.add(fragment)
    return fragments


NOISE_CHECKBOX_RE = re.compile(
    r"(?m)^\s*[-*]\s*\[(?P<mark>[ xX])\]\s*(?P<id>[RWF]\d+)\b"
)
NOISE_KIND_LABELS = {
    "R": "페이지 위·아래 반복 줄",
    "W": "한 페이지 안 반복 줄",
    "F": "줄 안의 반복 조각",
}


def format_page_ranges(pages: Iterable[object]) -> str:
    ranges: list[list[int]] = []
    for page in sorted({int(page) for page in pages}):
        if ranges and page == ranges[-1][1] + 1:
            ranges[-1][1] = page
        else:
            ranges.append([page, page])
    return ", ".join(
        str(first) if first == last else f"{first}~{last}" for first, last in ranges
    )


def load_noise_report() -> pd.DataFrame:
    if BODY_NOISE_REPORT_CSV.exists():
        return pd.read_csv(BODY_NOISE_REPORT_CSV, keep_default_na=False)
    return pd.DataFrame()


def noise_candidate_summary(report: pd.DataFrame) -> pd.DataFrame:
    """검토 후보를 ID별 한 줄로 요약합니다. 상단 블록과 노트북의 삭제 기록에 함께 씁니다."""
    columns = ["candidate_id", "kind", "count", "pages", "text", "applied"]
    if report.empty:
        return pd.DataFrame(columns=columns)
    candidates = report[report["status"].isin(["review", "applied"])]
    rows = []
    for candidate_id, group in candidates.groupby("candidate_id", sort=False):
        rows.append(
            {
                "candidate_id": str(candidate_id),
                "kind": NOISE_KIND_LABELS.get(str(candidate_id)[:1], ""),
                "count": len(group),
                "pages": format_page_ranges(group["body_pdf_page"]),
                "text": clean_value(group.iloc[0]["normalized_line"]),
                "applied": bool(group["status"].eq("applied").all()),
            }
        )
    return pd.DataFrame(rows, columns=columns)


def build_noise_control_block(report: pd.DataFrame) -> str:
    lines = [
        NOISE_CONTROL_START,
        "",
        "## 본문 노이즈 검토 후보",
        "",
        "지울 후보는 `[ ]`를 `[x]`로 바꿔 저장한 뒤, 노트북 6-B 반영·재로딩·미리보기 → 6-C 승인 셀을 실행하세요.",
        "이미 지운 후보는 `적용됨`으로 표시됩니다. 이 블록은 청킹 결과에 포함되지 않습니다.",
        "",
        "| 종류 | 무엇인가 | 체크하면 |",
        "|---|---|---|",
        "| R | 3쪽 이상에서 페이지 맨 위·아래에 되풀이되는 줄 (머리말·바닥글일 가능성이 높음) | 문서 전체에서 같은 줄을 모두 삭제 (표 안의 줄은 제외) |",
        "| W | 한 페이지 안에서 3번 이상 나온 줄 (표 주석·출처 같은 본문일 수 있으니 내용 확인) | 문서 전체에서 같은 줄을 모두 삭제 (표 안의 줄은 제외) |",
        "| F | 18자 이상 붙어 있는 문자열이 여러 번 반복된 조각 | 줄은 남기고 그 조각만 삭제 (표 안 포함) |",
        "",
        "일부만 지우려면 체크하지 말고 본문에서 그 줄을 직접 지우세요.",
        "",
        "### 후보 목록",
        "",
    ]
    summary = noise_candidate_summary(report)
    for _, row in summary.iterrows():
        preview = row["text"]
        if len(preview) > 100:
            preview = preview[:97] + "..."
        lines.append(
            f"- [{'x' if row['applied'] else ' '}] {row['candidate_id']} | {row['kind']} | "
            f"{row['count']}회 | {row['pages']}쪽 | `{preview}`"
            + (" | 적용됨" if row["applied"] else "")
        )
    if summary.empty:
        lines.append("검토 후보가 없습니다.")
    lines.extend(["", NOISE_CONTROL_END])
    return "\n".join(lines)


def refine_body_markdown(
    markdown_text: str,
    expected_body_page_count: int,
) -> tuple[str, pd.DataFrame, dict[str, int]]:
    markdown_text, formatting_stats = normalize_markdown_formatting(markdown_text)
    pages = split_markdown_pages(markdown_text)
    if len(pages) != expected_body_page_count:
        raise ValueError(
            "본문 Markdown과 편집된 body_pages.pdf 페이지 수가 다릅니다: "
            f"Markdown={len(pages)}, body PDF={expected_body_page_count}"
        )

    metadata_by_page = [
        page_line_metadata(page, body_pdf_page)
        for body_pdf_page, page in enumerate(pages, start=1)
    ]
    remove_lines = {normalize_noise_line(value) for value in REMOVE_LINES}
    page_edge_remove_lines = {normalize_noise_line(value) for value in PAGE_EDGE_REMOVE_LINES}
    auto_remove: dict[tuple[int, int], str] = {}

    for metadata in metadata_by_page:
        for item in metadata:
            if item["protected"] or not item["normalized"]:
                continue
            key = (int(item["body_pdf_page"]), int(item["line_index"]))
            normalized = str(item["normalized"])
            if normalized in remove_lines:
                auto_remove[key] = "remove_lines"
            elif is_organization_logo_noise(normalized):
                auto_remove[key] = "organization_logo"
            elif normalized.lower() == "kotra" and item["edge5"]:
                auto_remove[key] = "page_edge_kotra"
            elif item["edge3"] and re.fullmatch(r"\d{1,4}", normalized):
                auto_remove[key] = "page_edge_number"
            elif normalized in page_edge_remove_lines and item["edge_distance"] < PAGE_EDGE_LINE_COUNT:
                auto_remove[key] = "page_edge_remove_lines"

    line_occurrences: defaultdict[str, list[dict[str, object]]] = defaultdict(list)
    fragment_occurrences: defaultdict[str, list[dict[str, object]]] = defaultdict(list)
    within_page: defaultdict[tuple[int, str], list[dict[str, object]]] = defaultdict(list)
    for metadata in metadata_by_page:
        for item in metadata:
            key = (int(item["body_pdf_page"]), int(item["line_index"]))
            normalized = str(item["normalized"])
            if not normalized or key in auto_remove or item["protected"]:
                continue
            line_occurrences[normalized].append(item)
            within_page[(int(item["body_pdf_page"]), normalized)].append(item)
            if not item["is_code"]:
                for fragment in repeated_fragment_tokens(item["line"]):
                    fragment_item = dict(item)
                    fragment_item["normalized"] = fragment
                    fragment_occurrences[fragment].append(fragment_item)

    repeated_lines = [
        (value, occurrences)
        for value, occurrences in line_occurrences.items()
        if len({int(item["body_pdf_page"]) for item in occurrences}) >= 3
        and sum(bool(item["edge5"]) for item in occurrences) / len(occurrences) >= 0.8
    ]
    repeated_within_page = [
        (value, occurrences)
        for (_, value), occurrences in within_page.items()
        if len(occurrences) >= 3
    ]
    repeated_fragments = [
        (value, occurrences)
        for value, occurrences in fragment_occurrences.items()
        if len(occurrences) >= 5
        or len({int(item["body_pdf_page"]) for item in occurrences}) >= 3
    ]

    report_rows: list[dict[str, object]] = []
    for metadata in metadata_by_page:
        for item in metadata:
            key = (int(item["body_pdf_page"]), int(item["line_index"]))
            if key in auto_remove:
                report_rows.append(
                    {
                        "candidate_id": "",
                        "body_pdf_page": item["body_pdf_page"],
                        "line_number": item["line_number"],
                        "original_line": item["line"],
                        "normalized_line": item["normalized"],
                        "position": item["position"],
                        "rule": auto_remove[key],
                        "status": "auto_removed",
                    }
                )

    candidate_number = 1
    for prefix, rule, candidates in [
        ("R", "repeated_page_edge_line", repeated_lines),
        ("W", "repeated_within_page_line", repeated_within_page),
        ("F", "repeated_fragment", repeated_fragments),
    ]:
        for value, occurrences in sorted(
            candidates,
            key=lambda item: (int(item[1][0]["body_pdf_page"]), str(item[0])),
        ):
            candidate_id = f"{prefix}{candidate_number:03d}"
            candidate_number += 1
            for item in occurrences:
                report_rows.append(
                    {
                        "candidate_id": candidate_id,
                        "body_pdf_page": item["body_pdf_page"],
                        "line_number": item["line_number"],
                        "original_line": item["line"],
                        "normalized_line": value,
                        "position": item["position"],
                        "rule": rule,
                        "status": "review",
                    }
                )

    refined_pages = []
    for page, metadata in zip(pages, metadata_by_page):
        removed_indexes = {
            int(item["line_index"])
            for item in metadata
            if (int(item["body_pdf_page"]), int(item["line_index"])) in auto_remove
        }
        refined_pages.append(
            "\n".join(
                line
                for index, line in enumerate(page.splitlines())
                if index not in removed_indexes
            ).strip()
        )

    report = pd.DataFrame(
        report_rows,
        columns=[
            "candidate_id", "body_pdf_page", "line_number",
            "original_line", "normalized_line", "position", "rule", "status",
        ],
    )
    control_block = build_noise_control_block(report)
    refined_body = join_markdown_pages(refined_pages)
    return f"{control_block}\n\n{refined_body}\n", report, formatting_stats


def create_or_preserve_refined_markdown(
    mode: str,
) -> tuple[pd.DataFrame, dict[str, int]]:
    normalized_mode = mode.strip().lower()
    if normalized_mode not in {"preserve", "rebuild"}:
        raise ValueError("REFINED_MARKDOWN_MODE는 preserve 또는 rebuild여야 합니다.")
    source_hash = file_sha256(BODY_MARKDOWN_PATH)

    if BODY_REFINED_MARKDOWN_PATH.exists() and normalized_mode == "preserve":
        cached_manifest = load_json(BODY_REFINEMENT_MANIFEST_PATH, default={})
        if not isinstance(cached_manifest, dict) or cached_manifest.get("source_sha256") != source_hash:
            raise RuntimeError(
                "parsed_body.md가 기존 수정본 생성 이후 변경됐습니다. 기존 수정본을 "
                "백업한 뒤 6-A 설정에서 REFINED_MARKDOWN_MODE='rebuild'로 한 번 다시 생성하세요. "
                "기존 수동 수정과 검토 체크는 덮어써집니다. 생성 후 preserve로 돌려놓고 "
                "6-B 반영·미리보기 → 6-C 승인 순서로 진행하세요."
            )
        existing_refined = BODY_REFINED_MARKDOWN_PATH.read_text(encoding="utf-8")
        normalized_refined, formatting_stats = normalize_markdown_formatting(
            existing_refined
        )
        if normalized_refined != existing_refined:
            BODY_REFINED_MARKDOWN_PATH.write_text(
                normalized_refined,
                encoding="utf-8",
            )
        print("existing refined Markdown preserved:", BODY_REFINED_MARKDOWN_PATH)
        report = load_noise_report()
    else:
        refined_markdown, report, formatting_stats = refine_body_markdown(
            BODY_MARKDOWN_PATH.read_text(encoding="utf-8"),
            body_pdf_page_count,
        )
        BODY_REFINED_MARKDOWN_PATH.write_text(refined_markdown, encoding="utf-8")
        report.to_csv(BODY_NOISE_REPORT_CSV, index=False, encoding="utf-8-sig")
        print("refined Markdown created:", BODY_REFINED_MARKDOWN_PATH)

    save_json(
        {
            "source_sha256": source_hash,
            "refined_sha256": file_sha256(BODY_REFINED_MARKDOWN_PATH),
        },
        BODY_REFINEMENT_MANIFEST_PATH,
    )
    return report, formatting_stats


def remove_noise_control_block(markdown_text: str) -> str:
    return NOISE_CONTROL_BLOCK_RE.sub("", markdown_text, count=1).lstrip()


def remove_reviewed_fragment(line: str, fragment: str) -> tuple[str, int]:
    pattern = re.compile(
        rf"(?:\(\s*p\s*\)\s*)?{re.escape(fragment)}\)?",
        re.IGNORECASE,
    )
    return pattern.subn("", line)


def apply_noise_decisions(
    whole_line_ids: list[str],
    fragment_ids: list[str],
) -> dict[str, int]:
    selected_ids = list(dict.fromkeys([*whole_line_ids, *fragment_ids]))
    if not selected_ids:
        print("승인된 노이즈 삭제 후보가 없습니다.")
        return {}
    if not BODY_NOISE_REPORT_CSV.exists():
        raise FileNotFoundError(BODY_NOISE_REPORT_CSV)

    report = pd.read_csv(BODY_NOISE_REPORT_CSV, keep_default_na=False)
    available_ids = set(report["candidate_id"].astype(str))
    missing = sorted(set(selected_ids) - available_ids)
    if missing:
        raise ValueError(f"노이즈 보고서에 없는 후보 ID: {missing}")
    invalid_whole = [candidate_id for candidate_id in whole_line_ids if candidate_id.startswith("F")]
    invalid_fragment = [candidate_id for candidate_id in fragment_ids if not candidate_id.startswith("F")]
    if invalid_whole or invalid_fragment:
        raise ValueError(
            f"후보 종류와 목록이 맞지 않습니다: whole={invalid_whole}, fragment={invalid_fragment}"
        )

    selected = report[report["candidate_id"].isin(selected_ids)]
    whole_lines = {
        candidate_id: set(group["normalized_line"].astype(str))
        for candidate_id, group in selected[selected["candidate_id"].isin(whole_line_ids)].groupby("candidate_id")
    }
    fragments = {
        candidate_id: set(group["normalized_line"].astype(str))
        for candidate_id, group in selected[selected["candidate_id"].isin(fragment_ids)].groupby("candidate_id")
    }
    markdown_text = BODY_REFINED_MARKDOWN_PATH.read_text(encoding="utf-8")
    control_match = NOISE_CONTROL_BLOCK_RE.search(markdown_text)
    control_block = control_match.group(0).strip() if control_match else ""
    pages = split_markdown_pages(remove_noise_control_block(markdown_text))
    if len(pages) != body_pdf_page_count:
        raise ValueError("수정본과 편집된 body_pages.pdf 페이지 수가 다릅니다.")

    removed_counts = {candidate_id: 0 for candidate_id in selected_ids}
    updated_pages = []
    for body_pdf_page, page in enumerate(pages, start=1):
        metadata = page_line_metadata(page, body_pdf_page)
        kept_lines = []
        for item in metadata:
            matching_whole_ids = [
                candidate_id
                for candidate_id, values in whole_lines.items()
                if item["normalized"] in values
            ]
            if matching_whole_ids and not item["protected"]:
                for candidate_id in matching_whole_ids:
                    removed_counts[candidate_id] += 1
                continue

            updated_line = str(item["line"])
            if not item["is_code"]:
                for candidate_id, values in fragments.items():
                    for fragment in values:
                        updated_line, removed = remove_reviewed_fragment(updated_line, fragment)
                        removed_counts[candidate_id] += removed
            kept_lines.append(updated_line.rstrip())
        updated_pages.append("\n".join(kept_lines).strip())

    updated_body = join_markdown_pages(updated_pages)
    prefix = f"{control_block}\n\n" if control_block else ""
    BODY_REFINED_MARKDOWN_PATH.write_text(
        f"{prefix}{updated_body}\n",
        encoding="utf-8",
    )
    report.loc[report["candidate_id"].isin(selected_ids), "status"] = "applied"
    report.to_csv(BODY_NOISE_REPORT_CSV, index=False, encoding="utf-8-sig")
    return removed_counts


def checked_noise_candidate_ids(markdown_text: str) -> list[str]:
    match = NOISE_CONTROL_BLOCK_RE.search(markdown_text)
    if not match:
        return []
    return list(
        dict.fromkeys(
            item.group("id")
            for item in NOISE_CHECKBOX_RE.finditer(match.group(0))
            if item.group("mark") != " "
        )
    )


def apply_checked_noise_candidates() -> tuple[dict[str, int], pd.DataFrame]:
    """수정본 상단 블록에서 [x]로 표시한 후보를 지우고, 블록을 현재 상태로 다시 씁니다."""
    checked = checked_noise_candidate_ids(
        BODY_REFINED_MARKDOWN_PATH.read_text(encoding="utf-8")
    )
    report = load_noise_report()
    known_ids = set() if report.empty else set(report["candidate_id"].astype(str))
    unknown = [candidate_id for candidate_id in checked if candidate_id not in known_ids]
    if unknown:
        raise ValueError(f"노이즈 보고서에 없는 후보 ID가 체크돼 있습니다: {unknown}")
    pending_ids = (
        set()
        if report.empty
        else set(report.loc[report["status"].eq("review"), "candidate_id"].astype(str))
    )
    new_ids = [candidate_id for candidate_id in checked if candidate_id in pending_ids]
    removed = apply_noise_decisions(
        [candidate_id for candidate_id in new_ids if not candidate_id.startswith("F")],
        [candidate_id for candidate_id in new_ids if candidate_id.startswith("F")],
    )

    report = load_noise_report()
    markdown_text = BODY_REFINED_MARKDOWN_PATH.read_text(encoding="utf-8")
    updated = (
        f"{build_noise_control_block(report)}\n\n"
        f"{remove_noise_control_block(markdown_text)}"
    )
    if updated != markdown_text:
        BODY_REFINED_MARKDOWN_PATH.write_text(updated, encoding="utf-8")
    summary = noise_candidate_summary(report)
    summary["applied_now"] = summary["candidate_id"].isin(new_ids)
    return removed, summary


def validate_noise_settings() -> None:
    for name, value in (
        ("REMOVE_LINES", REMOVE_LINES),
        ("PAGE_EDGE_REMOVE_LINES", PAGE_EDGE_REMOVE_LINES),
    ):
        if (
            isinstance(value, str)
            or not isinstance(value, (set, list, tuple))
            or not all(isinstance(item, str) for item in value)
        ):
            raise ValueError(f'{name}는 {{"문구1", "문구2"}}처럼 문자열 집합으로 입력하세요.')
        single_characters = sorted(item for item in value if len(item.strip()) == 1)
        if single_characters:
            raise ValueError(
                f"{name}에 한 글자짜리 항목이 있습니다: {single_characters}. "
                'set("문구")로 쓰면 글자 단위로 쪼개집니다. {"문구"}처럼 쓰세요.'
            )
    if (
        isinstance(PAGE_EDGE_LINE_COUNT, bool)
        or not isinstance(PAGE_EDGE_LINE_COUNT, int)
        or PAGE_EDGE_LINE_COUNT < 1
    ):
        raise ValueError("PAGE_EDGE_LINE_COUNT는 1 이상의 정수여야 합니다.")

In [109]:
# 6-A. 정제본 생성·유지 설정
# preserve: 기존 parsed_body_refined.md를 유지합니다. 없으면 처음 생성합니다.
#           기존 수정본에도 서식·제목 각주 정리가 적용될 수 있습니다.
#           아래 삭제 목록을 바꿔도 기존 수정본에 새로 적용하지 않습니다.
# rebuild:  parsed_body.md에서 다시 만들어 정제본·노이즈 보고서를 덮어씁니다.
#           직접 수정한 내용과 검토 체크가 사라집니다. 백업 후 필요한 때 한 번만 사용하세요.
# 정제본을 직접 고친 뒤 반영하려는 경우는 모드 변경 없이 6-B → 6-C입니다.
REFINED_MARKDOWN_MODE = "preserve"  # preserve | rebuild

# 아래 문구는 수정본을 처음 만들 때(또는 rebuild할 때)만 적용되며, 줄 전체가 문구와 같을 때만 그 줄을 지웁니다.
# 비교할 때 앞뒤 공백·제목 표시(#)·굵게(**)는 무시하고, 표 안의 줄은 지우지 않습니다.
# 위치와 관계없이 지울 문구. 예: {"Korea Trade-Investment Promotion Agency", "Global Market Report 24-011"}
REMOVE_LINES: set[str] = set({"CHAPTER", "Ministry of Trade, Industry and Resources"})
# 페이지 위·아래 PAGE_EDGE_LINE_COUNT줄 안에 있을 때만 지울 머리글·꼬리글 문구. 예: {"Country Report", "조사부 국별조사팀"}
PAGE_EDGE_REMOVE_LINES: set[str] = set()
# 머리글·꼬리글로 볼 범위: 페이지마다 빈 줄을 뺀 처음 N줄과 마지막 N줄 (기본 5)
# 줄 수는 자동 정제 전 원문(parsed_body.md) 기준이라, 자동으로 지워지는 로고·쪽번호 줄도 한 줄로 셉니다.
PAGE_EDGE_LINE_COUNT = 5

In [110]:
# 6-A. 정제본 생성·유지 실행 — 수정본을 직접 고친 뒤에는 6-B부터 실행합니다.
begin_stage("refine")
validate_noise_settings()
keeps_existing_refined = (
    REFINED_MARKDOWN_MODE.strip().lower() == "preserve"
    and BODY_REFINED_MARKDOWN_PATH.exists()
)
noise_report, formatting_stats = create_or_preserve_refined_markdown(REFINED_MARKDOWN_MODE)
complete_stage("refine")
if keeps_existing_refined and (REMOVE_LINES or PAGE_EDGE_REMOVE_LINES):
    print(
        "기존 수정본을 유지했으므로 REMOVE_LINES·PAGE_EDGE_REMOVE_LINES는 이번에 적용되지 않았습니다. "
        "적용하려면 REFINED_MARKDOWN_MODE='rebuild'로 실행하세요(직접 고친 내용은 사라집니다)."
    )

auto_removed_rows = (
    noise_report[noise_report["status"].eq("auto_removed")]
    if not noise_report.empty else noise_report
)
review_noise_rows = (
    noise_report[noise_report["status"].eq("review")]
    if not noise_report.empty else noise_report
)
print("formatting normalization:", formatting_stats)
print("automatically removed lines:", len(auto_removed_rows))
print("review candidate occurrences:", len(review_noise_rows))
print(
    "review candidate groups:",
    review_noise_rows["candidate_id"].nunique() if not review_noise_rows.empty else 0,
)
display(auto_removed_rows.head(20))
display(review_noise_rows.head(40))

show_user_checkpoint(
    "6-A. 정제본 생성·유지 완료",
    [
        f"원본 보존: `{BODY_MARKDOWN_PATH}`",
        f"확인·수정할 파일: `{BODY_REFINED_MARKDOWN_PATH}`",
        f"노이즈 보고서: `{BODY_NOISE_REPORT_CSV}`",
    ],
    "수정본을 고치거나 [x]를 체크하고 저장한 뒤 6-B 반영·미리보기 셀을 실행하세요.",
)

existing refined Markdown preserved: /Users/main/Downloads/Kotra_Chunking-main/26년무역통상진흥시책_work/parsed_body_refined.md
기존 수정본을 유지했으므로 REMOVE_LINES·PAGE_EDGE_REMOVE_LINES는 이번에 적용되지 않았습니다. 적용하려면 REFINED_MARKDOWN_MODE='rebuild'로 실행하세요(직접 고친 내용은 사라집니다).
formatting normalization: {'extra_blank_lines_removed': 63}
automatically removed lines: 10
review candidate occurrences: 1675
review candidate groups: 117


,candidate_id,body_pdf_page,line_number,original_line,normalized_line,position,rule,status
0,,31,2,01,01,top,page_edge_number,auto_removed
1,,199,2,04,04,top,page_edge_number,auto_removed
2,,265,2,02,02,top,page_edge_number,auto_removed
3,,315,1,06,06,top,page_edge_number,auto_removed
4,,415,1,10,10,top,page_edge_number,auto_removed
5,,423,6,11,11,top,page_edge_number,auto_removed
6,,425,4,01,01,top,page_edge_number,auto_removed
7,,443,6,12,12,top,page_edge_number,auto_removed
8,,476,2,02,02,top,page_edge_number,auto_removed
9,,745,5,12,12,top,page_edge_number,auto_removed


,candidate_id,body_pdf_page,line_number,original_line,normalized_line,position,rule,status
10,R001,7,7,2026년도 무역통상진흥시책,2026년도 무역통상진흥시책,top,repeated_page_edge_line,review
11,R001,27,7,2026년도 무역통상진흥시책,2026년도 무역통상진흥시책,top,repeated_page_edge_line,review
12,R001,29,1,2026년도 무역통상진흥시책,2026년도 무역통상진흥시책,top,repeated_page_edge_line,review
13,R001,249,1,2026년도 무역통상진흥시책,2026년도 무역통상진흥시책,top,repeated_page_edge_line,review
14,R001,283,1,2026년도 무역통상진흥시책,2026년도 무역통상진흥시책,top,repeated_page_edge_line,review
15,R001,297,1,2026년도 무역통상진흥시책,2026년도 무역통상진흥시책,top,repeated_page_edge_line,review
16,R001,389,1,2026년도 무역통상진흥시책,2026년도 무역통상진흥시책,top,repeated_page_edge_line,review
17,R001,423,1,2026년도 무역통상진흥시책,2026년도 무역통상진흥시책,top,repeated_page_edge_line,review
18,R001,443,1,2026년도 무역통상진흥시책,2026년도 무역통상진흥시책,top,repeated_page_edge_line,review
19,R001,459,1,2026년도 무역통상진흥시책,2026년도 무역통상진흥시책,top,repeated_page_edge_line,review


### 사용자 확인: 6-A. 정제본 생성·유지 완료

- 원본 보존: `/Users/main/Downloads/Kotra_Chunking-main/26년무역통상진흥시책_work/parsed_body.md`
- 확인·수정할 파일: `/Users/main/Downloads/Kotra_Chunking-main/26년무역통상진흥시책_work/parsed_body_refined.md`
- 노이즈 보고서: `/Users/main/Downloads/Kotra_Chunking-main/26년무역통상진흥시책_work/body_noise_report.csv`

**다음 작업:** 수정본을 고치거나 [x]를 체크하고 저장한 뒤 6-B 반영·미리보기 셀을 실행하세요.

In [111]:
# 6-B. 수정본 반영·재로딩·미리보기 — 직접 수정하거나 [x]를 체크한 뒤 이 셀부터 실행합니다.
require_valid("refine")
approved_noise_removals, noise_candidates = apply_checked_noise_candidates()

refined_body = remove_noise_control_block(
    BODY_REFINED_MARKDOWN_PATH.read_text(encoding="utf-8")
)
refined_pages = split_markdown_pages(refined_body)
refined_headings = [
    line.strip()
    for line in refined_body.splitlines()
    if re.match(r"^#{1,6}\s+", line)
]
empty_refined_pages = sum(not page.strip() for page in refined_pages)
remaining_entities = len(HTML_ENTITY_PATTERN.findall(refined_body))
remaining_formatting_tags = len(HTML_FORMATTING_TAG_PATTERN.findall(refined_body))
remaining_blockquote_lines = len(BLOCKQUOTE_LINE_PATTERN.findall(refined_body))
unapplied_repeat_groups = int(
    (
        ~noise_candidates["applied"].astype(bool)
        & noise_candidates["candidate_id"].str.startswith(("R", "W"))
    ).sum()
)
body_review_state = preview_stage("body_review")

# 삭제 기록: 이 셀의 출력으로 노트북에 남습니다(전체 내역은 body_noise_report.csv).
deleted_noise_candidates = noise_candidates[noise_candidates["applied"].astype(bool)]
print("이번 실행에서 삭제한 후보(ID: 삭제 건수):", approved_noise_removals or "없음")
print("삭제한 노이즈 후보(누적):", len(deleted_noise_candidates), "/", len(noise_candidates))
if not deleted_noise_candidates.empty:
    display(
        deleted_noise_candidates.rename(
            columns={
                "candidate_id": "ID", "kind": "종류", "count": "횟수", "pages": "쪽",
                "text": "삭제한 내용", "applied_now": "이번 실행",
            }
        )[["ID", "종류", "횟수", "쪽", "삭제한 내용", "이번 실행"]]
    )

print("refined pages:", len(refined_pages))
print("empty refined pages:", empty_refined_pages)
print("headings:", len(refined_headings))
print(
    "table candidates:",
    sum(bool(MARKDOWN_TABLE_SEPARATOR_RE.match(line)) for line in refined_body.splitlines()),
)
print("remaining HTML entities:", remaining_entities)
print("remaining formatting tags:", remaining_formatting_tags)
print("remaining blockquote lines:", remaining_blockquote_lines)
print("unapplied repeated-line groups:", unapplied_repeat_groups)
display(pd.DataFrame({"heading": refined_headings}).head(50))

show_user_checkpoint(
    "6-B. 본문 수정본 미리보기",
    [
        f"확인할 파일: `{BODY_REFINED_MARKDOWN_PATH}`",
        f"빈 페이지 {empty_refined_pages}개, HTML entity {remaining_entities}개입니다.",
        f"서식 태그(sup/sub/mark/u) {remaining_formatting_tags}개, blockquote 줄 "
        f"{remaining_blockquote_lines}개입니다.",
        f"미적용 반복 줄 후보 {unapplied_repeat_groups}개입니다.",
    ],
    "본문이 정확하면 6-C 승인 셀의 BODY_REVIEW_APPROVED=True로 바꿔 실행하세요. 파일을 또 고쳤다면 6-B부터 다시 실행합니다.",
)

승인된 노이즈 삭제 후보가 없습니다.
이번 실행에서 삭제한 후보(ID: 삭제 건수): 없음
삭제한 노이즈 후보(누적): 0 / 117
refined pages: 932
empty refined pages: 23
headings: 3490
table candidates: 1119
remaining HTML entities: 0
remaining formatting tags: 0
remaining blockquote lines: 0
unapplied repeated-line groups: 101


,heading
0,# Ⅰ ’25년 수출 성과 및 평가
1,"## ◇ 美 관세 등 불확실성에도 사상 최초 수출 7,000억불 달성"
2,## ◇ 반도체 수출 강세 속 K-소비재 · 전력기기 등 유망품목도 성장
3,## ◇ 美 · 中 수출 감소에도 아세안 · 중남미 등 신흥시장 중심 수출 증가
4,## Ⅱ ’26년 수출여건 및 도전과제
5,### ◇ 반도체 수요 지속 · 한류 확산 등은 수출 긍정 요인
6,"### ◇ 관세, 보호무역주의 등 통상환경 불안정성은 증대될 전망"
7,"### ◇ 우리 수출구조는 일부 품목·시장·주체 의존도 高, 외부요인에 취약"
8,## Ⅲ ’26년 범부처 수출확대 전략
9,## 01 추진 방향


### 사용자 확인: 6-B. 본문 수정본 미리보기

- 확인할 파일: `/Users/main/Downloads/Kotra_Chunking-main/26년무역통상진흥시책_work/parsed_body_refined.md`
- 빈 페이지 23개, HTML entity 0개입니다.
- 서식 태그(sup/sub/mark/u) 0개, blockquote 줄 0개입니다.
- 미적용 반복 줄 후보 101개입니다.

**다음 작업:** 본문이 정확하면 6-C 승인 셀의 BODY_REVIEW_APPROVED=True로 바꿔 실행하세요. 파일을 또 고쳤다면 6-B부터 다시 실행합니다.

In [112]:
# 6-C. 본문 수정본 승인 — 6-B에서 확인한 파일 상태를 승인합니다.
BODY_REVIEW_APPROVED = True
approve_stage("body_review", BODY_REVIEW_APPROVED, body_review_state)

승인됨: 6-C. 본문 수정본 승인


## 7. Markdown 요소 파싱과 청킹 경계

승인된 본문 수정본을 제목·본문·표 요소로 나누고 표 문맥을 묶습니다.
표 문맥은 같은 페이지의 가까운 제목·도입문·단위와 표 아래 출처·주석·설명을 포함하며,
다음 Markdown 제목, `▶/▷/□/■` 주제, 표, 캡션 또는 페이지 경계에서 멈춥니다.

In [113]:
# Markdown 요소 파싱·표 문맥 함수

def update_heading_stack(
    stack: list[str],
    title: str,
    level: int,
) -> list[str]:
    next_stack = stack[: max(level - 1, 0)]
    next_stack.append(title.strip())
    return next_stack

def dataframe_payload(element: object) -> tuple[str, str, int, int]:
    table_frame = getattr(element, "table", None)
    if table_frame is None:
        return "[]", "[]", 0, 0
    columns = [str(column) for column in table_frame.columns]
    records = table_frame.fillna("").to_dict(orient="records")
    return (
        json.dumps(columns, ensure_ascii=False),
        json.dumps(records, ensure_ascii=False, default=str),
        int(table_frame.shape[0]),
        int(table_frame.shape[1]),
    )

def table_title_from_path(heading_path: object) -> str:
    value = clean_value(heading_path)
    return value.split(" > ")[-1].strip() if value else ""

def split_embedded_markdown_tables(content: str) -> list[tuple[str, str]]:
    lines = content.splitlines()
    pieces: list[tuple[str, str]] = []
    text_buffer: list[str] = []
    index = 0

    def flush_text() -> None:
        text = "\n".join(text_buffer).strip()
        if text:
            pieces.append(("text", text))
        text_buffer.clear()

    while index < len(lines):
        if lines[index].strip().startswith("|"):
            end = index
            while end < len(lines) and lines[end].strip().startswith("|"):
                end += 1
            block_lines = lines[index:end]
            if any(MARKDOWN_TABLE_SEPARATOR_RE.match(line) for line in block_lines):
                flush_text()
                pieces.append(("table_text", "\n".join(block_lines).strip()))
                index = end
                continue
        text_buffer.append(lines[index])
        index += 1
    flush_text()
    return pieces or [("text", content.strip())]

def parse_markdown_elements(markdown_path: Path) -> pd.DataFrame:
    source_markdown = remove_noise_control_block(
        markdown_path.read_text(encoding="utf-8")
    )
    markdown_pages = split_markdown_pages(source_markdown)
    # 페이지 경계를 유지해 다음 페이지의 로고/본문이 표 설명에 섞이지 않게 합니다.
    parsed_with_pages = []
    heading_lines = []
    parser = MarkdownElementNodeParser(llm=None, show_progress=False)
    for page_number, page in enumerate(markdown_pages, start=1):
        heading_lines.extend(
            line.strip() for line in page.splitlines()
            if re.match(r"^#{1,6}\s+", line)
        )
        parsed = parser.extract_elements(
            page, table_filters=[parser.filter_table],
            node_id=f"{markdown_path.stem}-p{page_number}",
        )
        parsed_with_pages.extend(
            (page_number, element) for element in parser.extract_html_tables(parsed)
        )
    title_level_index = 0

    heading_stack: list[str] = []
    base_rows: list[dict[str, object]] = []
    for base_order, (page_number, element) in enumerate(parsed_with_pages, start=1):
        element_type = str(element.type)
        content = clean_value(element.element)
        if not content:
            continue
        title_level = getattr(element, "title_level", None)
        if element_type == "title":
            original_heading = heading_lines[title_level_index]
            title_level = len(re.match(r"^#+", original_heading).group())
            title_level_index += 1
            # preserve로 읽은 기존 정제본도 제목·section path에 각주 번호가 남지 않게 합니다.
            # source_markdown은 원래 제목을 유지해 수정 전 내용을 추적할 수 있습니다.
            content = strip_title_footnote_markers(content).strip()
            if not content:
                continue
            heading_stack = update_heading_stack(
                heading_stack,
                content,
                int(title_level or 1),
            )
        heading_path = " > ".join(heading_stack)
        columns_json, records_json, row_count, column_count = dataframe_payload(element)
        base_rows.append(
            {
                "base_order": base_order,
                "body_pdf_page": page_number,
                "source_markdown": original_heading if element_type == "title" else content,
                "element_id": str(element.id),
                "element_type": element_type,
                "title_level": title_level,
                "heading_path": heading_path,
                "chunk_content": content,
                "table_columns_json": columns_json,
                "table_records_json": records_json,
                "table_row_count": row_count,
                "table_column_count": column_count,
                "table_kind": (
                    "dataframe" if element_type == "table"
                    else "raw_markdown" if element_type == "table_text"
                    else ""
                ),
            }
        )

    expanded_rows: list[dict[str, object]] = []
    for row in base_rows:
        if row["element_type"] != "text":
            expanded_rows.append(row)
            continue
        pieces = []
        for piece_type, piece_content in split_embedded_markdown_tables(str(row["chunk_content"])):
            if piece_type == "text":
                # 제목 없는 ▶ 소제목/캡션/단위도 독립 요소로 만들어 경계를 판정합니다.
                blocks = split_table_context_paragraphs(piece_content)
                pieces.extend(("text", block.strip()) for block in blocks if block.strip())
            else:
                pieces.append((piece_type, piece_content))
        for part_index, (piece_type, piece_content) in enumerate(pieces, start=1):
            piece = dict(row)
            piece["element_type"] = piece_type
            piece["chunk_content"] = piece_content
            piece["source_markdown"] = piece_content
            piece["element_id"] = f"{row['element_id']}-{part_index}"
            if piece_type == "table_text":
                piece["table_kind"] = "fallback_markdown"
            expanded_rows.append(piece)

    columns = [
        "base_order", "body_pdf_page", "source_markdown", "element_id", "element_type",
        "title_level", "heading_path", "chunk_content", "table_columns_json",
        "table_records_json", "table_row_count", "table_column_count", "table_kind",
    ]
    frame = pd.DataFrame(expanded_rows, columns=columns).reset_index(drop=True)
    frame.insert(0, "source_order", range(1, len(frame) + 1))
    frame["content_length"] = frame["chunk_content"].str.len()
    frame["table_title"] = frame["heading_path"].map(table_title_from_path)
    frame["is_table"] = frame["element_type"].isin(["table", "table_text"])
    return attach_table_context(frame)

def is_table_caption(text: str) -> bool:
    value = re.sub(r"^#{1,6}\s+", "", text).strip().strip("*").strip()
    return bool(re.fullmatch(r"(?:<[^<>]+>|〈[^〉]+〉|《[^》]+》|【[^】]+】)", value))

def is_table_unit(text: str) -> bool:
    return bool(re.match(r"^\(?\s*단위\s*[:：]", text.strip().strip("*")))

def is_table_context_boundary(text: str) -> bool:
    value = text.lstrip().lstrip("*").lstrip()
    return bool(TABLE_CONTEXT_STOP_RE.match(value)) or bool(
        re.match(r"^(?:logo|image|figure|engineering_drawing):", value, re.I)
    )

def split_table_context_paragraphs(content: str) -> list[str]:
    blocks = []
    lines = []
    for line in content.splitlines():
        if (not line.strip() or is_table_context_boundary(line)
                or is_table_caption(line) or is_table_unit(line)):
            if lines:
                blocks.append("\n".join(lines).strip())
                lines = []
        if line.strip():
            lines.append(line)
    if lines:
        blocks.append("\n".join(lines).strip())
    return blocks

def attach_table_context(elements: pd.DataFrame) -> pd.DataFrame:
    """원본 요소를 유지하고 제목·도입문·후속 설명의 소유 표를 기록합니다.

    동일 페이지 안에서 다음 제목/주제/표/캡션 전까지만 결합합니다.
    먼저 모든 표의 앞 문맥을 예약해 앞 표가 다음 표의 제목/단위를 가져가지 않게 합니다.
    """
    frame = elements.copy().reset_index(drop=True)
    frame["table_context_owner"] = 0
    frame["table_context_before"] = ""
    frame["table_context_after"] = ""
    frame["table_context_source_orders"] = "[]"
    table_indexes = frame.index[frame["is_table"]].tolist()
    for index in table_indexes:
        candidates = []
        length = 0
        anchored = False
        for previous in range(index - 1, -1, -1):
            row = frame.loc[previous]
            if row["body_pdf_page"] != frame.at[index, "body_pdf_page"] or row["is_table"]:
                break
            if row["table_context_owner"]:
                break
            text = clean_value(row["source_markdown"])
            if row["element_type"] == "title" or is_table_caption(text):
                candidates.append(previous)
                if not anchored:
                    # 가장 가까운 제목을 대표 제목으로 쓰되, 그 앞에 연속된 상위 제목도
                    # 표 문맥에 포함해 제목만 별도 본문 청크로 남지 않게 합니다.
                    frame.at[index, "table_title"] = clean_value(row["chunk_content"])
                anchored = True
                continue
            if anchored:
                # 제목/캡션에 도달한 뒤에는 바로 이어진 제목 묶음까지만 포함합니다.
                break
            if is_table_context_boundary(text):
                break
            length += len(text) + 2
            if length > TABLE_CONTEXT_MAX_BEFORE_CHARS:
                break
            candidates.append(previous)
        if not anchored:
            # 제목이 확인되지 않는 일반 본문을 역방향으로 흡수하지 않습니다.
            candidates = []
            for previous in range(index - 1, -1, -1):
                row = frame.loc[previous]
                if (row["body_pdf_page"] != frame.at[index, "body_pdf_page"]
                        or row["table_context_owner"] or not is_table_unit(clean_value(row["chunk_content"]))):
                    break
                candidates.append(previous)
        for previous in candidates:
            frame.at[previous, "table_context_owner"] = int(frame.at[index, "source_order"])

    for index in table_indexes:
        owner = int(frame.at[index, "source_order"])
        for following in range(index + 1, len(frame)):
            row = frame.loc[following]
            text = clean_value(row["source_markdown"])
            if (row["body_pdf_page"] != frame.at[index, "body_pdf_page"]
                    or row["is_table"] or row["element_type"] == "title"
                    or row["table_context_owner"] or is_table_caption(text)
                    or is_table_unit(text) or is_table_context_boundary(text)):
                break
            frame.at[following, "table_context_owner"] = owner
        owned = frame[frame["table_context_owner"].eq(owner)]
        before = owned[owned["source_order"].lt(owner)]
        after = owned[owned["source_order"].gt(owner)]
        frame.at[index, "table_context_before"] = "\n\n".join(before["source_markdown"])
        frame.at[index, "table_context_after"] = "\n\n".join(after["source_markdown"])
        frame.at[index, "table_context_source_orders"] = json.dumps(
            sorted([owner, *owned["source_order"].astype(int).tolist()])
        )
    return frame

In [114]:
# 표 제목/도입문을 역방향으로 탐색할 최대 길이. 표 아래 설명은 다음 경계까지 보존합니다.
TABLE_CONTEXT_MAX_BEFORE_CHARS = 800
# Markdown 제목 외에 새 본문 주제를 시작하는 보고서 기호입니다.
TABLE_CONTEXT_STOP_RE = re.compile(r"^(?:[▶▷□■]|제\s*\d+\s*[장절](?:\s|$)|[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩ]+[.)]\s*)")

In [115]:
begin_stage("elements")
raw_elements = parse_markdown_elements(BODY_REFINED_MARKDOWN_PATH)
complete_stage("elements")
print("parsed elements:", len(raw_elements))

parsed elements: 11164


In [116]:
require_valid("elements")
print(raw_elements["element_type"].value_counts().to_string())
print("fallback tables:", int(raw_elements["table_kind"].eq("fallback_markdown").sum()))
display(
    raw_elements[
        ["source_order", "element_type", "title_level", "heading_path", "chunk_content"]
    ].head(30)
)

element_type
text          6438
title         3490
table         1096
code           117
table_text      23
fallback tables: 23


,source_order,element_type,title_level,heading_path,chunk_content
0,1,title,1.0,Ⅰ ’25년 수출 성과 및 평가,Ⅰ ’25년 수출 성과 및 평가
1,2,title,2.0,Ⅰ ’25년 수출 성과 및 평가 > ◇ 美 관세 등 불확실성에도 사상 최초 수출 7...,"◇ 美 관세 등 불확실성에도 사상 최초 수출 7,000억불 달성"
2,3,text,NaN,Ⅰ ’25년 수출 성과 및 평가 > ◇ 美 관세 등 불확실성에도 사상 최초 수출 7...,"**’25년 수출은 7,093억불로 역대 최대, 774억불 흑자 달성**"
3,4,text,NaN,Ⅰ ’25년 수출 성과 및 평가 > ◇ 美 관세 등 불확실성에도 사상 최초 수출 7...,"* 새 정부 출범으로 상반기 국내·외 불안정성을 극복하고, 하반기 시장신뢰 회복으로..."
4,5,text,NaN,Ⅰ ’25년 수출 성과 및 평가 > ◇ 美 관세 등 불확실성에도 사상 최초 수출 7...,"**전 세계 6번째로 7,000억불을 달성하며 수출이 가속 성장 중**"
5,6,text,NaN,Ⅰ ’25년 수출 성과 및 평가 > ◇ 美 관세 등 불확실성에도 사상 최초 수출 7...,"* 6,000억불 달성 이후 7년 만에 달성 성과, 현재 전세계 수출 7위"
6,7,text,NaN,Ⅰ ’25년 수출 성과 및 평가 > ◇ 美 관세 등 불확실성에도 사상 최초 수출 7...,"\* 세계 순위(‘25.1~10 기준) : 중국, 미국, 독일, 네덜란드, 일본, ..."
7,8,title,2.0,Ⅰ ’25년 수출 성과 및 평가 > ◇ 반도체 수출 강세 속 K-소비재 · 전력기기...,◇ 반도체 수출 강세 속 K-소비재 · 전력기기 등 유망품목도 성장
8,9,text,NaN,Ⅰ ’25년 수출 성과 및 평가 > ◇ 반도체 수출 강세 속 K-소비재 · 전력기기...,"**전세계 AI 확산 트렌드 下 반도체 역대 최대 실적 경신(1,734억불)**"
9,10,text,NaN,Ⅰ ’25년 수출 성과 및 평가 > ◇ 반도체 수출 강세 속 K-소비재 · 전력기기...,"* 자동차·선박·바이오 등 주력 제조업 수출도 확대, 글로벌 경쟁력 재확인"


### 7-1. 청킹 경계

- 목차 있음: 검토한 목차 제목을 본문 제목과 매칭합니다. 표기가 달라 매칭되지 않으면 `BOUNDARY_ALIASES`로 보정합니다.
- 목차 없음: 본문 제목을 실제 등장 위치별로 사용합니다. 같은 제목이 반복돼도 각 위치에서 나눕니다.
- 본문 제목도 없음: 8번에서 문단·길이 기준으로 나눕니다.

In [117]:
# 경계 매칭 함수

UNICODE_ROMAN_TRANSLATION = str.maketrans(
    {
        "Ⅰ": "I", "Ⅱ": "II", "Ⅲ": "III", "Ⅳ": "IV", "Ⅴ": "V",
        "Ⅵ": "VI", "Ⅶ": "VII", "Ⅷ": "VIII", "Ⅸ": "IX", "Ⅹ": "X",
    }
)
BOUNDARY_LEADING_DECORATION_RE = re.compile(
    r"^[\s|｜:：;·•□◈◆◇○◎▶▷▪■\-–—]+"
)
BOUNDARY_NUMBER_PREFIX_PATTERNS = [
    r"^(?:(?:[IVX]+)(?:[.)]|\s+))\s*",
    r"^제\s*\d+\s*장\s*",
    r"^제\s*\d+\s*절\s*",
    r"^\d+(?:\.\d+)*(?:[.)]|\s+)\s*",
    r"^[가-하][.)]\s*",
]


def canonical_title(text: object) -> str:
    value = normalize_title(text).translate(UNICODE_ROMAN_TRANSLATION)
    value = re.sub(r"\s*[|｜]\s*", " ", value)
    return normalize_spaces(value).lower()


def clean_boundary_variant(value: str) -> str:
    return normalize_spaces(BOUNDARY_LEADING_DECORATION_RE.sub("", value))


def title_variants(text: object) -> set[str]:
    base = canonical_title(text)
    if not base:
        return set()
    variants = {base, clean_boundary_variant(base)}
    for pattern in BOUNDARY_NUMBER_PREFIX_PATTERNS:
        stripped = re.sub(pattern, "", base, flags=re.IGNORECASE).strip()
        stripped = clean_boundary_variant(stripped)
        if stripped:
            variants.add(stripped)

    variants.update(re.sub(r"\s+", "", value) for value in list(variants))
    return {value.lower() for value in variants if value}


def add_toc_paths(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.sort_values("order").copy()
    stack: list[str] = []
    paths = []
    parent_paths = []
    for _, row in result.iterrows():
        level = int(row["level"])
        title = clean_value(row["title"])
        stack = stack[: max(level - 1, 0)]
        parent_paths.append(" > ".join(stack))
        stack.append(title)
        paths.append(" > ".join(stack))
    result["toc_parent_path"] = parent_paths
    result["toc_path"] = paths
    return result


def infer_reviewed_toc_levels(titles: Iterable[object]) -> list[int]:
    """Infer hierarchy from numbering context after the reviewed file became a list."""
    levels: list[int] = []
    active_second_level_style: str | None = None

    for raw_title in titles:
        title = normalize_title(raw_title)
        if re.match(
            r"^(?:제\s*\d+\s*장|[IVX]+[.)\s]|[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩ]+[.)\s])",
            title,
            re.IGNORECASE,
        ):
            level = 1
            active_second_level_style = None
        elif re.match(r"^(?:제\s*\d+\s*절|\d+(?:\.\d+)*[.)\s])", title):
            level = 2
            active_second_level_style = "arabic"
        elif re.match(r"^[가-하][.)\s]", title):
            if active_second_level_style == "arabic":
                level = 3
            else:
                level = 2
                active_second_level_style = "korean"
        else:
            level = infer_toc_level(title, fallback=2)
            if level == 1:
                active_second_level_style = None
        levels.append(level)

    return levels


def select_boundary_items(frame: pd.DataFrame) -> pd.DataFrame:
    selected = frame[["order", "title"]].copy()
    selected["source_title"] = selected["title"]
    selected["level"] = infer_reviewed_toc_levels(selected["title"])
    return add_toc_paths(selected)


def path_parent_matches(toc_parent_path: str, body_heading_path: str) -> bool:
    if not toc_parent_path:
        return True
    toc_parent = toc_parent_path.split(" > ")[-1]
    body_parts = [part for part in body_heading_path.split(" > ") if part]
    if len(body_parts) <= 1:
        return False
    return bool(title_variants(toc_parent) & title_variants(body_parts[-2]))


def match_boundaries(
    boundary_frame: pd.DataFrame,
    elements: pd.DataFrame,
) -> pd.DataFrame:
    headings = elements[elements["element_type"].eq("title")].copy()
    coverage_rows = []
    last_matched_order = 0
    for _, boundary in boundary_frame.sort_values("order").iterrows():
        title = clean_value(boundary["title"])
        aliases = list(BOUNDARY_ALIASES.get(title, []))
        exact_titles = {canonical_title(value) for value in [title, *aliases]}
        variants = set()
        for value in [title, *aliases]:
            variants.update(title_variants(value))
        candidates = headings[
            headings["source_order"].gt(last_matched_order)
            & headings["chunk_content"].map(
                lambda value: bool(variants & title_variants(value))
            )
        ].copy()
        if len(candidates) > 1:
            exact_matches = candidates[
                candidates["chunk_content"].map(canonical_title).isin(exact_titles)
            ]
            if not exact_matches.empty:
                candidates = exact_matches
        if len(candidates) > 1:
            parent_matches = candidates[
                candidates["heading_path"].map(
                    lambda path: path_parent_matches(
                        clean_value(boundary.get("toc_parent_path")),
                        clean_value(path),
                    )
                )
            ]
            if not parent_matches.empty:
                candidates = parent_matches
        # 여기까지 후보가 2개 이상이면 제목 수준(#)으로 하나를 고르지 않고 duplicate로 표시합니다.
        # 반복 인쇄된 제목이 잘못 골라지면 뒤 목차 제목들이 줄줄이 missing이 되기 때문입니다.
        status = "matched" if len(candidates) == 1 else "missing" if candidates.empty else "duplicate"
        matched_order = int(candidates.iloc[0]["source_order"]) if len(candidates) == 1 else None
        if matched_order is not None:
            last_matched_order = matched_order
        coverage_rows.append(
            {
                "toc_order": int(boundary["order"]),
                "level": int(boundary["level"]),
                "reviewed_title": title,
                "toc_path": clean_value(boundary.get("toc_path")),
                "status": status,
                "match_count": len(candidates),
                "matched_source_order": matched_order,
                "candidate_source_orders": json.dumps(
                    candidates["source_order"].astype(int).tolist(),
                    ensure_ascii=False,
                ),
                "candidate_titles": json.dumps(
                    candidates["chunk_content"].astype(str).tolist(),
                    ensure_ascii=False,
                ),
            }
        )
    return pd.DataFrame(coverage_rows, columns=[
        "toc_order", "level", "reviewed_title", "toc_path", "status",
        "match_count", "matched_source_order", "candidate_source_orders", "candidate_titles",
    ])


def boundaries_from_body(elements: pd.DataFrame) -> pd.DataFrame:
    headings = elements.loc[elements["element_type"].eq("title")].sort_values("source_order")
    rows = []
    for index, (_, row) in enumerate(headings.iterrows(), start=1):
        order = int(row["source_order"])
        title = clean_value(row["chunk_content"])
        rows.append({
            "toc_order": index,
            "level": int(row["title_level"]) if pd.notna(row["title_level"]) else 1,
            "reviewed_title": title,
            "toc_path": clean_value(row["heading_path"]),
            "status": "matched",
            "match_count": 1,
            "matched_source_order": order,
            "candidate_source_orders": json.dumps([order]),
            "candidate_titles": json.dumps([title], ensure_ascii=False),
        })
    return pd.DataFrame(rows, columns=[
        "toc_order", "level", "reviewed_title", "toc_path", "status",
        "match_count", "matched_source_order", "candidate_source_orders", "candidate_titles",
    ])

In [118]:
# {검토 목차 제목: [본문에 실제로 적힌 표기, ...]}
BOUNDARY_ALIASES: dict[str, list[str]] = {}

In [119]:
begin_stage("boundaries")
if not all(
    isinstance(title, str)
    and isinstance(aliases, list)
    and all(isinstance(alias, str) for alias in aliases)
    for title, aliases in BOUNDARY_ALIASES.items()
):
    raise ValueError("BOUNDARY_ALIASES는 {제목: [별칭, ...]} 형식이어야 합니다.")

if HAS_TOC:
    boundary_items = select_boundary_items(reviewed_toc_frame)
    boundary_coverage = match_boundaries(boundary_items, raw_elements)
else:
    boundary_coverage = boundaries_from_body(raw_elements)
    boundary_items = boundary_coverage.rename(
        columns={"toc_order": "order", "reviewed_title": "title"},
    )[["order", "level", "title", "toc_path"]].copy()
    print("목차 매칭 생략: 본문 제목을 직접 사용합니다." if len(boundary_items)
          else "본문 제목 없음: 문단·길이 기준으로 청킹합니다.")
boundary_coverage.to_csv(
    BOUNDARY_COVERAGE_CSV,
    index=False,
    encoding="utf-8-sig",
)
boundary_errors = boundary_coverage[~boundary_coverage["status"].eq("matched")]

print("selected boundaries:", len(boundary_items))
print("matched boundaries:", int(boundary_coverage["status"].eq("matched").sum()))
print("boundary errors:", len(boundary_errors))
display(boundary_coverage)

if not boundary_errors.empty:
    page_by_order = dict(
        zip(raw_elements["source_order"].astype(int), raw_elements["body_pdf_page"].astype(int))
    )
    details = []
    for _, row in boundary_errors.iterrows():
        pages = [page_by_order.get(int(order)) for order in json.loads(row["candidate_source_orders"])]
        where = f" (본문 후보: {', '.join(f'{page}쪽' for page in pages)})" if pages else ""
        details.append(f"- {row['status']}: {row['reviewed_title']}{where}")
    raise RuntimeError(
        "미적중 또는 중복 boundary가 있습니다.\n" + "\n".join(details) + "\n"
        "missing은 toc_titles_reviewed.md의 제목이나 BOUNDARY_ALIASES를 고치고, "
        "duplicate는 parsed_body_refined.md에서 반복 인쇄된 제목 줄을 지운 뒤 다시 실행하세요."
    )

boundary_by_source_order = {
    int(row["matched_source_order"]): {
        "title": row["reviewed_title"],
        "toc_path": row["toc_path"],
    }
    for _, row in boundary_coverage.iterrows()
}
complete_stage("boundaries")

selected boundaries: 553
matched boundaries: 489
boundary errors: 64


,toc_order,level,reviewed_title,toc_path,status,match_count,matched_source_order,candidate_source_orders,candidate_titles
0,1,1,Ⅰ '25년 수출 성과 및 평가,Ⅰ '25년 수출 성과 및 평가,missing,0,NaN,[],[]
1,2,1,Ⅱ '26년 수출여건 및 도전과제,Ⅱ '26년 수출여건 및 도전과제,missing,0,NaN,[],[]
2,3,1,Ⅲ '26년 범부처 수출확대 전략,Ⅲ '26년 범부처 수출확대 전략,missing,0,NaN,[],[]
3,4,2,1. 추진 방향,Ⅲ '26년 범부처 수출확대 전략 > 1. 추진 방향,matched,1,42.0,[42],"[""01 추진 방향""]"
4,5,2,2. 주요 내용,Ⅲ '26년 범부처 수출확대 전략 > 2. 주요 내용,duplicate,2,NaN,"[59, 9974]","[""02 주요 내용"", ""주요내용""]"
...,...,...,...,...,...,...,...,...,...
548,549,2,31. 국내개최 국제전시회 등 참가경비 지원,Ⅳ 각 기관별 세부 추진사업 > 31. 국내개최 국제전시회 등 참가경비 지원,matched,1,11067.0,[11067],"[""31 국내개최 국제전시회 등 참가경비 지원""]"
549,550,2,32. 수출상품 샘플 국제특송 비용 지원,Ⅳ 각 기관별 세부 추진사업 > 32. 수출상품 샘플 국제특송 비용 지원,matched,1,11081.0,[11081],"[""32 수출상품 샘플 국제특송 비용 지원""]"
550,551,2,33. 수출중소기업 외국어 홍보물 제작 지원,Ⅳ 각 기관별 세부 추진사업 > 33. 수출중소기업 외국어 홍보물 제작 지원,matched,1,11103.0,[11103],"[""33 수출중소기업 외국어 홍보물 제작 지원""]"
551,552,2,34. 수출 농수산식품 영양성분 분석검사비 지원,Ⅳ 각 기관별 세부 추진사업 > 34. 수출 농수산식품 영양성분 분석검사비 지원,matched,1,11122.0,[11122],"[""34 수출 농수산식품 영양성분 분석검사비 지원""]"


RuntimeError: 미적중 또는 중복 boundary가 있습니다.
- missing: Ⅰ '25년 수출 성과 및 평가
- missing: Ⅱ '26년 수출여건 및 도전과제
- missing: Ⅲ '26년 범부처 수출확대 전략
- duplicate: 2. 주요 내용 (본문 후보: 6쪽, 839쪽)
- duplicate: Ⅳ 각 기관별 세부 추진사업 (본문 후보: 24쪽, 382쪽, 436쪽)
- duplicate: 9. 수출바우처 사업 (본문 후보: 48쪽, 221쪽, 260쪽)
- duplicate: 10. 해외지사화사업 (본문 후보: 51쪽, 216쪽, 266쪽)
- duplicate: 26. 해외바이어 초청 수출상담회 (본문 후보: 80쪽, 583쪽, 898쪽)
- missing: 39. 글로벌 의약품 수출상담회(Global Bio\&Pharma Plaza)
- missing: 44. 해외 수요처 연계 기술개발사업 R\&D 수요 검증
- missing: 2. 한국무역보험공사 (www.ksure.or.kr)
- missing: 3. 한국무역협회 (www.kita.net)
- duplicate: 12. 해외전시회 참가지원 (본문 후보: 165쪽, 899쪽)
- duplicate: 2. 해외지사화사업 (본문 후보: 216쪽, 266쪽)
- duplicate: 4. 수출바우처사업 (본문 후보: 221쪽, 260쪽)
- duplicate: 6. 글로벌비즈니스센터(GBC) (본문 후보: 257쪽, 262쪽)
- missing: 05 외교부
- missing: 1. 한국농어촌공사 (www.ekr.or.kr)
- missing: 2. 한국농수산식품유통공사 (www.at.or.kr)
- missing: 3. 한국동물약품협회 (www.kahpa.or.kr)
- missing: 1. 기후에너지환경부 (www.mcee.go.kr)
- duplicate: 6. 해외사무소 운영 (본문 후보: 367쪽, 739쪽, 846쪽)
- missing: 11. 탄소중립･그린 국제개발협력

## 8. 청킹

실행 셀 하나가 `본문 청킹 → 표 청크 → 결합 → ID 부여 → 원본 요소 coverage 검사`까지 수행합니다.

- 본문: 경계 제목마다 나누고 `MAX_CHARS`를 넘으면 문단·문장 단위로 분할합니다. `MIN_CHARS` 미만은 같은 상위 제목의
  다음·이전 청크와 병합하며, 병합 후에도 `DROP_SHORT_CHARS` 미만인 본문만 `dropped_short_chunks.csv`에 기록하고 제외합니다.
- 표: `제목·도입문·단위 → 원본 표 → 출처·주석·설명`을 한 청크로 보존하고 본문 뒤에 배치합니다.
  표 청크는 `MAX_CHARS`를 넘어도 분할하지 않고 QA 경고로만 표시합니다.
- ID: `{section}_{country_name}_{문서키}_{0부터 시작하는 순번}`. 문서키는 1번에서 확인한 값으로,
  `DOCUMENT_NO` → `FROM_URL`의 번호 → (URL이 없거나 `USE_URL_FOR_DOC_KEY=False`일 때) 문서 제목 해시 순입니다.

In [ ]:
# 본문 청킹·표 청크·ID 함수

def split_oversized_unit(text: str, max_chars: int) -> list[str]:
    text = text.strip()
    if len(text) <= max_chars:
        return [text] if text else []
    sentences = re.split(r"(?<=[.!?。])\s+|(?<=다\.)\s+", text)
    if len(sentences) == 1:
        return [text[index:index + max_chars] for index in range(0, len(text), max_chars)]
    pieces: list[str] = []
    current = ""
    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue
        if len(sentence) > max_chars:
            if current:
                pieces.append(current)
                current = ""
            pieces.extend(
                sentence[index:index + max_chars]
                for index in range(0, len(sentence), max_chars)
            )
            continue
        candidate = f"{current} {sentence}".strip()
        if current and len(candidate) > max_chars:
            pieces.append(current)
            current = sentence
        else:
            current = candidate
    if current:
        pieces.append(current)
    return pieces


def split_long_text(text: str, max_chars: int) -> list[str]:
    paragraphs = [part.strip() for part in re.split(r"\n\s*\n", text) if part.strip()]
    units: list[str] = []
    for paragraph in paragraphs or [text.strip()]:
        units.extend(split_oversized_unit(paragraph, max_chars))
    pieces: list[str] = []
    current = ""
    for unit in units:
        candidate = f"{current}\n\n{unit}".strip()
        if current and len(candidate) > max_chars:
            pieces.append(current)
            current = unit
        else:
            current = candidate
    if current:
        pieces.append(current)
    return pieces


def semantic_parent_path(chunk: dict[str, object]) -> str:
    path = clean_value(chunk.get("section_path"))
    return path.rsplit(" > ", 1)[0].strip() if " > " in path else ""


def shares_semantic_parent(
    left: dict[str, object],
    right: dict[str, object],
) -> bool:
    left_section = clean_value(left.get("current_section"))
    right_section = clean_value(right.get("current_section"))
    if left_section and left_section == right_section:
        return True
    left_parent = semantic_parent_path(left)
    right_parent = semantic_parent_path(right)
    return bool(left_parent and left_parent == right_parent)


def merge_chunk_pair(
    left: dict[str, object],
    right: dict[str, object],
) -> dict[str, object]:
    merged = dict(right)
    merged["chunk_content"] = (
        f"{clean_value(left['chunk_content'])}\n\n"
        f"{clean_value(right['chunk_content'])}"
    ).strip()
    merged["source_orders"] = sorted(
        set(left["source_orders"]) | set(right["source_orders"])
    )
    merged["original_order_start"] = min(merged["source_orders"])
    merged["original_order_end"] = max(merged["source_orders"])
    return merged


def merge_short_chunks(chunks: list[dict[str, object]]) -> list[dict[str, object]]:
    merged = [dict(chunk) for chunk in chunks]

    # 짧은 청크는 먼저 같은 상위 목차의 다음 청크에 붙인다.
    index = 0
    while index < len(merged) - 1:
        current = merged[index]
        following = merged[index + 1]
        combined_length = (
            len(clean_value(current["chunk_content"]))
            + len(clean_value(following["chunk_content"]))
            + 2
        )
        if (
            len(clean_value(current["chunk_content"])) < MIN_CHARS
            and shares_semantic_parent(current, following)
            and combined_length <= MAX_CHARS
        ):
            merged[index:index + 2] = [merge_chunk_pair(current, following)]
            continue
        index += 1

    # 마지막에 남은 짧은 청크는 같은 상위 목차의 이전 청크에 붙인다.
    index = len(merged) - 1
    while index > 0:
        previous = merged[index - 1]
        current = merged[index]
        combined_length = (
            len(clean_value(previous["chunk_content"]))
            + len(clean_value(current["chunk_content"]))
            + 2
        )
        if (
            len(clean_value(current["chunk_content"])) < MIN_CHARS
            and shares_semantic_parent(previous, current)
            and combined_length <= MAX_CHARS
        ):
            merged[index - 1:index + 1] = [merge_chunk_pair(previous, current)]
        index -= 1
    return merged


def chunk_body_elements(
    elements: pd.DataFrame,
    boundaries: dict[int, dict[str, str]],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    chunks: list[dict[str, object]] = []
    current_parts: list[str] = []
    current_orders: list[int] = []
    current_section = DOCUMENT_NAME if not boundaries else ""
    current_path = DOCUMENT_NAME if not boundaries else ""
    segment_id = 0
    current_has_prose = False

    def flush() -> None:
        nonlocal current_parts, current_orders, current_has_prose
        content = "\n\n".join(part for part in current_parts if part.strip()).strip()
        if content and current_orders:
            chunks.append(
                {
                    "chunk_type": "body",
                    "section": DOCUMENT_SECTION,
                    "current_section": current_section,
                    "section_path": current_path,
                    "chunk_content": content,
                    "source_orders": sorted(set(current_orders)),
                    "original_order_start": min(current_orders),
                    "original_order_end": max(current_orders),
                    "segment_id": segment_id,
                    "table_title": "",
                    "table_summary": "",
                    "table_kind": "",
                    "summary_source": "",
                }
            )
        current_parts = []
        current_orders = []
        current_has_prose = False

    for _, row in elements.sort_values("source_order").iterrows():
        order = int(row["source_order"])
        if int(row.get("table_context_owner", 0)):
            if order in boundaries:
                if current_has_prose:
                    flush()
                segment_id += 1
                current_section = boundaries[order]["title"]
                current_path = boundaries[order]["toc_path"]
            continue
        if bool(row["is_table"]):
            # 표는 별도 청크로 보존하되 본문 의미 흐름을 끊지 않는다.
            # 제목만 쌓인 표 전용 section은 다음 section으로 넘기지 않는다.
            if current_parts and not current_has_prose:
                flush()
            continue
        text = clean_value(row["chunk_content"])
        if not text:
            continue
        if order in boundaries:
            # 연속 제목뿐이면 다음 실제 본문까지 유지해 제목 전용 청크를 막는다.
            if current_has_prose:
                flush()
            segment_id += 1
            current_section = boundaries[order]["title"]
            current_path = boundaries[order]["toc_path"]

        for piece in split_long_text(text, MAX_CHARS):
            candidate = "\n\n".join([*current_parts, piece]).strip()
            if current_parts and len(candidate) > MAX_CHARS:
                flush()
            current_parts.append(piece)
            current_orders.append(order)
            if clean_value(row["element_type"]) != "title":
                current_has_prose = True
    flush()

    chunks = merge_short_chunks(chunks)
    body_columns = [
        "chunk_id", "chunk_type", "section", "current_section", "section_path",
        "chunk_content", "source_orders", "original_order_start", "original_order_end",
        "segment_id", "table_title", "table_summary", "table_kind", "summary_source",
        "content_length",
    ]
    if not chunks:
        empty = pd.DataFrame(columns=body_columns)
        return empty, empty.assign(drop_reason=pd.Series(dtype=str))
    all_candidates = pd.DataFrame(chunks)
    all_candidates["content_length"] = all_candidates["chunk_content"].str.len()
    # 실제 chunk_id는 본문과 표를 합친 최종 순서가 정해진 뒤 부여합니다.
    all_candidates.insert(0, "chunk_id", "")
    drop_mask = all_candidates["content_length"].lt(DROP_SHORT_CHARS)
    dropped = all_candidates.loc[drop_mask].copy()
    dropped["drop_reason"] = (
        f"residual_body_under_{DROP_SHORT_CHARS}_chars_after_semantic_merge"
    )
    frame = all_candidates.loc[~drop_mask].copy()
    if frame.empty:
        raise RuntimeError(
            f"{DROP_SHORT_CHARS}자 미만 잔여 청크를 제외한 뒤 본문 청크가 없습니다."
        )
    for target in (frame, dropped):
        target["source_orders"] = target["source_orders"].map(
            lambda values: json.dumps(values, ensure_ascii=False)
        )
    return frame.reset_index(drop=True), dropped.reset_index(drop=True)


def build_table_chunks(elements: pd.DataFrame) -> pd.DataFrame:
    tables = elements[elements["is_table"]].sort_values("source_order")
    rows = []
    for _, row in tables.iterrows():
        raw_table = clean_value(row["chunk_content"])
        if not raw_table:
            continue
        order = int(row["source_order"])
        context_before = clean_value(row.get("table_context_before"))
        context_after = clean_value(row.get("table_context_after"))
        content = "\n\n".join(part for part in [context_before, raw_table, context_after] if part)
        source_orders = json.loads(row["table_context_source_orders"])
        rows.append(
            {
                # 실제 ID는 본문과 표를 합친 최종 순서가 정해진 뒤 부여합니다.
                "chunk_id": "",
                "chunk_type": "table",
                "section": DOCUMENT_SECTION,
                "current_section": clean_value(row["table_title"]),
                "section_path": clean_value(row["heading_path"]),
                "chunk_content": content,
                "raw_table_content": raw_table,
                "table_context_before": context_before,
                "table_context_after": context_after,
                "body_pdf_page": int(row["body_pdf_page"]),
                "table_source_order": order,
                "source_orders": json.dumps(source_orders, ensure_ascii=False),
                "original_order_start": min(source_orders),
                "original_order_end": max(source_orders),
                "segment_id": order,
                "table_title": clean_value(row["table_title"]),
                "table_summary": "",
                "table_kind": clean_value(row["table_kind"]),
                "summary_source": "",
                "content_length": len(content),
                "table_columns_json": clean_value(row["table_columns_json"]),
                "table_records_json": clean_value(row["table_records_json"]),
                "table_row_count": int(row["table_row_count"] or 0),
                "table_column_count": int(row["table_column_count"] or 0),
            }
        )
    columns = [
        "chunk_id", "chunk_type", "section", "current_section", "section_path", "chunk_content",
        "raw_table_content", "table_context_before", "table_context_after", "body_pdf_page",
        "table_source_order", "source_orders", "original_order_start", "original_order_end",
        "segment_id", "table_title", "table_summary", "table_kind", "summary_source",
        "content_length", "table_columns_json", "table_records_json", "table_row_count", "table_column_count",
    ]
    return pd.DataFrame(rows, columns=columns)


def covered_source_orders(chunks: pd.DataFrame) -> set[int]:
    covered: set[int] = set()
    for value in chunks["source_orders"]:
        orders = json.loads(value) if isinstance(value, str) else list(value)
        covered.update(int(order) for order in orders)
    return covered


def combine_chunks_with_tables_last(
    body_frame: pd.DataFrame,
    table_frame: pd.DataFrame,
) -> pd.DataFrame:
    sort_columns = ["original_order_start", "original_order_end"]
    ordered_body = body_frame.sort_values(sort_columns, kind="stable")
    ordered_tables = (
        table_frame.sort_values(sort_columns, kind="stable")
        if not table_frame.empty
        else table_frame.copy()
    )
    combined = pd.concat(
        [ordered_body, ordered_tables],
        ignore_index=True,
        sort=False,
    )
    combined.insert(0, "output_order", range(1, len(combined) + 1))
    return combined


def assign_chunk_ids(
    df: pd.DataFrame,
    section: str,
    doc_no_col: str | None = None,
) -> pd.DataFrame:
    """{section}_{country_name}_{문서키}_{idx} 형식으로 안정적인 ID를 부여합니다."""
    result = df.copy()
    if "country_name" not in result.columns:
        raise ValueError("chunk ID 생성에는 country_name 컬럼이 필요합니다.")
    if doc_no_col and doc_no_col not in result.columns:
        raise ValueError(f"문서번호 컬럼이 없습니다: {doc_no_col}")
    result["_doc_key"] = result.apply(
        lambda row: _doc_key(
            doc_no=row[doc_no_col] if doc_no_col else None,
            from_url=row.get("from_url"),
            title=row.get("title"),
        ),
        axis=1,
    )
    result["_idx"] = result.groupby("_doc_key", sort=False).cumcount()
    normalized_section = clean_value(section)
    result["country_name"] = result["country_name"].map(clean_value)
    result["chunk_id"] = (
        normalized_section
        + "_" + result["country_name"].astype(str)
        + "_" + result["_doc_key"].astype(str)
        + "_" + result["_idx"].astype(str)
    )
    result["section"] = normalized_section
    return result.drop(columns=["_doc_key", "_idx"])

In [ ]:
MIN_CHARS = 150
DROP_SHORT_CHARS = 0  # 짧은 본문 보존; 필요하면 100 등으로 조정
MAX_CHARS = 1600

In [ ]:
begin_stage("chunks")
if not 0 <= DROP_SHORT_CHARS < MIN_CHARS < MAX_CHARS:
    raise ValueError(
        "길이 기준은 0 <= DROP_SHORT_CHARS < MIN_CHARS < MAX_CHARS여야 합니다."
    )
document_key, document_key_source = resolve_document_key()

body_chunks, dropped_short_chunks = chunk_body_elements(
    raw_elements,
    boundary_by_source_order,
)
if body_chunks["content_length"].gt(MAX_CHARS).any():
    raise RuntimeError("MAX_CHARS를 초과한 본문 청크가 있습니다.")

table_chunks = build_table_chunks(raw_elements)
base_chunks = combine_chunks_with_tables_last(body_chunks, table_chunks)
base_chunks["country_name"] = DOCUMENT_COUNTRY
# USE_URL_FOR_DOC_KEY=False이면 ID를 부여하는 동안만 URL을 비워 문서키가 다음 순위(문서 제목 해시)로 넘어가게 합니다.
base_chunks["from_url"] = FROM_URL if USE_URL_FOR_DOC_KEY else ""
base_chunks["title"] = DOCUMENT_NAME
base_chunks["document_no"] = DOCUMENT_NO or ""
base_chunks = assign_chunk_ids(
    base_chunks,
    DOCUMENT_SECTION,
    doc_no_col="document_no" if DOCUMENT_NO else None,
)
base_chunks["from_url"] = FROM_URL
body_chunks = base_chunks[base_chunks["chunk_type"].eq("body")].copy()
table_chunks = base_chunks[base_chunks["chunk_type"].eq("table")].copy()

expected_source_orders = set(raw_elements["source_order"].astype(int))
actual_source_orders = covered_source_orders(base_chunks)
intentionally_dropped_source_orders = (
    covered_source_orders(dropped_short_chunks)
    if not dropped_short_chunks.empty
    else set()
)
missing_source_orders = sorted(
    expected_source_orders
    - actual_source_orders
    - intentionally_dropped_source_orders
)
extra_source_orders = sorted(actual_source_orders - expected_source_orders)
if missing_source_orders or extra_source_orders:
    raise RuntimeError(
        "본문·표 청크의 원본 요소 coverage가 완전하지 않습니다: "
        f"missing={missing_source_orders[:20]}, extra={extra_source_orders[:20]}"
    )

dropped_short_chunks.to_csv(
    DROPPED_SHORT_CHUNKS_CSV,
    index=False,
    encoding="utf-8-sig",
)
body_chunks.to_csv(BODY_CHUNKS_CSV, index=False, encoding="utf-8-sig")
table_chunks.to_csv(TABLE_CHUNKS_CSV, index=False, encoding="utf-8-sig")
complete_stage("chunks")
print("body chunks:", len(body_chunks), "/ table chunks:", len(table_chunks))
print("document key:", document_key, f"({document_key_source})")
if not base_chunks.empty:
    print("first chunk_id:", base_chunks["chunk_id"].iloc[0])

In [ ]:
require_valid("chunks")
short_body_chunks = body_chunks[body_chunks["content_length"].lt(MIN_CHARS)]
print("chunk length:", MIN_CHARS, "-", MAX_CHARS)
print("body chunks:", len(body_chunks))
print(f"body chunks {DROP_SHORT_CHARS}-{MIN_CHARS - 1} chars retained:", len(short_body_chunks))
print(f"body chunks under {DROP_SHORT_CHARS} chars dropped:", len(dropped_short_chunks))
print("dropped audit:", DROPPED_SHORT_CHUNKS_CSV)
print("table chunks:", len(table_chunks))
print("table placement: after all body chunks")
print(
    "fallback table chunks:",
    int(table_chunks["table_kind"].eq("fallback_markdown").sum()) if not table_chunks.empty else 0,
)
display(
    base_chunks[
        ["output_order", "chunk_id", "chunk_type", "current_section", "content_length"]
    ].head(40)
)
if not dropped_short_chunks.empty:
    display(
        dropped_short_chunks[
            ["current_section", "content_length", "chunk_content"]
        ]
    )

## 9. 표 설명 생성

표 청크마다 검색용 설명을 만듭니다. 설명은 최종 CSV 컬럼에는 들어가지 않고 Prefix 생성의 참고자료와
`final_chunks.csv` 메타데이터로 쓰입니다. 같은 표·모델 조합은 캐시를 재사용합니다.

설명을 직접 고치려면 `TABLE_SUMMARY_OVERRIDES`에 `{chunk_id: 설명}`을 넣고 **10번 실행 셀부터** 다시 실행합니다.
항목을 지우면 생성된 설명으로 되돌아갑니다.

In [ ]:
# 표 설명·공통 API 호출 함수

def parse_json_list(value: object) -> list:
    if not isinstance(value, str) or not value.strip():
        return []
    try:
        parsed = json.loads(value)
    except json.JSONDecodeError:
        return []
    return parsed if isinstance(parsed, list) else []

def local_table_summary(row: pd.Series) -> str:
    title = clean_value(row.get("table_title"))
    columns = parse_json_list(row.get("table_columns_json"))
    row_count = int(row.get("table_row_count") or 0)
    column_count = int(row.get("table_column_count") or 0)
    column_text = ", ".join(str(column) for column in columns[:8] if str(column).strip())
    if len(columns) > 8:
        column_text += " 등"
    parts = [f"{title} 관련 표" if title else ""]
    if row_count or column_count:
        parts.append(f"{row_count}행 {column_count}열")
    if column_text:
        parts.append(f"주요 항목: {column_text}")
    return ", ".join(part for part in parts if part) or "문서 내 마크다운 표"

def table_summary_input(row: pd.Series, max_table_chars: int = 6000) -> str:
    columns = parse_json_list(row.get("table_columns_json"))
    records = parse_json_list(row.get("table_records_json"))
    return f"""다음은 KOTRA 문서에서 추출한 표입니다.

요구사항:
- 문서 검색에 사용할 한두 문장의 정확하고 간결한 표 설명을 작성하세요.
- 표 제목이나 섹션 제목이 있으면 반영하세요.
- 수치를 임의로 만들지 말고 표의 주요 주제와 항목을 중심으로 작성하세요.

섹션 경로:
{clean_value(row.get('section_path'))}

표 제목:
{clean_value(row.get('table_title'))}

열:
{json.dumps(columns, ensure_ascii=False)}

DataFrame 일부:
{json.dumps(records[:5], ensure_ascii=False, default=str)}

표 원문:
{clean_value(row.get('chunk_content'))[:max_table_chars]}
""".strip()

def table_summary_request(row: pd.Series) -> dict[str, object]:
    prompt = table_summary_input(row)
    return {
        "cache_key": stable_json_hash(
            {"kind": "table", "model": OPENAI_MODEL, "prompt": prompt}
        ),
        "params": {
            "messages": [
                {
                    "role": "system",
                    "content": "문서 검색용 표 설명을 간결하고 정확하게 작성한다.",
                },
                {"role": "user", "content": prompt},
            ],
            "temperature": 0,
        },
        "clean": clean_value,
    }

async def request_openai_cached(
    requests: list[dict[str, object]],
    cache: dict[str, str],
    cache_path: Path,
    label: str,
) -> list[str]:
    client = AsyncOpenAI()
    semaphore = asyncio.Semaphore(OPENAI_MAX_CONCURRENCY)
    completed = 0

    @retry(wait=wait_random_exponential(min=1, max=20), stop=stop_after_attempt(3))
    async def request_one(request: dict[str, object]) -> str:
        cache_key = request["cache_key"]
        if cache_key in cache:
            return cache[cache_key]
        params = dict(request["params"])
        messages = params.pop("messages")

        if "max_tokens" in params:
            params["max_output_tokens"] = params.pop("max_tokens")

        response = await client.responses.create(
            model=OPENAI_MODEL,
            input=messages,
            store=False,
            **params,
        )

        if response.status != "completed" or not response.output_text.strip():
            raise RuntimeError(
                f"응답 생성 미완료: {response.status}, "
                f"{response.incomplete_details}"
            )

        value = request["clean"](response.output_text)
        cache[cache_key] = value
        save_cache(cache, cache_path)
        return value

    async def run_one(request: dict[str, object]) -> str:
        nonlocal completed
        async with semaphore:
            result = await request_one(request)
        completed += 1
        if completed == len(requests) or completed % 25 == 0:
            print(f"{label}: {completed}/{len(requests)}", flush=True)
        return result

    return await asyncio.gather(*(run_one(request) for request in requests))

def generate_with_openai(
    requests: list[dict[str, object]],
    cache_path: Path,
    label: str,
) -> list[str]:
    cache = load_cache(cache_path)
    cache_hits = sum(request["cache_key"] in cache for request in requests)
    print(f"{label} cache hits:", cache_hits)
    print(f"{label} API calls expected:", len(requests) - cache_hits)
    if cache_hits == len(requests):
        return [cache[request["cache_key"]] for request in requests]
    load_dotenv(ENV_PATH)
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("OPENAI_API_KEY가 필요합니다.")
    return asyncio.run(request_openai_cached(requests, cache, cache_path, label))

def generate_table_summaries(tables: pd.DataFrame, mode: str) -> pd.DataFrame:
    """표 설명 생성 원본. override는 effective_table_summaries()가 복사본에 적용합니다."""
    result_columns = [
        "chunk_id", "table_title", "section_path", "table_summary", "summary_source"
    ]
    if tables.empty:
        return pd.DataFrame(columns=result_columns)
    rows = [row for _, row in tables.iterrows()]
    normalized_mode = mode.strip().lower()
    if normalized_mode == "local":
        summaries = [local_table_summary(row) for row in rows]
        source = "local"
    elif normalized_mode == "openai":
        summaries = generate_with_openai(
            [table_summary_request(row) for row in rows],
            TABLE_SUMMARY_CACHE_JSON,
            "table summaries",
        )
        source = f"openai:{OPENAI_MODEL}"
    else:
        raise ValueError(f"지원하지 않는 TABLE_SUMMARY_MODE: {mode}")

    results = tables[["chunk_id", "table_title", "section_path"]].copy()
    results["table_summary"] = summaries
    results["summary_source"] = source
    return results[result_columns]

def apply_overrides(
    generated: pd.DataFrame,
    overrides: dict[str, str],
    value_column: str,
    source_column: str,
    setting_name: str,
    clean: Callable[[object], str],
) -> pd.DataFrame:
    """생성 원본은 그대로 두고, 복사본에 현재 override만 적용합니다."""
    result = generated.copy()
    known_ids = set(result["chunk_id"])
    for chunk_id, value in overrides.items():
        if chunk_id not in known_ids:
            raise ValueError(f"{setting_name}의 chunk_id가 없습니다: {chunk_id}")
        selected = result["chunk_id"].eq(chunk_id)
        result.loc[selected, value_column] = clean(value)
        result.loc[selected, source_column] = "manual_override"
    return result

def effective_table_summaries(generated: pd.DataFrame) -> pd.DataFrame:
    return apply_overrides(
        generated,
        TABLE_SUMMARY_OVERRIDES,
        "table_summary",
        "summary_source",
        "TABLE_SUMMARY_OVERRIDES",
        clean_value,
    )

In [ ]:
# 표 설명과 Prefix 생성에 함께 쓰는 OpenAI 설정입니다.
OPENAI_MODEL = "gpt-4o-mini"
OPENAI_MAX_CONCURRENCY = 5

TABLE_SUMMARY_MODE = "openai"  # openai | local (API 없이 제목·열 이름으로 생성)
TABLE_SUMMARY_OVERRIDES: dict[str, str] = {}

In [ ]:
begin_stage("table_summaries")
table_summary_generated = generate_table_summaries(table_chunks, TABLE_SUMMARY_MODE)
complete_stage("table_summaries")
print("table summaries generated:", len(table_summary_generated))

In [ ]:
require_valid("table_summaries")
table_summary_preview = effective_table_summaries(table_summary_generated)
print("table summaries:", len(table_summary_preview))
print(
    "empty table summaries:",
    int(table_summary_preview["table_summary"].fillna("").str.strip().eq("").sum()),
)
print(
    "manual overrides:",
    int(table_summary_preview["summary_source"].eq("manual_override").sum()),
)
display(table_summary_preview.head(30))

## 10. Prefix 생성

실행 셀이 먼저 표 설명(override 포함)을 청크에 반영한 뒤 Prefix를 생성합니다.
같은 내용·모델·최대 길이 조합은 캐시를 재사용합니다.

Prefix를 직접 고치려면 `PREFIX_OVERRIDES`에 `{chunk_id: Prefix}`를 넣고 **11번 실행 셀부터** 다시 실행합니다.
항목을 지우면 생성된 Prefix로 되돌아갑니다.

In [ ]:
# 표 설명 반영·Prefix 함수

def prefix_input(row: pd.Series) -> str:
    parts = [
        clean_value(row.get("current_section")),
        clean_value(row.get("table_title")),
        clean_value(row.get("table_summary")),
        clean_value(row.get("chunk_content")),
    ]
    return "\n\n".join(part for part in parts if part)

def local_prefix(row: pd.Series) -> str:
    for column in ["table_summary", "current_section", "table_title", "chunk_content"]:
        value = clean_value(row.get(column))
        if value:
            return normalize_spaces(value)[:PREFIX_MAX_CHARS]
    raise ValueError("Prefix를 만들 내용이 없습니다.")

def clean_prefix(value: object) -> str:
    return normalize_spaces(clean_value(value))[:PREFIX_MAX_CHARS]

def prefix_request(row: pd.Series) -> dict[str, object]:
    source = prefix_input(row)
    prompt = (
        f"다음 내용을 검색용 핵심 명사구 한 줄로 {PREFIX_MAX_CHARS}자 이내로 "
        "요약하세요. 원문에 없는 사실은 추가하지 마세요.\n\n"
        f"{source[:6000]}"
    )
    return {
        "cache_key": stable_json_hash(
            {
                "kind": "prefix",
                "model": OPENAI_MODEL,
                "max_chars": PREFIX_MAX_CHARS,
                "source": source,
            }
        ),
        "params": {
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": 80,
            "temperature": 0.2,
        },
        "clean": clean_prefix,
    }

def generate_prefixes(chunks: pd.DataFrame, mode: str) -> pd.DataFrame:
    """Prefix 생성 원본. override는 effective_prefixes()가 복사본에 적용합니다."""
    rows = [row for _, row in chunks.iterrows()]
    normalized_mode = mode.strip().lower()
    if normalized_mode == "local":
        prefixes = [local_prefix(row) for row in rows]
        source = "local"
    elif normalized_mode == "openai":
        prefixes = generate_with_openai(
            [prefix_request(row) for row in rows],
            PREFIX_CACHE_JSON,
            "prefixes",
        )
        source = f"openai:{OPENAI_MODEL}"
    else:
        raise ValueError(f"지원하지 않는 PREFIX_MODE: {mode}")

    results = chunks[["chunk_id", "chunk_type", "current_section", "chunk_content"]].copy()
    results["prefix_content"] = prefixes
    results["prefix_source"] = source
    return results

def effective_prefixes(generated: pd.DataFrame) -> pd.DataFrame:
    return apply_overrides(
        generated,
        PREFIX_OVERRIDES,
        "prefix_content",
        "prefix_source",
        "PREFIX_OVERRIDES",
        clean_prefix,
    )

def attach_table_summaries(
    chunks: pd.DataFrame,
    summaries: pd.DataFrame,
) -> pd.DataFrame:
    result = chunks.copy()
    if summaries.empty:
        return result
    summary_by_id = summaries.set_index("chunk_id")
    for index, row in result[result["chunk_type"].eq("table")].iterrows():
        chunk_id = row["chunk_id"]
        if chunk_id not in summary_by_id.index:
            raise ValueError(f"표 설명이 없는 chunk_id: {chunk_id}")
        result.at[index, "table_summary"] = clean_value(
            summary_by_id.loc[chunk_id, "table_summary"]
        )
        result.at[index, "summary_source"] = clean_value(
            summary_by_id.loc[chunk_id, "summary_source"]
        )
    return result

In [ ]:
PREFIX_MODE = "openai"  # openai | local (API 없이 제목 기반으로 생성)
PREFIX_MAX_CHARS = 40
PREFIX_OVERRIDES: dict[str, str] = {}

In [ ]:
begin_stage("prefixes")
# 표 설명 반영: 생성 원본의 복사본에 현재 override를 적용해 저장합니다.
table_summary_results = effective_table_summaries(table_summary_generated)
table_summary_results.to_csv(
    TABLE_SUMMARIES_CSV,
    index=False,
    encoding="utf-8-sig",
)
enriched_chunks = attach_table_summaries(base_chunks, table_summary_results)
prefix_generated = generate_prefixes(enriched_chunks, PREFIX_MODE)
complete_stage("prefixes")
print("prefixes generated:", len(prefix_generated))

In [ ]:
require_valid("prefixes")
prefix_preview = effective_prefixes(prefix_generated)
prefix_values = prefix_preview["prefix_content"].fillna("")
print("prefixes:", len(prefix_preview))
print("empty prefixes:", int(prefix_values.str.strip().eq("").sum()))
print("long prefixes:", int(prefix_values.str.len().gt(PREFIX_MAX_CHARS).sum()))
print(
    "duplicate prefixes:",
    int(prefix_values[prefix_values.str.strip().ne("")].duplicated().sum()),
)
print("manual overrides:", int(prefix_preview["prefix_source"].eq("manual_override").sum()))
display(prefix_preview.head(30))

## 11. 최종 확인과 저장

실행 셀이 Prefix(override 포함)를 반영해 최종 결과를 만들고 QA를 수행합니다.
표 행의 `chunk_content`에는 제목·원본 표·원문 설명을 함께 저장하고, `embedded_text`에는 Prefix와 그 전체 내용을 저장합니다.

본문·표 청크 앞에는 그 청크의 `section_path`(예: `제1장 … > 제1절 … > □ …`)를 한 줄로 붙입니다.
Prefix를 만든 뒤에 붙이므로 API를 다시 호출하지 않습니다. 원문·chunk_id·순서·컬럼은 그대로입니다.

QA 이후 본문·길이 기준·override 등을 바꾸면 저장 셀이 중단되며 다시 실행할 셀을 알려 줍니다.

QA 오류가 있으면 저장하지 않습니다. 확인해 보니 사소해서 그대로 저장해도 되는 오류라면,
승인 셀의 `FORCE_SAVE_WITH_QA_ERRORS=True`로 강제 저장할 수 있습니다.
건너뛴 오류 항목은 `final_qa.csv`의 `ignored` 칸에 `True`로 남습니다. 다음 문서에서는 다시 `False`로 돌려 두세요.

In [ ]:
# 최종 결과·QA 함수

FINAL_COLUMNS = [
    "chunk_id",
    "section",
    "country_name",
    "chunk_content",
    "prefix_content",
    "embedded_text",
    "from_url",
]


def compact_blank_lines(text: str) -> str:
    return re.sub(r"\n{3,}", "\n\n", text).strip()


def add_section_path(content: object, section_path: object) -> str:
    """청크 앞에 section_path(상위 > 하위 제목 경로)를 한 줄로 붙입니다."""
    original = clean_value(content)
    path = clean_value(section_path)
    return f"{path}\n\n{original}" if path else original


def build_final_frames(
    chunks: pd.DataFrame,
    prefixes_frame: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    prefix_by_id = prefixes_frame.set_index("chunk_id")["prefix_content"].to_dict()
    detailed = chunks.copy()
    detailed["prefix_content"] = detailed["chunk_id"].map(prefix_by_id).fillna("")
    detailed["from_url"] = FROM_URL
    # section_path는 Prefix 생성 뒤에 붙이므로 Prefix 캐시·API 호출에 영향을 주지 않습니다.
    detailed["chunk_content"] = [
        add_section_path(content, path)
        for content, path in zip(detailed["chunk_content"], detailed["section_path"])
    ]

    output_rows = []
    embedded_values = []
    for _, row in detailed.iterrows():
        prefix = clean_value(row["prefix_content"])
        chunk_content = clean_value(row["chunk_content"])
        embedded_text = compact_blank_lines(
            "\n\n".join(dict.fromkeys(part for part in [prefix, chunk_content] if part))
        )
        embedded_values.append(embedded_text)
        output_rows.append(
            {
                "chunk_id": row["chunk_id"],
                "section": DOCUMENT_SECTION,
                "country_name": DOCUMENT_COUNTRY,
                "chunk_content": chunk_content,
                "prefix_content": prefix,
                "embedded_text": embedded_text,
                "from_url": FROM_URL,
            }
        )
    detailed["embedded_text"] = embedded_values
    return detailed, pd.DataFrame(output_rows, columns=FINAL_COLUMNS)


def final_qa(
    detailed: pd.DataFrame,
    output: pd.DataFrame,
) -> tuple[pd.DataFrame, bool]:
    required_nonempty = [
        "chunk_id", "section", "chunk_content", "prefix_content", "embedded_text"
    ]
    empty_counts = {
        column: int(output[column].fillna("").astype(str).str.strip().eq("").sum())
        for column in required_nonempty
    }

    def remaining(pattern: re.Pattern[str]) -> int:
        return sum(
            len(pattern.findall(clean_value(value)))
            for column in output.columns
            for value in output[column]
            if isinstance(value, str)
        )

    body_rows = detailed[detailed["chunk_type"].eq("body")]
    table_rows = detailed[detailed["chunk_type"].eq("table")]
    expected_source_orders = set(raw_elements["source_order"].astype(int))
    actual_source_orders = covered_source_orders(detailed)
    dropped_source_orders = (
        covered_source_orders(dropped_short_chunks)
        if not dropped_short_chunks.empty
        else set()
    )
    table_orders = [
        order for value in table_rows["source_orders"] for order in json.loads(value)
    ]
    prefixes = output["prefix_content"].fillna("").astype(str)
    metrics = [
        ("has_toc", HAS_TOC, "info"),
        ("body_pdf_pages", body_pdf_page_count, "info"),
        ("selected_boundaries", len(boundary_items), "info"),
        ("total_chunks", len(detailed), "info"),
        ("body_chunks", len(body_rows), "info"),
        ("table_chunks", len(table_rows), "info"),
        ("dropped_short_body_chunks", len(dropped_short_chunks), "info"),
        ("duplicate_chunk_ids", int(output["chunk_id"].duplicated().sum()), "error"),
        ("empty_required_values", sum(empty_counts.values()), "error"),
        ("boundary_errors", len(boundary_errors), "error"),
        (
            "missing_source_orders",
            len(expected_source_orders - actual_source_orders - dropped_source_orders),
            "error",
        ),
        ("extra_source_orders", len(actual_source_orders - expected_source_orders), "error"),
        ("duplicated_table_source_orders", len(table_orders) - len(set(table_orders)), "error"),
        (
            "body_table_source_overlap",
            len(covered_source_orders(body_rows) & set(table_orders)),
            "error",
        ),
        ("body_over_max", int(body_rows["content_length"].gt(MAX_CHARS).sum()), "error"),
        ("body_under_min", int(body_rows["content_length"].lt(MIN_CHARS).sum()), "warning"),
        ("table_over_max", int(table_rows["content_length"].gt(MAX_CHARS).sum()), "warning"),
        ("remaining_html_entities", remaining(HTML_ENTITY_PATTERN), "error"),
        ("remaining_formatting_tags", remaining(HTML_FORMATTING_TAG_PATTERN), "error"),
        ("remaining_blockquote_lines", remaining(BLOCKQUOTE_LINE_PATTERN), "error"),
        (
            "body_source_order_monotonic",
            bool(body_rows["original_order_start"].is_monotonic_increasing),
            "boolean_error",
        ),
        (
            "table_source_order_monotonic",
            bool(table_rows["original_order_start"].is_monotonic_increasing),
            "boolean_error",
        ),
        (
            "empty_table_summaries",
            int(table_rows["table_summary"].fillna("").astype(str).str.strip().eq("").sum()),
            "error",
        ),
        ("empty_prefixes", int(prefixes.str.strip().eq("").sum()), "error"),
        ("long_prefixes", int(prefixes.str.len().gt(PREFIX_MAX_CHARS).sum()), "error"),
    ]
    rows = []
    passed = True
    for metric, value, severity in metrics:
        if severity == "error":
            ok = value == 0
        elif severity == "boolean_error":
            ok = bool(value)
        else:
            ok = True
        if not ok:
            passed = False
        rows.append(
            {
                "metric": metric,
                "value": value,
                "severity": "error" if not ok else severity.replace("_error", ""),
                "passed": ok,
            }
        )
    return pd.DataFrame(rows), passed

In [ ]:
begin_stage("final")
# Prefix 반영: 생성 원본의 복사본에 현재 override를 적용해 저장합니다.
prefix_results = effective_prefixes(prefix_generated)
prefix_results.to_csv(PREFIX_RESULTS_CSV, index=False, encoding="utf-8-sig")
final_chunks, final_output = build_final_frames(enriched_chunks, prefix_results)
qa_results, qa_passed = final_qa(final_chunks, final_output)
complete_stage("final")
final_review_state = preview_stage("final_save")

chunk_lengths = final_chunks["content_length"].fillna(0).astype(int)
print("chunks:", len(final_chunks))
print(
    "content length min/median/max:",
    int(chunk_lengths.min()),
    int(chunk_lengths.median()),
    int(chunk_lengths.max()),
)
print("QA passed:", qa_passed)
display(qa_results)
display(final_output.head(20))

show_user_checkpoint(
    "최종 결과",
    [
        f"최종 행 수: {len(final_output)}",
        f"QA 통과 여부: {qa_passed}",
        f"저장 예정 파일: `{FINAL_OUTPUT_CSV}`",
    ],
    "결과가 맞으면 아래 승인 셀의 FINAL_SAVE_APPROVED=True로 바꿔 실행하세요. "
    "QA 오류가 사소해서 그대로 저장하려면 FORCE_SAVE_WITH_QA_ERRORS=True로 바꾸세요.",
)

In [ ]:
FINAL_SAVE_APPROVED = True
# True이면 QA 오류가 있어도 저장합니다. 위 QA 표에서 오류 내용을 확인한 뒤에만 켜고, 다음 문서에서는 False로 돌려 두세요.
FORCE_SAVE_WITH_QA_ERRORS = False
approve_stage("final_save", FINAL_SAVE_APPROVED, final_review_state)

In [ ]:
require_valid("final_save")
if not isinstance(FORCE_SAVE_WITH_QA_ERRORS, bool):
    raise ValueError("FORCE_SAVE_WITH_QA_ERRORS는 True 또는 False여야 합니다.")
failed_metrics = qa_results.loc[~qa_results["passed"], "metric"].tolist()
if failed_metrics and not FORCE_SAVE_WITH_QA_ERRORS:
    raise RuntimeError(
        f"최종 QA 실패 항목이 있어 저장하지 않습니다: {failed_metrics}. "
        "확인 후 그대로 저장하려면 승인 셀의 FORCE_SAVE_WITH_QA_ERRORS=True로 바꾸고 승인 셀부터 다시 실행하세요."
    )
ignored_metrics = failed_metrics if FORCE_SAVE_WITH_QA_ERRORS else []
if ignored_metrics:
    print("주의: QA 오류가 있지만 강제 저장합니다:", ignored_metrics)
saved_qa_results = qa_results.assign(ignored=qa_results["metric"].isin(ignored_metrics))

final_chunks.to_csv(FINAL_CHUNKS_CSV, index=False, encoding="utf-8-sig")
final_output.to_csv(FINAL_OUTPUT_CSV, index=False, encoding="utf-8-sig")
saved_qa_results.to_csv(FINAL_QA_CSV, index=False, encoding="utf-8-sig")

print("saved:", FINAL_CHUNKS_CSV)
print("saved:", FINAL_OUTPUT_CSV)
print("saved:", FINAL_QA_CSV)